## Import

In [1]:
from pathlib import Path
from itertools import product
import time, os, random, json, hashlib
from importlib.metadata import PackageNotFoundError, version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


import gc # Per pulizia memoria

from sklearn.preprocessing import QuantileTransformer
from sklearn.covariance import LedoitWolf
from scipy.stats import genpareto
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, average_precision_score,   # ← NUOVI
)

from sklearn.metrics import precision_recall_curve

# Windows + AMD: il plugin TensorFlow-DirectML legge questa variabile prima
# dell'import di TensorFlow. 0 e' normalmente la Radeon discreta.
DIRECTML_ADAPTER_INDEX = 0
os.environ.setdefault("DML_VISIBLE_DEVICES", str(DIRECTML_ADAPTER_INDEX))

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LeakyReLU, BatchNormalization, GaussianNoise,  # ← GaussianNoise NUOVO
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

sns.set_theme(style="whitegrid")

def installed_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None

DIRECTML_PLUGIN_VERSION = installed_version('tensorflow-directml-plugin')

def configure_accelerator():
    """Rileva la GPU TensorFlow senza forzare API CUDA.

    Con tensorflow-directml-plugin l'adapter si seleziona tramite
    DML_VISIBLE_DEVICES, non con set_visible_devices o memory_growth.
    """
    physical_gpus = tf.config.list_physical_devices('GPU')
    logical_gpus = tf.config.list_logical_devices('GPU')
    is_directml = DIRECTML_PLUGIN_VERSION is not None

    if not logical_gpus:
        setup = (
            'Installa l ambiente DirectML: conda env create -f environment_directml.yml'
            if not is_directml else
            'Il plugin DirectML e installato, ma TensorFlow non vede una GPU DirectX 12.'
        )
        return {
            'gpu_available': False, 'backend': 'DirectML' if is_directml else 'CPU',
            'label': 'CPU', 'logical_device': '/CPU:0', 'message': setup,
        }

    if is_directml:
        return {
            'gpu_available': True, 'backend': 'DirectML',
            'label': f'AMD Radeon adapter {os.environ["DML_VISIBLE_DEVICES"]}',
            'logical_device': logical_gpus[0].name,
            'message': f'tensorflow-directml-plugin {DIRECTML_PLUGIN_VERSION}',
        }

    # Fallback per eventuali backend GPU TensorFlow non-DirectML.
    gpu = physical_gpus[0]
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except (RuntimeError, ValueError):
        pass
    return {
        'gpu_available': True, 'backend': 'TensorFlow GPU',
        'label': str(gpu), 'logical_device': logical_gpus[0].name, 'message': '',
    }

ACCELERATOR = configure_accelerator()
print('TensorFlow:', tf.__version__)
if ACCELERATOR['gpu_available']:
    print(f"Acceleratore selezionato: {ACCELERATOR['backend']} - {ACCELERATOR['label']} ({ACCELERATOR['logical_device']})")
    # Verifica effettiva: il risultato deve essere assegnato alla GPU.
    with tf.device(ACCELERATOR['logical_device']):
        gpu_check = tf.matmul(tf.ones((128, 128)), tf.ones((128, 128)))
    print(f"Verifica device: {gpu_check.device}")
    del gpu_check
else:
    print(f"ATTENZIONE: addestramento su CPU. {ACCELERATOR['message']}")


TensorFlow: 2.10.0
Acceleratore selezionato: DirectML - AMD Radeon adapter 0 (/device:GPU:0)
Verifica device: /job:localhost/replica:0/task:0/device:GPU:0


## Associa a Variabili il Path Dei File E Divisione del Dataset

In [2]:
DATASET_DIR_2018 = Path("dataset_final_2018")
DATASET_DIR_2017 = Path("dataset_final_2017")

FILES_2018 = sorted(DATASET_DIR_2018.glob("*.csv"))
FILES_2017 = sorted(DATASET_DIR_2017.glob("*.csv"))

print(f"File 2018 (train + grid search): {len(FILES_2018)}")
for f in FILES_2018:
    print(" ", f.name)

print(f"\nFile 2017 (stress test esterno, usato una sola volta): {len(FILES_2017)}")
for f in FILES_2017:
    print(" ", f.name)


File 2018 (train + grid search): 10
  Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
  Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
  Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
  Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
  Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
  Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
  Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
  Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
  Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
  Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv

File 2017 (stress test esterno, usato una sola volta): 8
  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  Friday-WorkingHours-Morning-Bot.pcap_ISCX.csv
  Monday-WorkingHours.pcap_ISCX.csv
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  Tuesday-WorkingHours.pcap_ISCX.csv
  Wednesday-workingHours.pcap_ISCX.csv


## Verifica della presenza corretta di righe Benigne o Malevoli (Con definizione dei tipi)

In [3]:
def report_labels(files: list[Path]) -> pd.DataFrame:
    report = {}
    for path in files:
        df = pd.read_csv(path)
        df.columns = df.columns.str.strip()
        report[path.name] = df["Label"].value_counts()
    return pd.DataFrame(report).fillna(0).astype(int)

print("=" * 65)
print("Label count — 2018")
print("=" * 65)
print(report_labels(FILES_2018))

print("\n" + "=" * 65)
print("Label count — 2017")
print("=" * 65)
print(report_labels(FILES_2017))


Label count — 2018
                          Friday-02-03-2018_TrafficForML_CICFlowMeter.csv  \
Benign                                                             758334   
Bot                                                                286191   
Brute Force -Web                                                        0   
Brute Force -XSS                                                        0   
DDOS attack-HOIC                                                        0   
DDOS attack-LOIC-UDP                                                    0   
DDoS attacks-LOIC-HTTP                                                  0   
DoS attacks-GoldenEye                                                   0   
DoS attacks-Hulk                                                        0   
DoS attacks-SlowHTTPTest                                                0   
DoS attacks-Slowloris                                                   0   
FTP-BruteForce                                           

## Audit dataset e preprocessing anti-leakage

Questa sezione controlla la lineage dei dataset senza modificare i file originali. L'obiettivo e' verificare schema finale, ordine colonne e distribuzione label nelle cartelle raw/clean/final. Le informazioni sono diagnostiche e servono a documentare la qualita' del preprocessing.


In [4]:
# FINAL_COLUMNS_COMPAT_V1
def infer_final_columns_from_csv():
    reference_files = sorted(Path("dataset_final_2017").glob("*.csv"))
    if not reference_files:
        reference_files = sorted(Path("dataset_final_2018").glob("*.csv"))
    if not reference_files:
        return []
    return pd.read_csv(reference_files[0], nrows=0).columns.str.strip().tolist()


FINAL_COLUMNS = infer_final_columns_from_csv()
assert FINAL_COLUMNS, "Impossibile inferire FINAL_COLUMNS dai CSV finali."
assert len(FINAL_COLUMNS) == 60, f"Schema finale inatteso: {len(FINAL_COLUMNS)} colonne."
print(f"FINAL_COLUMNS inferito dai CSV finali: {len(FINAL_COLUMNS)} colonne.")


FINAL_COLUMNS inferito dai CSV finali: 60 colonne.


### Audit delle distribuzioni delle etichette

Calcolo delle distribuzioni per classe e verifica della coerenza dei dataset elaborati.


In [5]:
# DATASET_LINEAGE_AUDIT_V3
def summarize_label_distribution(dataset_dir: Path, version_name: str) -> pd.DataFrame:
    rows = []
    dataset_dir = Path(dataset_dir)
    if not dataset_dir.exists():
        return pd.DataFrame([{
            "version": version_name,
            "file": None,
            "Label": None,
            "count": 0,
            "status": "MISSING_DIR",
        }])

    csv_files = sorted(dataset_dir.glob("*.csv"))
    if not csv_files:
        return pd.DataFrame([{
            "version": version_name,
            "file": None,
            "Label": None,
            "count": 0,
            "status": "EMPTY_DIR",
        }])

    for csv_file in csv_files:
        try:
            header_cols = pd.read_csv(csv_file, nrows=0).columns.tolist()
            stripped_to_raw = {col.strip(): col for col in header_cols}
            if "Label" not in stripped_to_raw:
                raise ValueError("Label column not found after stripping column names")

            labels = pd.read_csv(csv_file, usecols=[stripped_to_raw["Label"]], low_memory=False)
            labels.columns = labels.columns.str.strip()
            labels = labels[labels["Label"] != "Label"]
            counts = labels["Label"].value_counts(dropna=False)
            for label, count in counts.items():
                rows.append({
                    "version": version_name,
                    "file": csv_file.name,
                    "Label": label,
                    "count": int(count),
                    "status": "OK",
                })
        except Exception as exc:
            rows.append({
                "version": version_name,
                "file": csv_file.name,
                "Label": None,
                "count": 0,
                "status": f"ERROR: {type(exc).__name__}: {exc}",
            })
    return pd.DataFrame(rows)


def infer_expected_final_columns():
    if "FINAL_COLUMNS" in globals() and FINAL_COLUMNS:
        return list(FINAL_COLUMNS)
    return infer_final_columns_from_csv()


EXPECTED_FINAL_COLUMNS = infer_expected_final_columns()
FINAL_COLUMNS = EXPECTED_FINAL_COLUMNS


def schema_report(dataset_dir: Path, version_name: str) -> pd.DataFrame:
    rows = []
    dataset_dir = Path(dataset_dir)
    for csv_file in sorted(dataset_dir.glob("*.csv")) if dataset_dir.exists() else []:
        try:
            cols = pd.read_csv(csv_file, nrows=0).columns.str.strip().tolist()
            rows.append({
                "version": version_name,
                "file": csv_file.name,
                "n_columns": len(cols),
                "has_label": "Label" in cols,
                "matches_final_columns": bool(EXPECTED_FINAL_COLUMNS) and cols == EXPECTED_FINAL_COLUMNS,
                "extra_columns": ", ".join([c for c in cols if c not in EXPECTED_FINAL_COLUMNS]),
                "missing_columns": ", ".join([c for c in EXPECTED_FINAL_COLUMNS if c not in cols]),
            })
        except Exception as exc:
            rows.append({
                "version": version_name,
                "file": csv_file.name,
                "n_columns": np.nan,
                "has_label": False,
                "matches_final_columns": False,
                "extra_columns": "",
                "missing_columns": f"ERROR: {type(exc).__name__}: {exc}",
            })
    return pd.DataFrame(rows)


dataset_lineage_df = pd.concat([
    summarize_label_distribution(Path("dataset"), "2017_raw"),
    summarize_label_distribution(Path("dataset_clean"), "2017_clean_dropna"),
    summarize_label_distribution(Path("dataset_final_2017"), "2017_final_current"),
    summarize_label_distribution(Path("dataset_2018"), "2018_raw"),
    summarize_label_distribution(Path("dataset_clean_2018"), "2018_clean_dropna"),
    summarize_label_distribution(Path("dataset_final_2018"), "2018_final_current"),
], ignore_index=True)

dataset_schema_df = pd.concat([
    schema_report(Path("dataset_final_2017"), "2017_final_current"),
    schema_report(Path("dataset_final_2018"), "2018_final_current"),
], ignore_index=True)

print("="*100 + "\nDATASET_SCHEMA_DF\n" + "="*100)
print(dataset_schema_df.to_string(index=False))

print("\n" + "="*100 + "\nDATASET_LINEAGE_DF - riepilogo per versione/label\n" + "="*100)
lineage_summary = (
    dataset_lineage_df
    .groupby(["version", "Label"], dropna=False)["count"]
    .sum()
    .reset_index()
    .sort_values(["version", "Label"])
)
print(lineage_summary.to_string(index=False))

if not dataset_schema_df.empty:
    assert EXPECTED_FINAL_COLUMNS, "Impossibile inferire lo schema finale atteso dai CSV finali."
    schema_ok = dataset_schema_df["matches_final_columns"].fillna(False).all()
    if not schema_ok:
        print("\nATTENZIONE: schema finale non coerente. Righe problematiche:")
        print(dataset_schema_df[~dataset_schema_df["matches_final_columns"].fillna(False)].to_string(index=False))
    assert schema_ok, "Schema finale non coerente tra 2017 e 2018."
    assert len(EXPECTED_FINAL_COLUMNS) == 60, (
        f"Schema finale inatteso: {len(EXPECTED_FINAL_COLUMNS)} colonne, "
        "attese 59 feature + Label."
    )
    print("\nSchema finale verificato: 59 feature + Label, stesso ordine per tutti i CSV finali.")


DATASET_SCHEMA_DF
           version                                                        file  n_columns  has_label  matches_final_columns extra_columns missing_columns
2017_final_current            Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv         60       True                   True                              
2017_final_current        Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv         60       True                   True                              
2017_final_current               Friday-WorkingHours-Morning-Bot.pcap_ISCX.csv         60       True                   True                              
2017_final_current                           Monday-WorkingHours.pcap_ISCX.csv         60       True                   True                              
2017_final_current Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv         60       True                   True                              
2017_final_current      Thursday-WorkingHours-Morning-WebA

## Creazione SEED Per Salvataggi più deterministici

In [6]:
SEED = 42

os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"

def set_seed(seed=42):
    import random, numpy as np, tensorflow as tf, os
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(SEED)


tf.config.experimental.enable_op_determinism() # RENDE IL MODELLO PIU DETERMINISTICO POSSIBILE (ma non garantisce la riproducibilità assoluta)


## Data Loading e Numeric Clean

In [7]:
def load_dataset(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path, low_memory=False)
    df.columns = df.columns.str.strip()
    print(f"  [LOAD] {csv_path.name}  shape={df.shape}")
    return df


def to_numeric_clean(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """Conversione numerica memory-aware.

    Evita DataFrame.apply su milioni di righe perché pandas può materializzare un
    blocco float64 molto grande. Costruire una colonna alla volta in float32 riduce
    il picco di memoria senza cambiare il protocollo sperimentale.
    """
    out = pd.DataFrame(index=df.index)
    for col in cols:
        s = pd.to_numeric(df[col], errors="coerce")
        s = s.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(np.float32, copy=False)
        out[col] = s
    return out


## Costruzione del Set di Training

In [8]:
print("\n" + "=" * 65)
print("STEP 1 — Benign Pool 2018 (Train/Val/Calib disaccoppiato da TestHoldout)")
print("=" * 65)

TRAIN_POOL_SIZE   = 3_000_000   # righe per Train+Val+Calib — controlla la recall
TEST_HOLDOUT_SIZE = 2_000_000   # righe SOLO per il negativo del test interno — controlla il realismo Precision/FPR

benign_frames_2018 = []
attack_frames_2018 = []

for csv_file in FILES_2018:
    df = load_dataset(csv_file)
    is_benign = df["Label"].str.upper() == "BENIGN"
    benign_frames_2018.append(df[is_benign])
    attack_frames_2018.append(df[~is_benign])

all_benign_2018  = pd.concat(benign_frames_2018, ignore_index=True)
all_attacks_2018 = pd.concat(attack_frames_2018, ignore_index=True)
del benign_frames_2018, attack_frames_2018

total_benign_available = len(all_benign_2018)
print(f"  Benign totale disponibile: {total_benign_available:,}")
print(f"  Attacchi totali: {len(all_attacks_2018):,}")

# 1) Estrai PRIMA il TestHoldout dal pool totale — indipendente dal train
test_holdout_ben = all_benign_2018.sample(n=TEST_HOLDOUT_SIZE, random_state=SEED)
remaining_ben     = all_benign_2018.drop(test_holdout_ben.index)   # rimozione per indice → zero overlap

print(f"\n  TestHoldout (negativo test interno) : {len(test_holdout_ben):,}")
print(f"  Benign rimanente per Train/Val/Calib : {len(remaining_ben):,}")

# 2) Campiona il pool Train/Val/Calib dal rimanente — controlla SOLO questo per la recall
if TRAIN_POOL_SIZE is not None and len(remaining_ben) > TRAIN_POOL_SIZE:
    trainval_pool = remaining_ben.sample(n=TRAIN_POOL_SIZE, random_state=SEED)
else:
    trainval_pool = remaining_ben

# Split 70/15/15 — SOLO su Train/Val/Calib, il TestHoldout è già separato
# Protocol split 70/10/5/15: train, early-stop val, model-select benign, threshold calib.
train_ben, remaining_ben_protocol = train_test_split(trainval_pool, test_size=0.30, random_state=SEED)
val_ben, selectcalib_ben = train_test_split(remaining_ben_protocol, test_size=2/3, random_state=SEED)
select_ben, calib_ben = train_test_split(selectcalib_ben, test_size=0.75, random_state=SEED)

print(f"\n  Train = {len(train_ben):,}")
print(f"  Val   = {len(val_ben):,}")
print(f"  Select benign = {len(select_ben):,}")
print(f"  Calib = {len(calib_ben):,}")
print(f"  TestHoldout = {len(test_holdout_ben):,}")
print(f"\n  Rapporto Benigno:Attacco nel test interno = "
      f"{len(test_holdout_ben)/len(all_attacks_2018):.2f} : 1")



STEP 1 — Benign Pool 2018 (Train/Val/Calib disaccoppiato da TestHoldout)
  [LOAD] Friday-02-03-2018_TrafficForML_CICFlowMeter.csv  shape=(1044525, 60)
  [LOAD] Friday-16-02-2018_TrafficForML_CICFlowMeter.csv  shape=(1048574, 60)
  [LOAD] Friday-23-02-2018_TrafficForML_CICFlowMeter.csv  shape=(1042866, 60)
  [LOAD] Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv  shape=(7889295, 60)
  [LOAD] Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv  shape=(329266, 60)
  [LOAD] Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv  shape=(1040548, 60)
  [LOAD] Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv  shape=(1042956, 60)
  [LOAD] Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv  shape=(1044746, 60)
  [LOAD] Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv  shape=(1048575, 60)
  [LOAD] Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv  shape=(609030, 60)
  Benign totale disponibile: 13,392,990
  Attacchi totali: 2,747,391

  TestHoldout (negativo test interno) : 2,000,000
  Benign ri

## Controllo Eventuale Leakage

In [9]:
# ── Verifica esplicita: zero overlap tra TestHoldout e Train/Val/Calib ──
idx_test  = set(test_holdout_ben.index)
idx_train = set(train_ben.index)
idx_val   = set(val_ben.index)
idx_select = set(select_ben.index)
idx_calib = set(calib_ben.index)

overlap_train = idx_test & idx_train
overlap_val   = idx_test & idx_val
overlap_select = idx_test & idx_select
overlap_calib = idx_test & idx_calib

print(f"Overlap TestHoldout ∩ Train : {len(overlap_train)} righe")
print(f"Overlap TestHoldout ∩ Val   : {len(overlap_val)} righe")
print(f"Overlap TestHoldout ∩ Calib : {len(overlap_calib)} righe")

assert len(overlap_train) == 0, "LEAKAGE: righe duplicate tra TestHoldout e Train!"
assert len(overlap_val)   == 0, "LEAKAGE: righe duplicate tra TestHoldout e Val!"
assert len(overlap_select) == 0, "LEAKAGE: righe duplicate tra TestHoldout e Select!"
assert len(overlap_calib) == 0, "LEAKAGE: righe duplicate tra TestHoldout e Calib!"
protocol_index_sets = {
    "train": idx_train, "val": idx_val, "select": idx_select,
    "calib": idx_calib, "test_benign": idx_test,
}
for left_pos, (left_name, left_idx) in enumerate(protocol_index_sets.items()):
    for right_name, right_idx in list(protocol_index_sets.items())[left_pos + 1:]:
        assert left_idx.isdisjoint(right_idx), f"LEAKAGE: overlap {left_name}/{right_name}"
print("\n✅ Nessun leakage per indice: le partizioni sono disgiunte.")


Overlap TestHoldout ∩ Train : 0 righe
Overlap TestHoldout ∩ Val   : 0 righe
Overlap TestHoldout ∩ Calib : 0 righe

✅ Nessun leakage per indice: le partizioni sono disgiunte.


## Costruzione del Set di Test 

In [10]:
print("\n" + "=" * 65)
print("STEP 2 — Test Set Interno 2018 (TestHoldout benign + tutti gli attacchi 2018)")
print("=" * 65)

test_df_internal = pd.concat([test_holdout_ben, all_attacks_2018], ignore_index=True)

print(f"  shape={test_df_internal.shape}")
print(test_df_internal["Label"].value_counts().to_string())

y_test_internal        = test_df_internal["Label"].values
y_test_internal_binary = (pd.Series(y_test_internal).str.upper() != "BENIGN").astype(int).values



STEP 2 — Test Set Interno 2018 (TestHoldout benign + tutti gli attacchi 2018)
  shape=(4747391, 60)
Benign                      2000000
DDOS attack-HOIC             686012
DDoS attacks-LOIC-HTTP       576191
DoS attacks-Hulk             461912
Bot                          286191
FTP-BruteForce               193354
SSH-Bruteforce               187589
Infilteration                161096
DoS attacks-SlowHTTPTest     139890
DoS attacks-GoldenEye         41508
DoS attacks-Slowloris         10990
DDOS attack-LOIC-UDP           1730
Brute Force -Web                611
Brute Force -XSS                230
SQL Injection                    87


## Costruzione del Set di Test Esterno Usando DataSet 2017

In [11]:
print("\n" + "=" * 65)
print("STEP 2.5 — Stress Test Esterno (CIC-IDS2017, mai toccato finora)")
print("=" * 65)

# Conserva soltanto un ID intero del file di origine. Non entra nelle feature:
# serve alla calibrazione cross-fitted per capture della v5.
frames_2017 = []
stress_file_id_parts_2017 = []
stress_file_names_2017 = [path.name for path in FILES_2017]
for file_id, csv_file in enumerate(FILES_2017):
    frame_2017 = load_dataset(csv_file)
    frames_2017.append(frame_2017)
    stress_file_id_parts_2017.append(
        np.full(len(frame_2017), file_id, dtype=np.int8)
    )

stress_test_df_2017 = pd.concat(frames_2017, ignore_index=True)
stress_file_ids_2017 = np.concatenate(stress_file_id_parts_2017)
del frames_2017, stress_file_id_parts_2017, frame_2017

assert len(stress_file_ids_2017) == len(stress_test_df_2017)
print(f"\n  shape={stress_test_df_2017.shape}")
print(stress_test_df_2017["Label"].value_counts().to_string())
print("  Capture ID preservati per cross-fitting v5:", len(stress_file_names_2017))

y_stress_2017        = stress_test_df_2017["Label"].values
y_stress_2017_binary = (pd.Series(y_stress_2017).str.upper() != "BENIGN").astype(int).values



STEP 2.5 — Stress Test Esterno (CIC-IDS2017, mai toccato finora)
  [LOAD] Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv  shape=(225705, 60)
  [LOAD] Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv  shape=(286055, 60)
  [LOAD] Friday-WorkingHours-Morning-Bot.pcap_ISCX.csv  shape=(190887, 60)
  [LOAD] Monday-WorkingHours.pcap_ISCX.csv  shape=(529443, 60)
  [LOAD] Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv  shape=(288379, 60)
  [LOAD] Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv  shape=(170206, 60)
  [LOAD] Tuesday-WorkingHours.pcap_ISCX.csv  shape=(445612, 60)
  [LOAD] Wednesday-workingHours.pcap_ISCX.csv  shape=(691343, 60)

  shape=(2827630, 60)
BENIGN                        2271089
DoS Hulk                       230111
PortScan                       158804
DDoS                           128024
DoS GoldenEye                   10292
FTP-Patator                      7935
SSH-Patator                      5897
DoS slowloris                    5796
DoS Sl

## Preprocessing: Rimozione della Colonna Label e Quantile Transform
##### Passaggi critici: rimozione colonne costanti, scaling robusto, clipping dei valori


In [12]:
print("\n" + "=" * 65)
print("STEP 3 — Preprocessing")
print("=" * 65)

FEATURE_COLS = [c for c in train_ben.columns if c != "Label"]

X_train_raw         = to_numeric_clean(train_ben,           FEATURE_COLS)
X_val_raw           = to_numeric_clean(val_ben,              FEATURE_COLS)
X_select_raw        = to_numeric_clean(select_ben,           FEATURE_COLS)
X_calib_raw         = to_numeric_clean(calib_ben,             FEATURE_COLS)
X_test_internal_raw = to_numeric_clean(test_df_internal,      FEATURE_COLS)
X_stress_2017_raw   = to_numeric_clean(stress_test_df_2017,   FEATURE_COLS)

# feature costanti — calcolate SOLO sul train 2018
constant_mask = X_train_raw.nunique() <= 1
const_cols    = X_train_raw.columns[constant_mask].tolist()
print(f"  Feature costanti (da 2018 train): {const_cols}")

for df_ in [X_train_raw, X_val_raw, X_select_raw, X_calib_raw, X_test_internal_raw, X_stress_2017_raw]:
    df_.drop(columns=const_cols, inplace=True)

FEATURE_NAMES = X_train_raw.columns.tolist()

# split continue/binarie — calcolato SOLO sul train 2018
binary_mask = X_train_raw.nunique() <= 2
candidate_binary_cols = X_train_raw.columns[binary_mask].tolist()

BINARY_COLS = []
for col in candidate_binary_cols:
    vals = set(X_train_raw[col].unique())
    if vals <= {0, 1, 0.0, 1.0}:
        BINARY_COLS.append(col)
    else:
        print(f"  ATTENZIONE: '{col}' ha valori {vals}, non {{0,1}} → trattata come continua")

CONTINUOUS_COLS = [c for c in FEATURE_NAMES if c not in BINARY_COLS]
print(f"  Feature continue: {len(CONTINUOUS_COLS)}  |  Feature binarie: {len(BINARY_COLS)}")

# Contratto cross-dataset: stesso ordine e feature binarie realmente compatibili.
assert X_stress_2017_raw.columns.tolist() == FEATURE_NAMES, (
    'Ordine o insieme delle feature 2017 diverso dal training 2018 dopo il preprocessing.'
)
feature_contract_rows = []
for col in BINARY_COLS:
    target_values = X_stress_2017_raw[col]
    invalid_mask = ~target_values.isin([0.0, 1.0])
    invalid_examples = target_values.loc[invalid_mask].drop_duplicates().head(8).tolist()
    feature_contract_rows.append({
        'feature': col,
        'source_contract': '{0,1}',
        'target_invalid_count': int(invalid_mask.sum()),
        'target_invalid_examples': ' | '.join(map(str, invalid_examples)) if invalid_examples else '',
    })
cross_dataset_feature_contract_df = pd.DataFrame(feature_contract_rows)
invalid_binary_contract = cross_dataset_feature_contract_df['target_invalid_count'].sum() if len(cross_dataset_feature_contract_df) else 0
if invalid_binary_contract:
    print('\nCONTRATTO FEATURE BINARIE NON VALIDO')
    print(cross_dataset_feature_contract_df.query('target_invalid_count > 0').to_string(index=False))
    raise ValueError('Il target 2017 contiene valori incompatibili con feature classificate binarie sul source 2018.')
print(f'  Contratto cross-dataset verificato: ordine coerente e {len(BINARY_COLS)} feature binarie compatibili.')

# ── Scaler: fit SOLO sul train 2018, transform su tutto il resto INCLUSO il 2017 ──
scaler = QuantileTransformer(n_quantiles=1000, output_distribution="normal", random_state=42)
scaler.fit(X_train_raw[CONTINUOUS_COLS])

def transform_split(df_raw):
    cont = scaler.transform(df_raw[CONTINUOUS_COLS])
    cont = np.where(np.isfinite(cont), cont, 0.0).astype(np.float32)
    bin_ = df_raw[BINARY_COLS].to_numpy(dtype=np.float32) * 2 - 1
    return np.concatenate([cont, bin_], axis=1)

X_train         = transform_split(X_train_raw)
X_val           = transform_split(X_val_raw)
X_select        = transform_split(X_select_raw)
X_calib         = transform_split(X_calib_raw)
X_test_internal = transform_split(X_test_internal_raw)
X_stress_2017   = transform_split(X_stress_2017_raw)   # ← SOLO transform, scaler già fittato sul 2018

# clipping — bounds calcolati SOLO sul train 2018
n_cont = len(CONTINUOUS_COLS)
clip_lo = np.percentile(X_train[:, :n_cont], 0.1,  axis=0)
clip_hi = np.percentile(X_train[:, :n_cont], 99.9, axis=0)

for arr in (X_train, X_val, X_select, X_calib, X_test_internal, X_stress_2017):
    arr[:, :n_cont] = np.clip(arr[:, :n_cont], clip_lo, clip_hi)

FEATURE_NAMES_FINAL = CONTINUOUS_COLS + BINARY_COLS
INPUT_DIM = len(FEATURE_NAMES_FINAL)
print(f"  Feature finali: {INPUT_DIM}")

# Libera copie tabellari non piu' necessarie; X_stress_2017_raw serve a Option B.
del X_train_raw, X_val_raw, X_select_raw, X_calib_raw, X_test_internal_raw
del train_ben, val_ben, select_ben, calib_ben, remaining_ben_protocol, selectcalib_ben
del trainval_pool, remaining_ben, all_benign_2018, all_attacks_2018, test_holdout_ben
del test_df_internal, stress_test_df_2017
gc.collect()



STEP 3 — Preprocessing
  Feature costanti (da 2018 train): []
  Feature continue: 52  |  Feature binarie: 7
  Contratto cross-dataset verificato: ordine coerente e 7 feature binarie compatibili.
  Feature finali: 59


20

## Loss di ricostruzione

La grid confronta architetture con loss combined.


In [13]:
def combined_loss(y_true, y_pred):
    """
    Loss di training: 0.5·MAE + 0.5·MSE
    MAE  → gradiente costante, robusta
    MSE  → penalizza errori grandi durante il training
    """
    mae = tf.reduce_mean(tf.abs(y_true - y_pred))
    mse = tf.reduce_mean(tf.square(y_true - y_pred))
    return 0.5 * mae + 0.5 * mse


## Funzioni di scoring e metriche

Lo scoring è eseguito a batch per limitare la memoria.


In [14]:
def compute_score_bundle(model, X, batch_size=8192):
    parts = {'mse': [], 'mae': []}
    for start in range(0, len(X), batch_size):
        batch = X[start:start + batch_size]
        pred = model.predict_on_batch(batch)
        diff = batch - pred
        parts['mse'].append(np.mean(np.square(diff), axis=1, dtype=np.float32))
        parts['mae'].append(np.mean(np.abs(diff), axis=1, dtype=np.float32))
    mse = np.concatenate(parts['mse'])
    mae = np.concatenate(parts['mae'])
    return {'mse': mse, 'mae': mae, 'combined': 0.5 * mse + 0.5 * mae}

def compute_score(model, X, mode='mse'):
    scores = compute_score_bundle(model, X)
    if mode not in scores:
        raise ValueError(f'Score mode non supportato: {mode}')
    return scores[mode]

def reconstruction_error(model, X):
    return compute_score(model, X, 'mse')

def binary_metrics_row(dataset, scenario, y_true, preds, scores=None, status='DIAGNOSTICO', note='', threshold=np.nan):
    (tn, fp, fn, tp) = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    row = {'dataset': dataset, 'scenario': scenario, 'status': status, 'threshold': threshold, 'accuracy': accuracy_score(y_true, preds), 'precision': precision_score(y_true, preds, zero_division=0), 'recall': recall_score(y_true, preds, zero_division=0), 'f1': f1_score(y_true, preds, zero_division=0), 'fpr_benign': fp / (fp + tn + 1e-12), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp), 'note': note}
    row['roc_auc'] = roc_auc_score(y_true, scores) if scores is not None else np.nan
    row['pr_auc'] = average_precision_score(y_true, scores) if scores is not None else np.nan
    return row


## Architettura dell'autoencoder


In [15]:

def build_autoencoder_v2(
    input_dim: int, bottleneck: int = 8, dropout: float = 0.1,
    loss: str = "combined", depth: str = "deep",
    noise_std: float = 0.05, use_batchnorm: bool = True,
) -> Model:
    inp = Input(shape=(input_dim,), name="input")
    x = GaussianNoise(noise_std, name="input_noise")(inp) if noise_std > 0 else inp

    x = Dense(256, name="enc_256")(x)
    if use_batchnorm: x = BatchNormalization(name="enc_256_bn")(x)
    x = LeakyReLU(0.1)(x); x = Dropout(dropout)(x)

    x = Dense(128, name="enc_128")(x)
    if use_batchnorm: x = BatchNormalization(name="enc_128_bn")(x)
    x = LeakyReLU(0.1)(x)

    x = Dense(64, name="enc_64")(x)
    if use_batchnorm: x = BatchNormalization(name="enc_64_bn")(x)
    x = LeakyReLU(0.1)(x)

    if depth == "deeper":
        x = Dense(32, name="enc_32")(x)
        if use_batchnorm: x = BatchNormalization(name="enc_32_bn")(x)
        x = LeakyReLU(0.1)(x)

    latent = Dense(bottleneck, name="latent")(x)

    if depth == "deeper":
        x = Dense(32, name="dec_32")(latent); x = LeakyReLU(0.1)(x)
        x = Dense(64, name="dec_64")(x);      x = LeakyReLU(0.1)(x)
    else:
        x = Dense(64, name="dec_64")(latent); x = LeakyReLU(0.1)(x)

    x = Dense(128, name="dec_128")(x); x = LeakyReLU(0.1)(x)
    x = Dense(256, name="dec_256")(x); x = LeakyReLU(0.1)(x)
    out = Dense(input_dim, name="reconstruction")(x)

    model = Model(inputs=inp, outputs=out,
                  name=f"ae_v2_{depth}_bn{bottleneck}_do{int(dropout*10)}_{loss}_noise{noise_std}_bn{int(use_batchnorm)}")
    model.compile(optimizer=Adam(learning_rate=1e-3),
                  loss=(combined_loss if loss == "combined" else "mae"))
    return model


## Calibrazione della soglia F1


In [16]:
def select_f1_threshold(y, s):
    (p, r, t) = precision_recall_curve(y, s)
    f = 2 * p[:-1] * r[:-1] / (p[:-1] + r[:-1] + 1e-12)
    ix = np.flatnonzero(np.isclose(f, f.max()))
    return float(t[ix[np.argmax(r[:-1][ix])]])

def evaluate_calibrated(model, Xcal, ycal, Xev, yev):
    th = select_f1_threshold(ycal, compute_score(model, Xcal))
    z = compute_score(model, Xev)
    pred = (z >= th).astype(np.int8)
    return {'threshold': th, 'f1': f1_score(yev, pred, zero_division=0), 'recall': recall_score(yev, pred, zero_division=0), 'precision': precision_score(yev, pred, zero_division=0), 'roc_auc': roc_auc_score(yev, z), 'pr_auc': average_precision_score(yev, z), 'scores': z, 'predictions': pred}


## Configurazione congelata, protocollo sperimentale e guardrail

La configurazione dell'autoencoder e il seed di retraining sono congelati dai risultati della precedente selezione grid e multi-seed. Questa run non riapre la ricerca degli iperparametri: riaddestra una sola configurazione e concentra la valutazione sul confronto 2018--2017 e sulla domain adaptation. I guardrail restano invariati: seed fissato, early stopping, tetto di epoche, riduzione del learning rate e split disgiunti.


In [17]:
if ACCELERATOR['gpu_available']:
    print(f"Addestramento configurato su GPU: {ACCELERATOR['label']} ({ACCELERATOR['logical_device']})")
else:
    print("ATTENZIONE: TensorFlow non vede una GPU; l'addestramento usera la CPU.")

# Configurazione congelata dalla grid e dal controllo multi-seed storici.
# La nuova run non riapre la ricerca architetturale.
FROZEN_BEST_CFG = {
    'bottleneck': 9,
    'dropout': 0.05,
    'loss': 'combined',
    'depth': 'deep',
    'noise_std': 0.015,
    'use_batchnorm': False,
    'batch_size': 2048,
}
FROZEN_RETRAIN_SEED = 123
BATCH_SIZE = 2048
FINAL_RETRAIN_MAX_EPOCHS = 400
FINAL_RETRAIN_EARLY_STOPPING_PATIENCE = 60

# Nuove ablation indicate dal report: due soli training aggiuntivi.
RUN_STABLE_CORE_MODEL = True
RUN_DRIFT_MASK_COMPACT_MODEL = True
STABLE_CORE_MAX_EPOCHS = 120
STABLE_CORE_PATIENCE = 20
ROBUST_FINETUNE_MAX_EPOCHS = 40
ROBUST_FINETUNE_PATIENCE = 8
ROBUST_FINETUNE_LEARNING_RATE = 1e-4
ROBUST_COMPACTNESS_WEIGHT = 1e-4
ROBUST_RISKY_MASK_PROBABILITY = 0.15
ROBUST_STABLE_MASK_PROBABILITY = 0.02
DRIFT_RISKY_TOP_K = 8
SOURCE_GUARDRAIL_MAX_F1_DROP = 0.02

# Scoring e soglie operative.
SCORE_REFERENCE_SAMPLE = 150_000
SCORE_TOP_K = 4
OPERATIONAL_FPR_LEVELS = (0.05, 0.01, 0.005, 0.001)
EVT_TAIL_QUANTILE = 0.95
HISTORICAL_V32_TARGET_F1 = 0.5349876268724738

class AutoencoderBatchSequence(tf.keras.utils.Sequence):
    """Generatore memory-safe per il training ricostruzione input->input."""

    def __init__(self, X, batch_size=BATCH_SIZE, shuffle=True, seed=SEED):
        super().__init__()
        self.X = X
        self.batch_size = int(batch_size)
        self.shuffle = bool(shuffle)
        self.rng = np.random.default_rng(seed)
        self.indices = np.arange(len(X))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, index):
        selected = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        batch = self.X[selected]
        return batch, batch

    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.indices)

def train_autoencoder(cfg, seed, max_epochs, patience, verbose=2):
    gc.collect()
    set_seed(seed)
    batch_size = int(cfg.get('batch_size', BATCH_SIZE))
    model_cfg = {key: value for key, value in cfg.items() if key != 'batch_size'}
    train_sequence = AutoencoderBatchSequence(X_train, batch_size, True, seed)
    validation_sequence = AutoencoderBatchSequence(X_val, batch_size, False, seed)
    model = build_autoencoder_v2(INPUT_DIM, **model_cfg)
    history = model.fit(
        train_sequence,
        validation_data=validation_sequence,
        epochs=max_epochs,
        verbose=verbose,
        callbacks=[
            EarlyStopping(
                monitor='val_loss',
                patience=patience,
                restore_best_weights=True,
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.7,
                patience=5,
                min_lr=1e-6,
            ),
        ],
    )
    del train_sequence, validation_sequence
    return model, history

# Split interno 2018: ranking 40%, calibrazione 30%, final 30%.
# La stratificazione mantiene il rapporto benigni/attacchi.
all_internal_indices = np.arange(len(y_test_internal_binary))
idx_rank, internal_remaining = train_test_split(
    all_internal_indices,
    test_size=0.60,
    random_state=SEED,
    stratify=y_test_internal_binary,
)
idx_cal, idx_final = train_test_split(
    internal_remaining,
    test_size=0.50,
    random_state=SEED + 1,
    stratify=y_test_internal_binary[internal_remaining],
)

best_cfg = dict(FROZEN_BEST_CFG)
best_seed = int(FROZEN_RETRAIN_SEED)
frozen_model_selection_df = pd.DataFrame([{
    'selection_mode': 'FROZEN_FROM_PRIOR_GRID_MULTI_SEED',
    'config': best_cfg,
    'seed': best_seed,
    'selection_evidence': 'Configurazione congelata prima del protocollo cross-dataset v4.',
}])

print('RICERCA IPERPARAMETRI AE: non rieseguita.')
print(f'Configurazione congelata: {best_cfg}')
print(f'Seed congelato: {best_seed}')
print(
    f'Nuove ablation: stable-core={RUN_STABLE_CORE_MODEL}; '
    f'drift-mask+compactness={RUN_DRIFT_MASK_COMPACT_MODEL}.'
)


Addestramento configurato su GPU: AMD Radeon adapter 0 (/device:GPU:0)
RICERCA IPERPARAMETRI AE: non rieseguita.
Configurazione congelata: {'bottleneck': 9, 'dropout': 0.05, 'loss': 'combined', 'depth': 'deep', 'noise_std': 0.015, 'use_batchnorm': False, 'batch_size': 2048}
Seed congelato: 123
Nuove ablation: stable-core=True; drift-mask+compactness=True.


## Retraining finale dell'autoencoder source congelato

Addestra una sola volta il modello di riferimento sui benigni 2018. Il budget
massimo non obbliga a eseguire tutte le epoche: early stopping ripristina la
migliore validation loss.


In [18]:
print('=' * 100)
print('RETRAINING FINALE DEL BASELINE SOURCE')
print('=' * 100)
print(f'Configurazione: {best_cfg}')
print(f'Seed: {best_seed}')
print(
    f'Budget massimo={FINAL_RETRAIN_MAX_EPOCHS}; '
    f'patience={FINAL_RETRAIN_EARLY_STOPPING_PATIENCE}'
)

retraining_start = time.perf_counter()
best_model, best_history = train_autoencoder(
    best_cfg,
    best_seed,
    max_epochs=FINAL_RETRAIN_MAX_EPOCHS,
    patience=FINAL_RETRAIN_EARLY_STOPPING_PATIENCE,
    verbose=2,
)
retraining_seconds = time.perf_counter() - retraining_start
best_val_epoch = int(np.argmin(best_history.history['val_loss']) + 1)
final_retraining_df = pd.DataFrame([{
    'model': 'baseline_source',
    'config': best_cfg,
    'seed': best_seed,
    'max_epochs': FINAL_RETRAIN_MAX_EPOCHS,
    'patience': FINAL_RETRAIN_EARLY_STOPPING_PATIENCE,
    'epochs_trained': len(best_history.history['loss']),
    'best_val_epoch': best_val_epoch,
    'best_val_loss': float(np.min(best_history.history['val_loss'])),
    'elapsed_minutes': retraining_seconds / 60.0,
}])
print('\nRIEPILOGO RETRAINING')
print(final_retraining_df.round(6).to_string(index=False))


RETRAINING FINALE DEL BASELINE SOURCE
Configurazione: {'bottleneck': 9, 'dropout': 0.05, 'loss': 'combined', 'depth': 'deep', 'noise_std': 0.015, 'use_batchnorm': False, 'batch_size': 2048}
Seed: 123
Budget massimo=400; patience=60
Epoch 1/400
1026/1026 - 8s - loss: 0.2416 - val_loss: 0.0632 - lr: 0.0010 - 8s/epoch - 7ms/step
Epoch 2/400
1026/1026 - 6s - loss: 0.0622 - val_loss: 0.0502 - lr: 0.0010 - 6s/epoch - 6ms/step
Epoch 3/400
1026/1026 - 6s - loss: 0.0514 - val_loss: 0.0606 - lr: 0.0010 - 6s/epoch - 6ms/step
Epoch 4/400
1026/1026 - 6s - loss: 0.0452 - val_loss: 0.0540 - lr: 0.0010 - 6s/epoch - 6ms/step
Epoch 5/400
1026/1026 - 5s - loss: 0.0411 - val_loss: 0.0484 - lr: 0.0010 - 5s/epoch - 5ms/step
Epoch 6/400
1026/1026 - 5s - loss: 0.0379 - val_loss: 0.0523 - lr: 0.0010 - 5s/epoch - 5ms/step
Epoch 7/400
1026/1026 - 5s - loss: 0.0354 - val_loss: 0.0577 - lr: 0.0010 - 5s/epoch - 5ms/step
Epoch 8/400
1026/1026 - 5s - loss: 0.0336 - val_loss: 0.0609 - lr: 0.0010 - 5s/epoch - 5ms/step


## Valutazione baseline CSE-CIC-IDS2018

La soglia primaria massimizza F1 sulla calibrazione etichettata `idx_cal`.
Il final `idx_final` viene usato soltanto dopo il congelamento della soglia.


In [19]:
print('=' * 100)
print('BASELINE SOURCE 2018')
print('=' * 100)

source_cal_scores_baseline = compute_score(best_model, X_test_internal[idx_cal])
source_baseline_threshold = select_f1_threshold(
    y_test_internal_binary[idx_cal],
    source_cal_scores_baseline,
)
source_final_scores_baseline = compute_score(best_model, X_test_internal[idx_final])
source_final_predictions_baseline = (
    source_final_scores_baseline >= source_baseline_threshold
).astype(np.int8)

source_cal_benign = source_cal_scores_baseline[
    y_test_internal_binary[idx_cal] == 0
]
source_benign_below_threshold_pct = (
    100.0 * np.mean(source_cal_benign <= source_baseline_threshold)
)
source_baseline_row = binary_metrics_row(
    'CSE-CIC-IDS2018',
    'baseline_mse_f1_calibrated',
    y_test_internal_binary[idx_final],
    source_final_predictions_baseline,
    source_final_scores_baseline,
    'FINAL_SOURCE_BASELINE',
    'AE one-class; soglia F1 su idx_cal; final separato.',
    source_baseline_threshold,
)
results_2018_baseline_df = pd.DataFrame([source_baseline_row])

MATRIX_COLUMNS = [
    'dataset', 'scenario', 'threshold', 'f1', 'precision', 'recall',
    'roc_auc', 'pr_auc', 'fpr_benign', 'accuracy', 'tn', 'fp', 'fn', 'tp',
]
def result_matrix(dataframe):
    return dataframe[[column for column in MATRIX_COLUMNS if column in dataframe.columns]].round(4)

print(
    f'Soglia={source_baseline_threshold:.8f}; '
    f'benigni calibration sotto soglia={source_benign_below_threshold_pct:.2f}%'
)
print(result_matrix(results_2018_baseline_df).to_string(index=False))


BASELINE SOURCE 2018
Soglia=0.01224825; benigni calibration sotto soglia=83.66%
        dataset                   scenario  threshold     f1  precision  recall  roc_auc  pr_auc  fpr_benign  accuracy     tn    fp    fn     tp
CSE-CIC-IDS2018 baseline_mse_f1_calibrated     0.0122 0.9192     0.8889  0.9516   0.9196  0.8971      0.1633    0.9032 502001 97999 39912 784306


## Protocollo cross-dataset v4: split adaptation, calibration, ranking e final

Il target 2017 viene separato prima degli esperimenti. `adaptation` costruisce
riferimenti senza label; `calibration` sceglie le soglie; `ranking` sceglie
modello e score; `final` viene letto dopo il congelamento.


In [20]:
target_rng = np.random.default_rng(SEED + 4101)
target_order = target_rng.permutation(len(X_stress_2017))
adaptation_end = len(target_order) // 2
cda_adapt_idx = target_order[:adaptation_end]
cda_remaining_idx = target_order[adaptation_end:]

cda_rank_idx, cda_cal_final_idx = train_test_split(
    cda_remaining_idx,
    test_size=2 / 3,
    random_state=SEED + 4102,
    stratify=y_stress_2017_binary[cda_remaining_idx],
)
cda_cal_idx, cda_final_idx = train_test_split(
    cda_cal_final_idx,
    test_size=0.50,
    random_state=SEED + 4103,
    stratify=y_stress_2017_binary[cda_cal_final_idx],
)

cross_dataset_v4_split_df = pd.DataFrame([
    {'split': 'adaptation', 'rows': len(cda_adapt_idx), 'label_role': 'non usate nella costruzione'},
    {'split': 'calibration', 'rows': len(cda_cal_idx), 'label_role': 'soglia F1/F-beta'},
    {'split': 'ranking', 'rows': len(cda_rank_idx), 'label_role': 'selezione candidata'},
    {'split': 'final', 'rows': len(cda_final_idx), 'label_role': 'metriche dopo congelamento'},
])
assert len(set(cda_adapt_idx) & set(cda_cal_idx)) == 0
assert len(set(cda_adapt_idx) & set(cda_rank_idx)) == 0
assert len(set(cda_adapt_idx) & set(cda_final_idx)) == 0
assert len(set(cda_cal_idx) & set(cda_rank_idx)) == 0
assert len(set(cda_cal_idx) & set(cda_final_idx)) == 0
assert len(set(cda_rank_idx) & set(cda_final_idx)) == 0

print('SPLIT CROSS-DATASET V4')
print(cross_dataset_v4_split_df.to_string(index=False))
print('Final 2017 non consultato nelle celle di costruzione e ranking.')


SPLIT CROSS-DATASET V4
      split    rows                  label_role
 adaptation 1413815 non usate nella costruzione
calibration  471272            soglia F1/F-beta
    ranking  471271         selezione candidata
      final  471272  metriche dopo congelamento
Final 2017 non consultato nelle celle di costruzione e ranking.


## Riferimento pseudo-benigno, feature drift e stable-core

K-means viene applicato al latente del solo adaptation split. Il cluster più
numeroso è usato come riferimento pseudo-benigno senza consultare le label.
Le otto feature con drift maggiore formano il gruppo `risky`; le altre il
gruppo `stable-core`.


In [21]:
from sklearn.cluster import MiniBatchKMeans

def predict_latent(model, values, batch_size=8192):
    encoder = Model(model.input, model.get_layer('latent').output)
    output = []
    for start in range(0, len(values), batch_size):
        output.append(encoder.predict_on_batch(values[start:start + batch_size]))
    return np.concatenate(output, axis=0).astype(np.float32, copy=False)

def sample_indices(indices, max_rows, rng):
    indices = np.asarray(indices)
    if len(indices) <= max_rows:
        return indices
    return rng.choice(indices, size=max_rows, replace=False)

# Prima vista target: QT fit su adaptation non etichettato, usato soltanto per
# rendere più stabile il clustering latente.
adaptation_quantile = QuantileTransformer(
    n_quantiles=min(1000, len(cda_adapt_idx)),
    output_distribution='normal',
    random_state=SEED + 4201,
)
adaptation_quantile.fit(
    X_stress_2017_raw[CONTINUOUS_COLS].iloc[cda_adapt_idx]
)
reference_rng = np.random.default_rng(SEED + 4202)
cluster_fit_idx = sample_indices(cda_adapt_idx, 250_000, reference_rng)
target_adapt_continuous_sample = adaptation_quantile.transform(
    X_stress_2017_raw[CONTINUOUS_COLS].iloc[cluster_fit_idx]
).astype(np.float32)
X_target_adaptation_sample = np.concatenate(
    [
        np.clip(target_adapt_continuous_sample, clip_lo, clip_hi),
        X_stress_2017[cluster_fit_idx, n_cont:],
    ],
    axis=1,
).astype(np.float32)

cluster_latent = predict_latent(
    best_model,
    X_target_adaptation_sample,
)
target_kmeans = MiniBatchKMeans(
    n_clusters=2,
    random_state=SEED + 4203,
    batch_size=8192,
    n_init=10,
)
cluster_labels = target_kmeans.fit_predict(cluster_latent)
cluster_counts = np.bincount(cluster_labels, minlength=2)
dominant_cluster = int(np.argmax(cluster_counts))
target_reference_idx = cluster_fit_idx[cluster_labels == dominant_cluster]
target_reference_latent = cluster_latent[cluster_labels == dominant_cluster]

# Ordine di confidence non etichettato: distanza dal centroide + score source.
reference_center = target_reference_latent.mean(axis=0)
reference_distance = np.linalg.norm(
    target_reference_latent - reference_center,
    axis=1,
)
reference_source_score = compute_score(
    best_model,
    X_stress_2017[target_reference_idx],
)
def rank01(values):
    order = np.argsort(values, kind='mergesort')
    ranks = np.empty(len(values), dtype=np.float32)
    ranks[order] = np.arange(len(values), dtype=np.float32)
    return ranks / max(len(values) - 1, 1)

reference_burden = (
    0.5 * rank01(reference_distance)
    + 0.5 * rank01(reference_source_score)
)
target_reference_idx = target_reference_idx[
    np.argsort(reference_burden, kind='mergesort')
]

# Trasformatore definitivo fit sul cluster pseudo-benigno.
pseudobenign_quantile = QuantileTransformer(
    n_quantiles=min(1000, len(target_reference_idx)),
    output_distribution='normal',
    random_state=SEED + 4204,
)
pseudobenign_quantile.fit(
    X_stress_2017_raw[CONTINUOUS_COLS].iloc[target_reference_idx]
)
target_pseudobenign_continuous = pseudobenign_quantile.transform(
    X_stress_2017_raw[CONTINUOUS_COLS]
).astype(np.float32)
X_target_pseudobenign = np.concatenate(
    [
        np.clip(target_pseudobenign_continuous, clip_lo, clip_hi),
        X_stress_2017[:, n_cont:],
    ],
    axis=1,
).astype(np.float32)
del target_adapt_continuous_sample, X_target_adaptation_sample
del target_pseudobenign_continuous, X_stress_2017_raw
gc.collect()

def unsupervised_drift_score(source, target):
    source_mean = source.mean(axis=0)
    target_mean = target.mean(axis=0)
    source_std = source.std(axis=0) + 1e-8
    target_std = target.std(axis=0) + 1e-8
    mean_shift = np.abs(source_mean - target_mean) / np.sqrt(
        (source_std ** 2 + target_std ** 2) / 2.0
    )
    std_shift = np.abs(np.log(target_std / source_std))
    quantiles = np.array([0.05, 0.25, 0.50, 0.75, 0.95])
    quantile_gap = np.mean(
        np.abs(
            np.quantile(source, quantiles, axis=0)
            - np.quantile(target, quantiles, axis=0)
        ),
        axis=0,
    )
    return mean_shift + std_shift + quantile_gap

drift_sample_size = min(200_000, len(target_reference_idx), len(X_train))
source_drift_idx = reference_rng.choice(
    len(X_train),
    size=drift_sample_size,
    replace=False,
)
target_drift_idx = target_reference_idx[:drift_sample_size]
drift_values = unsupervised_drift_score(
    X_train[source_drift_idx],
    X_stress_2017[target_drift_idx],
)
cross_dataset_v4_feature_drift_df = pd.DataFrame({
    'feature': FEATURE_NAMES_FINAL,
    'drift_score': drift_values,
}).sort_values('drift_score', ascending=False).reset_index(drop=True)

RISKY_FEATURE_NAMES = cross_dataset_v4_feature_drift_df.head(
    DRIFT_RISKY_TOP_K
)['feature'].tolist()
RISKY_FEATURE_POSITIONS = np.array(
    [FEATURE_NAMES_FINAL.index(name) for name in RISKY_FEATURE_NAMES],
    dtype=int,
)
STABLE_FEATURE_MASK = np.array(
    [name not in RISKY_FEATURE_NAMES for name in FEATURE_NAMES_FINAL],
    dtype=bool,
)
STABLE_FEATURE_NAMES = [
    name for name, keep in zip(FEATURE_NAMES_FINAL, STABLE_FEATURE_MASK) if keep
]

pseudobenign_reference_diagnostics_df = pd.DataFrame([{
    'cluster_fit_rows': len(cluster_fit_idx),
    'dominant_cluster_rows': len(target_reference_idx),
    'dominant_cluster_fraction': len(target_reference_idx) / len(cluster_fit_idx),
    'diagnostic_benign_fraction_not_used': float(
        np.mean(y_stress_2017_binary[target_reference_idx] == 0)
    ),
    'adaptation_labels_used_for_selection': False,
    'risky_features': ' | '.join(RISKY_FEATURE_NAMES),
}])

print('RIFERIMENTO PSEUDO-BENIGNO')
print(pseudobenign_reference_diagnostics_df.round(4).to_string(index=False))
print('\nTOP FEATURE PER DRIFT NON SUPERVISIONATO')
print(cross_dataset_v4_feature_drift_df.head(15).round(5).to_string(index=False))
print(f'\nStable-core: {STABLE_FEATURE_MASK.sum()} / {INPUT_DIM} feature.')


RIFERIMENTO PSEUDO-BENIGNO
 cluster_fit_rows  dominant_cluster_rows  dominant_cluster_fraction  diagnostic_benign_fraction_not_used  adaptation_labels_used_for_selection                                                                                                                               risky_features
           250000                 177956                     0.7118                                0.853                                 False ECE Flag Count | RST Flag Count | Bwd Packet Length Std | Bwd IAT Std | Idle Std | Active Std | min_seg_size_forward | Fwd Packet Length Std

TOP FEATURE PER DRIFT NON SUPERVISIONATO
              feature  drift_score
       ECE Flag Count     19.21601
       RST Flag Count     19.21601
Bwd Packet Length Std      5.25524
          Bwd IAT Std      4.65399
             Idle Std      4.20019
           Active Std      4.08719
 min_seg_size_forward      3.15946
Fwd Packet Length Std      3.10412
             Idle Max      2.71923
            I

## Training robusto: stable-core e drift-aware masking con compactness

Vengono confrontati due interventi suggeriti dal report. Il modello stable-core
non vede le otto feature a drift più alto. Il modello drift-aware mantiene tutte
le feature, ma maschera più spesso quelle rischiose e viene regolarizzato verso
il centro benigno latente. Entrambi restano one-class.


In [22]:
class FeatureSubsetSequence(tf.keras.utils.Sequence):
    def __init__(self, X, feature_mask, batch_size, shuffle, seed):
        super().__init__()
        self.X = X
        self.feature_mask = np.asarray(feature_mask)
        self.batch_size = int(batch_size)
        self.shuffle = bool(shuffle)
        self.rng = np.random.default_rng(seed)
        self.indices = np.arange(len(X))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, index):
        selected = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        batch = self.X[selected][:, self.feature_mask]
        return batch, batch

    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.indices)

robust_training_rows = []
stable_core_model = None
stable_core_history = None
if RUN_STABLE_CORE_MODEL:
    set_seed(best_seed + 100)
    stable_cfg = {
        key: value for key, value in best_cfg.items() if key != 'batch_size'
    }
    stable_core_model = build_autoencoder_v2(
        int(STABLE_FEATURE_MASK.sum()),
        **stable_cfg,
    )
    stable_train_sequence = FeatureSubsetSequence(
        X_train, STABLE_FEATURE_MASK, BATCH_SIZE, True, best_seed + 100
    )
    stable_val_sequence = FeatureSubsetSequence(
        X_val, STABLE_FEATURE_MASK, BATCH_SIZE, False, best_seed + 100
    )
    stable_start = time.perf_counter()
    stable_core_history = stable_core_model.fit(
        stable_train_sequence,
        validation_data=stable_val_sequence,
        epochs=STABLE_CORE_MAX_EPOCHS,
        verbose=2,
        callbacks=[
            EarlyStopping(
                monitor='val_loss',
                patience=STABLE_CORE_PATIENCE,
                restore_best_weights=True,
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.7,
                patience=5,
                min_lr=1e-6,
            ),
        ],
    )
    robust_training_rows.append({
        'model': 'stable_core',
        'epochs': len(stable_core_history.history['loss']),
        'best_val_loss': float(np.min(stable_core_history.history['val_loss'])),
        'elapsed_minutes': (time.perf_counter() - stable_start) / 60.0,
        'input_features': int(STABLE_FEATURE_MASK.sum()),
        'method': 'rimozione top drift non supervisionato',
    })
    del stable_train_sequence, stable_val_sequence

baseline_center_sample_rng = np.random.default_rng(best_seed + 200)
baseline_center_idx = baseline_center_sample_rng.choice(
    len(X_select),
    size=min(200_000, len(X_select)),
    replace=False,
)
baseline_latent_center = predict_latent(
    best_model,
    X_select[baseline_center_idx],
).mean(axis=0).astype(np.float32)

class DriftAwareCompactSequence(tf.keras.utils.Sequence):
    def __init__(
        self, X, center, fill_values, feature_probabilities, batch_size,
        shuffle, seed, apply_masking,
    ):
        super().__init__()
        self.X = X
        self.center = np.asarray(center, dtype=np.float32)
        self.feature_probabilities = np.asarray(
            feature_probabilities, dtype=np.float32
        )
        self.batch_size = int(batch_size)
        self.shuffle = bool(shuffle)
        self.apply_masking = bool(apply_masking)
        self.rng = np.random.default_rng(seed)
        self.indices = np.arange(len(X))
        self.fill_values = np.asarray(fill_values, dtype=np.float32)
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, index):
        selected = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        clean = self.X[selected]
        corrupted = clean.copy()
        if self.apply_masking:
            mask = self.rng.random(clean.shape) < self.feature_probabilities[None, :]
            corrupted[mask] = np.broadcast_to(
                self.fill_values, clean.shape
            )[mask]
        center_target = np.repeat(
            self.center[None, :], len(clean), axis=0
        )
        return corrupted, (clean, center_target)

    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.indices)

def latent_compactness_loss(center_target, latent_prediction):
    return tf.reduce_mean(
        tf.reduce_sum(
            tf.square(latent_prediction - center_target),
            axis=1,
        )
    )

drift_robust_model = None
drift_robust_history = None
if RUN_DRIFT_MASK_COMPACT_MODEL:
    set_seed(best_seed + 200)
    drift_robust_model = tf.keras.models.clone_model(best_model)
    drift_robust_model.set_weights(best_model.get_weights())
    drift_joint_model = Model(
        drift_robust_model.input,
        [
            drift_robust_model.output,
            drift_robust_model.get_layer('latent').output,
        ],
        name='drift_mask_compact_joint',
    )
    drift_joint_model.compile(
        optimizer=Adam(learning_rate=ROBUST_FINETUNE_LEARNING_RATE),
        loss=[combined_loss, latent_compactness_loss],
        loss_weights=[1.0, ROBUST_COMPACTNESS_WEIGHT],
    )
    feature_mask_probabilities = np.full(
        INPUT_DIM,
        ROBUST_STABLE_MASK_PROBABILITY,
        dtype=np.float32,
    )
    feature_mask_probabilities[RISKY_FEATURE_POSITIONS] = (
        ROBUST_RISKY_MASK_PROBABILITY
    )
    # La mediana di un campione ampio evita una copia temporanea dell'intero
    # train (milioni di righe) senza rendere instabile il valore di masking.
    robust_fill_rng = np.random.default_rng(best_seed + 202)
    robust_fill_idx = robust_fill_rng.choice(
        len(X_train),
        size=min(200_000, len(X_train)),
        replace=False,
    )
    robust_fill_values = np.median(
        X_train[robust_fill_idx],
        axis=0,
    ).astype(np.float32)
    robust_train_sequence = DriftAwareCompactSequence(
        X_train,
        baseline_latent_center,
        robust_fill_values,
        feature_mask_probabilities,
        BATCH_SIZE,
        True,
        best_seed + 200,
        True,
    )
    robust_val_sequence = DriftAwareCompactSequence(
        X_val,
        baseline_latent_center,
        robust_fill_values,
        np.zeros(INPUT_DIM, dtype=np.float32),
        BATCH_SIZE,
        False,
        best_seed + 201,
        False,
    )
    robust_start = time.perf_counter()
    drift_robust_history = drift_joint_model.fit(
        robust_train_sequence,
        validation_data=robust_val_sequence,
        epochs=ROBUST_FINETUNE_MAX_EPOCHS,
        verbose=2,
        callbacks=[
            EarlyStopping(
                monitor='val_loss',
                patience=ROBUST_FINETUNE_PATIENCE,
                restore_best_weights=True,
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=4,
                min_lr=1e-7,
            ),
        ],
    )
    robust_training_rows.append({
        'model': 'drift_mask_compact',
        'epochs': len(drift_robust_history.history['loss']),
        'best_val_loss': float(np.min(drift_robust_history.history['val_loss'])),
        'elapsed_minutes': (time.perf_counter() - robust_start) / 60.0,
        'input_features': INPUT_DIM,
        'method': (
            f'masking risky={ROBUST_RISKY_MASK_PROBABILITY}; '
            f'stable={ROBUST_STABLE_MASK_PROBABILITY}; '
            f'compactness={ROBUST_COMPACTNESS_WEIGHT}'
        ),
    })
    del robust_train_sequence, robust_val_sequence, drift_joint_model

robust_training_df = pd.DataFrame(robust_training_rows)
print('\nRIEPILOGO TRAINING ROBUSTI')
print(robust_training_df.round(6).to_string(index=False))


Epoch 1/120
1026/1026 - 7s - loss: 0.2091 - val_loss: 0.0573 - lr: 0.0010 - 7s/epoch - 6ms/step
Epoch 2/120
1026/1026 - 6s - loss: 0.0553 - val_loss: 0.0622 - lr: 0.0010 - 6s/epoch - 6ms/step
Epoch 3/120
1026/1026 - 6s - loss: 0.0460 - val_loss: 0.0501 - lr: 0.0010 - 6s/epoch - 6ms/step
Epoch 4/120
1026/1026 - 6s - loss: 0.0404 - val_loss: 0.0607 - lr: 0.0010 - 6s/epoch - 6ms/step
Epoch 5/120
1026/1026 - 6s - loss: 0.0364 - val_loss: 0.0570 - lr: 0.0010 - 6s/epoch - 6ms/step
Epoch 6/120
1026/1026 - 6s - loss: 0.0334 - val_loss: 0.0512 - lr: 0.0010 - 6s/epoch - 6ms/step
Epoch 7/120
1026/1026 - 6s - loss: 0.0311 - val_loss: 0.0463 - lr: 0.0010 - 6s/epoch - 6ms/step
Epoch 8/120
1026/1026 - 6s - loss: 0.0295 - val_loss: 0.0586 - lr: 0.0010 - 6s/epoch - 6ms/step
Epoch 9/120
1026/1026 - 6s - loss: 0.0282 - val_loss: 0.0493 - lr: 0.0010 - 6s/epoch - 6ms/step
Epoch 10/120
1026/1026 - 6s - loss: 0.0324 - val_loss: 0.0635 - lr: 0.0010 - 6s/epoch - 6ms/step
Epoch 11/120
1026/1026 - 6s - loss: 0.0

## Score ibrido one-class: ricostruzione, top-k e Mahalanobis latente

Per ogni modello vengono stimati riferimenti benigni separati. Gli score grezzi
sono convertiti in percentili ECDF e combinati senza modificare i pesi del
modello. La distanza Mahalanobis usa covarianza Ledoit-Wolf.


In [23]:
def select_rows(values, feature_mask):
    return values if feature_mask is None else values[:, feature_mask]

def fit_score_reference(model, values, feature_names, risky_names, max_rows, seed):
    rng = np.random.default_rng(seed)
    if len(values) > max_rows:
        values = values[rng.choice(len(values), size=max_rows, replace=False)]
    joint = Model(
        model.input,
        [model.output, model.get_layer('latent').output],
    )
    residual_parts = []
    latent_parts = []
    for start in range(0, len(values), 8192):
        batch = values[start:start + 8192]
        reconstruction, latent = joint.predict_on_batch(batch)
        residual_parts.append(np.square(batch - reconstruction).astype(np.float32))
        latent_parts.append(np.asarray(latent, dtype=np.float32))
    residual = np.concatenate(residual_parts)
    latent = np.concatenate(latent_parts)
    residual_median = np.median(residual, axis=0).astype(np.float32)
    residual_mad = np.median(
        np.abs(residual - residual_median),
        axis=0,
    ).astype(np.float32)
    residual_scale = np.maximum(1.4826 * residual_mad, 1e-6)
    covariance = LedoitWolf().fit(latent)
    latent_center = latent.mean(axis=0).astype(np.float32)
    hard_keep = np.array(
        [name not in risky_names for name in feature_names],
        dtype=bool,
    )
    if not hard_keep.any():
        hard_keep[:] = True
    reference = {
        'residual_median': residual_median,
        'residual_scale': residual_scale,
        'latent_location': covariance.location_.astype(np.float32),
        'latent_precision': covariance.precision_.astype(np.float32),
        'latent_center': latent_center,
        'hard_keep': hard_keep,
    }
    raw = raw_score_components(model, values, reference, SCORE_TOP_K)
    reference['sorted'] = {
        key: np.sort(component.astype(np.float32))
        for key, component in raw.items()
    }
    return reference

def raw_score_components(model, values, reference, top_k):
    joint = Model(
        model.input,
        [model.output, model.get_layer('latent').output],
    )
    output = {
        'mse': [],
        'hardtrim': [],
        'topk': [],
        'mahalanobis': [],
        'center': [],
    }
    k = min(int(top_k), values.shape[1])
    for start in range(0, len(values), 8192):
        batch = values[start:start + 8192]
        reconstruction, latent = joint.predict_on_batch(batch)
        residual = np.square(batch - reconstruction).astype(np.float32)
        robust_z = np.maximum(
            (residual - reference['residual_median'])
            / reference['residual_scale'],
            0.0,
        )
        top_values = np.partition(robust_z, -k, axis=1)[:, -k:]
        latent = np.asarray(latent, dtype=np.float32)
        centered = latent - reference['latent_location']
        mahalanobis = np.einsum(
            'bi,ij,bj->b',
            centered,
            reference['latent_precision'],
            centered,
            optimize=True,
        )
        output['mse'].append(np.mean(residual, axis=1))
        output['hardtrim'].append(
            np.mean(residual[:, reference['hard_keep']], axis=1)
        )
        output['topk'].append(np.mean(top_values, axis=1))
        output['mahalanobis'].append(mahalanobis)
        output['center'].append(
            np.sum(np.square(latent - reference['latent_center']), axis=1)
        )
    return {
        key: np.concatenate(parts).astype(np.float32, copy=False)
        for key, parts in output.items()
    }

def empirical_cdf(values, sorted_reference):
    return (
        np.searchsorted(sorted_reference, values, side='right')
        .astype(np.float32)
        / max(len(sorted_reference), 1)
    )

def percentile_components(raw, reference):
    return {
        key: empirical_cdf(values, reference['sorted'][key])
        for key, values in raw.items()
    }

def ephad_style(prior_anomaly, auxiliary_anomaly, beta):
    prior_normal = np.clip(
        1.0 - np.asarray(prior_anomaly, dtype=np.float32),
        1e-5,
        1.0,
    )
    evidence_normal = np.clip(
        1.0 - np.asarray(auxiliary_anomaly, dtype=np.float32),
        0.0,
        1.0,
    )
    return (
        -np.log(prior_normal)
        - evidence_normal / max(float(beta), 1e-6)
    ).astype(np.float32)

SCORE_RECIPES = [
    {'name': 'mse', 'kind': 'mse'},
    {'name': 'mse_mahal_w025', 'kind': 'linear', 'weights': (0.75, 0.00, 0.25)},
    {'name': 'mse_mahal_w050', 'kind': 'linear', 'weights': (0.50, 0.00, 0.50)},
    {'name': 'mse_topk_w010', 'kind': 'linear', 'weights': (0.90, 0.10, 0.00)},
    {'name': 'mse_topk_w025', 'kind': 'linear', 'weights': (0.75, 0.25, 0.00)},
    {'name': 'tri_hybrid', 'kind': 'linear', 'weights': (0.60, 0.20, 0.20)},
    {'name': 'ephad_b025', 'kind': 'ephad', 'beta': 0.25},
    {'name': 'ephad_b035', 'kind': 'ephad', 'beta': 0.35},
    {'name': 'ephad_b050', 'kind': 'ephad', 'beta': 0.50},
]

def apply_score_recipe(components, recipe):
    if recipe['kind'] == 'mse':
        return components['mse']
    if recipe['kind'] == 'linear':
        mse_weight, topk_weight, mahal_weight = recipe['weights']
        return (
            mse_weight * components['mse']
            + topk_weight * components['topk']
            + mahal_weight * components['mahalanobis']
        ).astype(np.float32)
    if recipe['kind'] == 'ephad':
        auxiliary = (
            0.5 * components['hardtrim']
            + 0.5 * components['mahalanobis']
        )
        return ephad_style(
            components['mse'],
            auxiliary,
            recipe['beta'],
        )
    raise ValueError(f"Ricetta sconosciuta: {recipe}")

model_variants = {
    'baseline_full': {
        'model': best_model,
        'feature_mask': None,
        'feature_names': list(FEATURE_NAMES_FINAL),
    },
}
if stable_core_model is not None:
    model_variants['stable_core'] = {
        'model': stable_core_model,
        'feature_mask': STABLE_FEATURE_MASK,
        'feature_names': list(STABLE_FEATURE_NAMES),
    }
if drift_robust_model is not None:
    model_variants['drift_mask_compact'] = {
        'model': drift_robust_model,
        'feature_mask': None,
        'feature_names': list(FEATURE_NAMES_FINAL),
    }

score_reference_rng = np.random.default_rng(SEED + 5001)
source_reference_idx = score_reference_rng.choice(
    len(X_select),
    size=min(SCORE_REFERENCE_SAMPLE, len(X_select)),
    replace=False,
)
target_score_reference_idx = target_reference_idx[:min(
    SCORE_REFERENCE_SAMPLE,
    len(target_reference_idx),
)]

for position, (variant_name, state) in enumerate(model_variants.items(), start=1):
    print(f'[{position}/{len(model_variants)}] Fit riferimenti score: {variant_name}')
    source_reference_values = select_rows(
        X_select[source_reference_idx],
        state['feature_mask'],
    )
    target_reference_values = select_rows(
        X_target_pseudobenign[target_score_reference_idx],
        state['feature_mask'],
    )
    local_risky = set(state['feature_names']) & set(RISKY_FEATURE_NAMES)
    state['source_reference'] = fit_score_reference(
        state['model'],
        source_reference_values,
        state['feature_names'],
        local_risky,
        SCORE_REFERENCE_SAMPLE,
        SEED + 5100 + position,
    )
    state['target_reference'] = fit_score_reference(
        state['model'],
        target_reference_values,
        state['feature_names'],
        local_risky,
        SCORE_REFERENCE_SAMPLE,
        SEED + 5200 + position,
    )
print('Riferimenti MSE/top-k/Mahalanobis completati.')


[1/3] Fit riferimenti score: baseline_full
[2/3] Fit riferimenti score: stable_core
[3/3] Fit riferimenti score: drift_mask_compact
Riferimenti MSE/top-k/Mahalanobis completati.


## Calibrazione, guardrail e ranking delle candidate v4

Ogni combinazione modello/score riceve una soglia source e target sulla rispettiva
calibrazione. La scelta usa il ranking 2017 ma ammette soltanto candidate con
perdita F1 source non superiore a 0,02.


In [24]:
def score_variant_values(variant_state, full_values, domain, recipe):
    values = select_rows(full_values, variant_state['feature_mask'])
    reference = variant_state[f'{domain}_reference']
    raw = raw_score_components(
        variant_state['model'],
        values,
        reference,
        SCORE_TOP_K,
    )
    components = percentile_components(raw, reference)
    return apply_score_recipe(components, recipe)

def score_variant_components(variant_state, full_values, domain):
    """Calcola le componenti una sola volta e le riusa per tutte le ricette."""
    values = select_rows(full_values, variant_state['feature_mask'])
    reference = variant_state[f'{domain}_reference']
    raw = raw_score_components(
        variant_state['model'],
        values,
        reference,
        SCORE_TOP_K,
    )
    return percentile_components(raw, reference)

def score_metrics(y_true, scores, threshold):
    predictions = (scores >= threshold).astype(np.int8)
    return {
        'predictions': predictions,
        'f1': f1_score(y_true, predictions, zero_division=0),
        'precision': precision_score(y_true, predictions, zero_division=0),
        'recall': recall_score(y_true, predictions, zero_division=0),
        'roc_auc': roc_auc_score(y_true, scores),
        'pr_auc': average_precision_score(y_true, scores),
        'fpr_benign': float(np.mean(predictions[y_true == 0])),
    }

ranking_rows = []
candidate_states = {}
source_baseline_rank_f1 = None

for variant_position, (variant_name, variant_state) in enumerate(
    model_variants.items(),
    start=1,
):
    print(f'\nMODELLO {variant_position}/{len(model_variants)}: {variant_name}')
    print('  Calcolo delle quattro matrici di score (riutilizzate da tutte le ricette)...')
    source_cal_components = score_variant_components(
        variant_state,
        X_test_internal[idx_cal],
        'source',
    )
    source_rank_components = score_variant_components(
        variant_state,
        X_test_internal[idx_rank],
        'source',
    )
    target_cal_components = score_variant_components(
        variant_state,
        X_target_pseudobenign[cda_cal_idx],
        'target',
    )
    target_rank_components = score_variant_components(
        variant_state,
        X_target_pseudobenign[cda_rank_idx],
        'target',
    )
    for recipe in SCORE_RECIPES:
        recipe_name = recipe['name']
        source_cal_scores = apply_score_recipe(source_cal_components, recipe)
        source_rank_scores = apply_score_recipe(source_rank_components, recipe)
        target_cal_scores = apply_score_recipe(target_cal_components, recipe)
        target_rank_scores = apply_score_recipe(target_rank_components, recipe)
        source_threshold = select_f1_threshold(
            y_test_internal_binary[idx_cal],
            source_cal_scores,
        )
        target_threshold = select_f1_threshold(
            y_stress_2017_binary[cda_cal_idx],
            target_cal_scores,
        )
        source_rank_metrics = score_metrics(
            y_test_internal_binary[idx_rank],
            source_rank_scores,
            source_threshold,
        )
        target_rank_metrics = score_metrics(
            y_stress_2017_binary[cda_rank_idx],
            target_rank_scores,
            target_threshold,
        )
        if variant_name == 'baseline_full' and recipe_name == 'mse':
            source_baseline_rank_f1 = source_rank_metrics['f1']
        candidate_name = f'{variant_name}__{recipe_name}'
        row = {
            'candidate': candidate_name,
            'model_variant': variant_name,
            'score_recipe': recipe_name,
            'source_2018_rank_f1': source_rank_metrics['f1'],
            'source_2018_rank_pr_auc': source_rank_metrics['pr_auc'],
            'target_2017_rank_f1': target_rank_metrics['f1'],
            'target_2017_rank_precision': target_rank_metrics['precision'],
            'target_2017_rank_recall': target_rank_metrics['recall'],
            'target_2017_rank_roc_auc': target_rank_metrics['roc_auc'],
            'target_2017_rank_pr_auc': target_rank_metrics['pr_auc'],
            'target_2017_rank_fpr_benign': target_rank_metrics['fpr_benign'],
            'source_threshold': source_threshold,
            'target_threshold': target_threshold,
        }
        ranking_rows.append(row)
        candidate_states[candidate_name] = {
            'variant_name': variant_name,
            'recipe': dict(recipe),
            'source_threshold': source_threshold,
            'target_threshold': target_threshold,
        }
        print(
            f'  {recipe_name:<18} target F1={target_rank_metrics["f1"]:.4f} '
            f'Rec={target_rank_metrics["recall"]:.4f} '
            f'Prec={target_rank_metrics["precision"]:.4f} '
            f'| source F1={source_rank_metrics["f1"]:.4f}'
        )
    del (
        source_cal_components,
        source_rank_components,
        target_cal_components,
        target_rank_components,
    )
    gc.collect()

cross_dataset_v4_ranking_df = pd.DataFrame(ranking_rows)
if source_baseline_rank_f1 is None:
    raise RuntimeError('Baseline source ranking non calcolato.')
cross_dataset_v4_ranking_df['eligible_source_guardrail'] = (
    cross_dataset_v4_ranking_df['source_2018_rank_f1']
    >= source_baseline_rank_f1 - SOURCE_GUARDRAIL_MAX_F1_DROP
)
cross_dataset_v4_ranking_df = cross_dataset_v4_ranking_df.sort_values(
    [
        'eligible_source_guardrail',
        'target_2017_rank_f1',
        'target_2017_rank_pr_auc',
        'target_2017_rank_recall',
        'source_2018_rank_f1',
    ],
    ascending=[False, False, False, False, False],
).reset_index(drop=True)
eligible_candidates = cross_dataset_v4_ranking_df.loc[
    cross_dataset_v4_ranking_df['eligible_source_guardrail']
]
if eligible_candidates.empty:
    raise RuntimeError('Nessuna candidata rispetta il guardrail source.')
selected_v4_name = str(eligible_candidates.iloc[0]['candidate'])
selected_v4_state = candidate_states[selected_v4_name]

print('\nTOP 20 V4 - FINAL NON LETTO')
print(
    cross_dataset_v4_ranking_df[[
        'candidate',
        'target_2017_rank_f1',
        'target_2017_rank_precision',
        'target_2017_rank_recall',
        'target_2017_rank_pr_auc',
        'target_2017_rank_fpr_benign',
        'source_2018_rank_f1',
        'eligible_source_guardrail',
    ]].head(20).round(4).to_string(index=False)
)
print(f'\nCandidata congelata: {selected_v4_name}')



MODELLO 1/3: baseline_full
  Calcolo delle quattro matrici di score (riutilizzate da tutte le ricette)...
  mse                target F1=0.4800 Rec=0.6964 Prec=0.3662 | source F1=0.9191
  mse_mahal_w025     target F1=0.5012 Rec=0.7404 Prec=0.3788 | source F1=0.8991
  mse_mahal_w050     target F1=0.4835 Rec=0.6267 Prec=0.3935 | source F1=0.8730
  mse_topk_w010      target F1=0.4821 Rec=0.6976 Prec=0.3684 | source F1=0.9168
  mse_topk_w025      target F1=0.4863 Rec=0.7059 Prec=0.3709 | source F1=0.9135
  tri_hybrid         target F1=0.5035 Rec=0.7536 Prec=0.3780 | source F1=0.9067
  ephad_b025         target F1=0.4981 Rec=0.7056 Prec=0.3849 | source F1=0.9010
  ephad_b035         target F1=0.4971 Rec=0.7043 Prec=0.3841 | source F1=0.9060
  ephad_b050         target F1=0.4946 Rec=0.7050 Prec=0.3809 | source F1=0.9112

MODELLO 2/3: stable_core
  Calcolo delle quattro matrici di score (riutilizzate da tutte le ricette)...
  mse                target F1=0.5021 Rec=0.6664 Prec=0.4028 | sourc

## Soglie operative: percentile benigno ed EVT Peaks-over-Threshold

Oltre alla soglia F1 supervisionata vengono pre-calibrate soglie senza attacchi
target: percentili del riferimento pseudo-benigno ed EVT sulla coda superiore.
I livelli corrispondono a FPR operativi desiderati.


In [25]:
def fit_evt_threshold(reference_scores, target_fpr, tail_quantile=0.95):
    reference_scores = np.asarray(reference_scores, dtype=np.float64)
    tail_start = float(np.quantile(reference_scores, tail_quantile))
    excess = reference_scores[reference_scores > tail_start] - tail_start
    tail_probability = len(excess) / max(len(reference_scores), 1)
    if len(excess) < 100 or target_fpr >= tail_probability:
        return float(np.quantile(reference_scores, 1.0 - target_fpr)), {
            'evt_used': False,
            'reason': 'coda insufficiente o FPR fuori dalla coda',
        }
    shape, _, scale = genpareto.fit(excess, floc=0.0)
    conditional_cdf = 1.0 - target_fpr / tail_probability
    conditional_cdf = float(np.clip(conditional_cdf, 0.0, 1.0 - 1e-10))
    threshold = tail_start + genpareto.ppf(
        conditional_cdf,
        shape,
        loc=0.0,
        scale=scale,
    )
    if not np.isfinite(threshold):
        return float(np.quantile(reference_scores, 1.0 - target_fpr)), {
            'evt_used': False,
            'reason': 'fit EVT non finito; fallback percentile',
        }
    return float(threshold), {
        'evt_used': True,
        'tail_start': tail_start,
        'tail_probability': tail_probability,
        'shape': float(shape),
        'scale': float(scale),
    }

selected_variant = model_variants[selected_v4_state['variant_name']]
selected_recipe = selected_v4_state['recipe']
selected_target_reference_scores = score_variant_values(
    selected_variant,
    X_target_pseudobenign[target_score_reference_idx],
    'target',
    selected_recipe,
)

operational_threshold_rows = [{
    'policy': 'target_f1_supervised',
    'target_fpr_requested': np.nan,
    'threshold': selected_v4_state['target_threshold'],
    'uses_target_attack_labels': True,
    'evt_used': False,
    'details': 'F1 massimo su cda_cal',
}]
for target_fpr in OPERATIONAL_FPR_LEVELS:
    percentile_threshold = float(np.quantile(
        selected_target_reference_scores,
        1.0 - target_fpr,
    ))
    operational_threshold_rows.append({
        'policy': f'pseudobenign_percentile_fpr_{target_fpr:g}',
        'target_fpr_requested': target_fpr,
        'threshold': percentile_threshold,
        'uses_target_attack_labels': False,
        'evt_used': False,
        'details': f'quantile {1.0 - target_fpr:.4f} del riferimento pseudo-benigno',
    })
    evt_threshold, evt_info = fit_evt_threshold(
        selected_target_reference_scores,
        target_fpr,
        EVT_TAIL_QUANTILE,
    )
    operational_threshold_rows.append({
        'policy': f'evt_pot_fpr_{target_fpr:g}',
        'target_fpr_requested': target_fpr,
        'threshold': evt_threshold,
        'uses_target_attack_labels': False,
        'evt_used': evt_info.get('evt_used', False),
        'details': json.dumps(evt_info, sort_keys=True),
    })

cross_dataset_v4_operational_thresholds_df = pd.DataFrame(
    operational_threshold_rows
)
print('SOGLIE CONGELATE PRIMA DEL FINAL')
print(
    cross_dataset_v4_operational_thresholds_df.round(6).to_string(index=False)
)


SOGLIE CONGELATE PRIMA DEL FINAL
                           policy  target_fpr_requested  threshold  uses_target_attack_labels  evt_used                                                                                                                                                    details
             target_f1_supervised                   NaN   0.810000                       True     False                                                                                                                                      F1 massimo su cda_cal
 pseudobenign_percentile_fpr_0.05                 0.050   0.931333                      False     False                                                                                                             quantile 0.9500 del riferimento pseudo-benigno
                 evt_pot_fpr_0.05                 0.050   0.931333                      False     False                                                                                 {"evt_

## Valutazione finale unica e metriche operative

La candidata è già congelata. Questa è la sola cella v4 che legge `idx_final`
e `cda_final_idx`. Le metriche Recall@FPR calcolate direttamente sul final sono
marcate come diagnostica oracle e non partecipano alla selezione.


In [26]:
from sklearn.metrics import roc_curve

selected_source_final_scores = score_variant_values(
    selected_variant,
    X_test_internal[idx_final],
    'source',
    selected_recipe,
)
selected_target_final_scores = score_variant_values(
    selected_variant,
    X_target_pseudobenign[cda_final_idx],
    'target',
    selected_recipe,
)

# Due baseline dichiarate a priori: zero-shot source-only e adattamento
# pseudo-benigno con modello/score MSE invariati. Servono a separare l'effetto
# dell'adattamento del dominio da quello del nuovo modello/score.
zero_shot_target_scores = compute_score(
    best_model,
    X_stress_2017[cda_final_idx],
)
zero_shot_target_predictions = (
    zero_shot_target_scores >= source_baseline_threshold
).astype(np.int8)
target_adapted_baseline_state = candidate_states['baseline_full__mse']
target_adapted_baseline_scores = score_variant_values(
    model_variants['baseline_full'],
    X_target_pseudobenign[cda_final_idx],
    'target',
    target_adapted_baseline_state['recipe'],
)
target_adapted_baseline_predictions = (
    target_adapted_baseline_scores
    >= target_adapted_baseline_state['target_threshold']
).astype(np.int8)

source_final_predictions_v4 = (
    selected_source_final_scores >= selected_v4_state['source_threshold']
).astype(np.int8)
target_final_predictions_v4 = (
    selected_target_final_scores >= selected_v4_state['target_threshold']
).astype(np.int8)

v4_note = (
    f'Candidata congelata sul ranking: {selected_v4_name}; '
    f'guardrail source drop<={SOURCE_GUARDRAIL_MAX_F1_DROP:.2f}; '
    'riferimento target costruito senza label; soglia primaria F1 su cda_cal.'
)
final_rows = [
    binary_metrics_row(
        'CSE-CIC-IDS2018',
        f'v4_selected_{selected_v4_name}',
        y_test_internal_binary[idx_final],
        source_final_predictions_v4,
        selected_source_final_scores,
        'FINAL_V4_AFTER_RANKING',
        v4_note,
        selected_v4_state['source_threshold'],
    ),
    binary_metrics_row(
        'CIC-IDS2017',
        'v4_baseline_zero_shot_source_only',
        y_stress_2017_binary[cda_final_idx],
        zero_shot_target_predictions,
        zero_shot_target_scores,
        'FINAL_V4_PREDECLARED_BASELINE',
        (
            'Scaler, modello, score MSE e soglia appresi esclusivamente sul '
            'source 2018; nessun target usato per adattamento o calibrazione.'
        ),
        source_baseline_threshold,
    ),
    binary_metrics_row(
        'CIC-IDS2017',
        'v4_baseline_pseudobenign_target_mse',
        y_stress_2017_binary[cda_final_idx],
        target_adapted_baseline_predictions,
        target_adapted_baseline_scores,
        'FINAL_V4_PREDECLARED_BASELINE',
        (
            'Stesso AE baseline; preprocessing/riferimento pseudo-benigno '
            'target e soglia F1 sulla sola cda_cal.'
        ),
        target_adapted_baseline_state['target_threshold'],
    ),
    binary_metrics_row(
        'CIC-IDS2017',
        f'v4_selected_{selected_v4_name}__target_f1_supervised',
        y_stress_2017_binary[cda_final_idx],
        target_final_predictions_v4,
        selected_target_final_scores,
        'FINAL_V4_AFTER_RANKING',
        v4_note,
        selected_v4_state['target_threshold'],
    ),
]

for threshold_row in operational_threshold_rows[1:]:
    predictions = (
        selected_target_final_scores >= threshold_row['threshold']
    ).astype(np.int8)
    final_rows.append(binary_metrics_row(
        'CIC-IDS2017',
        f'v4_selected_{selected_v4_name}__{threshold_row["policy"]}',
        y_stress_2017_binary[cda_final_idx],
        predictions,
        selected_target_final_scores,
        'FINAL_V4_OPERATIONAL_THRESHOLD',
        (
            'Soglia costruita senza label di attacco target; '
            f'FPR richiesto={threshold_row["target_fpr_requested"]}.'
        ),
        threshold_row['threshold'],
    ))

cross_dataset_v4_final_df = pd.DataFrame(final_rows)
selected_target_final_row = cross_dataset_v4_final_df.loc[
    cross_dataset_v4_final_df['scenario'].str.endswith(
        '__target_f1_supervised'
    )
].iloc[0]
cross_dataset_v4_delta_historical_df = pd.DataFrame([{
    'selected_candidate': selected_v4_name,
    'historical_v32_final_f1': HISTORICAL_V32_TARGET_F1,
    'current_v4_final_f1': float(selected_target_final_row['f1']),
    'delta_f1': float(
        selected_target_final_row['f1'] - HISTORICAL_V32_TARGET_F1
    ),
    'comparison_status': (
        'REPORT_ONLY_SAME_HISTORIC_FINAL_REQUIRES_EXTERNAL_CONFIRMATION'
    ),
}])

# Detection rate per famiglia sul final, solo report.
final_attack_labels = np.asarray(y_stress_2017)[cda_final_idx].astype(str)
attack_frame = pd.DataFrame({
    'attack_type': final_attack_labels,
    'detected': target_final_predictions_v4,
})
attack_frame = attack_frame.loc[
    attack_frame['attack_type'].str.strip().str.upper().ne('BENIGN')
]
cross_dataset_v4_per_attack_df = (
    attack_frame.groupby('attack_type', as_index=False)
    .agg(
        total_samples=('detected', 'size'),
        detected_samples=('detected', 'sum'),
    )
)
cross_dataset_v4_per_attack_df['detection_rate'] = (
    cross_dataset_v4_per_attack_df['detected_samples']
    / cross_dataset_v4_per_attack_df['total_samples']
)
cross_dataset_v4_per_attack_df = cross_dataset_v4_per_attack_df.sort_values(
    ['detection_rate', 'total_samples'],
    ascending=[True, False],
).reset_index(drop=True)

# Diagnostica oracle della curva finale: non seleziona soglia o candidata.
final_fpr_curve, final_tpr_curve, _ = roc_curve(
    y_stress_2017_binary[cda_final_idx],
    selected_target_final_scores,
)
recall_at_fpr_rows = []
for requested_fpr in OPERATIONAL_FPR_LEVELS:
    valid = final_fpr_curve <= requested_fpr
    recall_at_fpr_rows.append({
        'requested_fpr': requested_fpr,
        'oracle_recall_at_fpr_final': (
            float(np.max(final_tpr_curve[valid])) if valid.any() else 0.0
        ),
        'status': 'REPORT_ONLY_FINAL_CURVE_NOT_FOR_SELECTION',
    })
cross_dataset_v4_recall_at_fpr_df = pd.DataFrame(recall_at_fpr_rows)

print('RISULTATI FINALI V4')
print(result_matrix(cross_dataset_v4_final_df).to_string(index=False))
print('\nDELTA RISPETTO AL VINCITORE V3.2 STORICO')
print(cross_dataset_v4_delta_historical_df.round(6).to_string(index=False))
print('\nRECALL@FPR - DIAGNOSTICA ORACLE SUL FINAL')
print(cross_dataset_v4_recall_at_fpr_df.round(6).to_string(index=False))
print('\nDETECTION RATE PER ATTACCO')
print(cross_dataset_v4_per_attack_df.round(4).to_string(index=False))


RISULTATI FINALI V4
        dataset                                                                  scenario  threshold     f1  precision  recall  roc_auc  pr_auc  fpr_benign  accuracy     tn     fp    fn     tp
CSE-CIC-IDS2018                                    v4_selected_stable_core__mse_topk_w025     0.8842 0.9173     0.9281  0.9067   0.9242  0.9161      0.0965    0.9054 542115  57885 76891 747327
    CIC-IDS2017                                         v4_baseline_zero_shot_source_only     0.0122 0.3334     0.2001  0.9995   0.6601  0.2690      0.9790    0.2135   7939 370598    48  92687
    CIC-IDS2017                                       v4_baseline_pseudobenign_target_mse     0.7552 0.4783     0.3646  0.6951   0.6809  0.2863      0.2968    0.7016 266170 112367 28271  64464
    CIC-IDS2017              v4_selected_stable_core__mse_topk_w025__target_f1_supervised     0.8100 0.5255     0.4322  0.6703   0.6874  0.3142      0.2157    0.7618 296878  81659 30575  62160
    CIC-IDS2017

## Diagnostica finale delle feature e dei falsi positivi

La Random Forest resta un controllo supervisionato separato. L'attribuzione
one-class confronta il residuo medio per feature fra benigni e attacchi senza
modificare il modello o la soglia.


In [27]:
from sklearn.ensemble import RandomForestClassifier

diagnostic_rng = np.random.default_rng(SEED + 7001)

def bounded_sample(indices, max_rows, rng):
    indices = np.asarray(indices)
    if len(indices) <= max_rows:
        return indices
    return rng.choice(indices, size=max_rows, replace=False)

# RF diagnostica: 100k per classe su idx_cal, valutazione idx_rank.
rf_fit_parts = []
for class_value in (0, 1):
    class_indices = idx_cal[
        y_test_internal_binary[idx_cal] == class_value
    ]
    rf_fit_parts.append(
        bounded_sample(class_indices, 100_000, diagnostic_rng)
    )
rf_fit_idx = np.concatenate(rf_fit_parts)
diagnostic_rng.shuffle(rf_fit_idx)
rf_diagnostic = RandomForestClassifier(
    n_estimators=300,
    max_features='sqrt',
    min_samples_leaf=2,
    class_weight='balanced_subsample',
    n_jobs=-1,
    random_state=SEED,
)
rf_diagnostic.fit(
    X_test_internal[rf_fit_idx],
    y_test_internal_binary[rf_fit_idx],
)
rf_rank_predictions = rf_diagnostic.predict(X_test_internal[idx_rank])
rf_diagnostic_metrics_df = pd.DataFrame([{
    'purpose': 'diagnostica supervisionata; non modifica AE o policy',
    'fit_samples': len(rf_fit_idx),
    'ranking_f1': f1_score(
        y_test_internal_binary[idx_rank],
        rf_rank_predictions,
        zero_division=0,
    ),
    'ranking_precision': precision_score(
        y_test_internal_binary[idx_rank],
        rf_rank_predictions,
        zero_division=0,
    ),
    'ranking_recall': recall_score(
        y_test_internal_binary[idx_rank],
        rf_rank_predictions,
        zero_division=0,
    ),
}])
rf_feature_importance_df = pd.DataFrame({
    'feature': FEATURE_NAMES_FINAL,
    'importance': rf_diagnostic.feature_importances_,
}).sort_values('importance', ascending=False).reset_index(drop=True)

# Attribuzione AE baseline sul final 2018: massimo 100k per classe.
xai_parts = []
for class_value in (0, 1):
    class_indices = idx_final[
        y_test_internal_binary[idx_final] == class_value
    ]
    xai_parts.append(
        bounded_sample(class_indices, 100_000, diagnostic_rng)
    )
xai_idx = np.concatenate(xai_parts)
benign_residual_sum = np.zeros(INPUT_DIM, dtype=np.float64)
attack_residual_sum = np.zeros(INPUT_DIM, dtype=np.float64)
benign_count = 0
attack_count = 0
for start in range(0, len(xai_idx), 8192):
    batch_idx = xai_idx[start:start + 8192]
    batch = X_test_internal[batch_idx]
    reconstruction = best_model.predict_on_batch(batch)
    residual = np.square(batch - reconstruction).astype(np.float64)
    labels = y_test_internal_binary[batch_idx]
    benign_mask = labels == 0
    attack_mask = labels == 1
    if benign_mask.any():
        benign_residual_sum += residual[benign_mask].sum(axis=0)
        benign_count += int(benign_mask.sum())
    if attack_mask.any():
        attack_residual_sum += residual[attack_mask].sum(axis=0)
        attack_count += int(attack_mask.sum())

ae_feature_attribution_2018_df = pd.DataFrame({
    'feature': FEATURE_NAMES_FINAL,
    'mean_squared_residual_benign': (
        benign_residual_sum / max(benign_count, 1)
    ),
    'mean_squared_residual_attack': (
        attack_residual_sum / max(attack_count, 1)
    ),
})
ae_feature_attribution_2018_df['attack_minus_benign'] = (
    ae_feature_attribution_2018_df['mean_squared_residual_attack']
    - ae_feature_attribution_2018_df['mean_squared_residual_benign']
)
ae_feature_attribution_2018_df['drift_score'] = (
    ae_feature_attribution_2018_df['feature']
    .map(
        cross_dataset_v4_feature_drift_df.set_index('feature')['drift_score']
    )
)
ae_feature_attribution_2018_df['importance_drift_risk'] = (
    ae_feature_attribution_2018_df['attack_minus_benign']
    * ae_feature_attribution_2018_df['drift_score']
)
ae_feature_attribution_2018_df = ae_feature_attribution_2018_df.sort_values(
    'attack_minus_benign',
    ascending=False,
).reset_index(drop=True)

print('RANDOM FOREST DIAGNOSTICA')
print(rf_diagnostic_metrics_df.round(4).to_string(index=False))
print('\nTOP FEATURE RF')
print(rf_feature_importance_df.head(15).round(5).to_string(index=False))
print('\nTOP ATTRIBUZIONE AE')
print(ae_feature_attribution_2018_df.head(15).round(6).to_string(index=False))


RANDOM FOREST DIAGNOSTICA
                                             purpose  fit_samples  ranking_f1  ranking_precision  ranking_recall
diagnostica supervisionata; non modifica AE o policy       200000      0.9739             0.9926          0.9558

TOP FEATURE RF
                    feature  importance
Total Length of Fwd Packets     0.06963
     Init_Win_bytes_forward     0.06267
      Fwd Packet Length Max     0.05794
    Init_Win_bytes_backward     0.04756
     Fwd Packet Length Mean     0.04397
          Fwd Header Length     0.03666
       min_seg_size_forward     0.03560
               Fwd IAT Mean     0.02944
              Flow Duration     0.02805
              Fwd IAT Total     0.02766
               Flow IAT Min     0.02634
                Fwd IAT Max     0.02613
               Flow Bytes/s     0.02612
                Fwd IAT Std     0.02608
              Flow IAT Mean     0.02578

TOP ATTRIBUZIONE AE
                    feature  mean_squared_residual_benign  mean_squared

## Esportazione del protocollo v4 e tracciabilità

Salva soltanto gli artefatti del protocollo compatto corrente in una cartella
dedicata, evitando di confonderli con CSV delle ablation storiche.


In [28]:
V4_EXPORT_DIR = Path('Artefatti') / 'final_protocol_v4'
V4_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

research_report_assessment_df = pd.DataFrame([
    {
        'recommendation': 'source-only zero-shot baseline',
        'decision': 'IMPLEMENTED',
        'implementation': 'scaler, AE, score e soglia appresi solo sul source 2018',
    },
    {
        'recommendation': 'stable feature space',
        'decision': 'IMPLEMENTED',
        'implementation': 'stable-core AE senza top-8 feature per drift non supervisionato',
    },
    {
        'recommendation': 'targeted stochastic feature masking',
        'decision': 'IMPLEMENTED',
        'implementation': 'masking 15% risky, 2% stable, fill con mediana source',
    },
    {
        'recommendation': 'one-class compactness',
        'decision': 'IMPLEMENTED',
        'implementation': 'fine-tuning dal baseline con centro latente benigno congelato',
    },
    {
        'recommendation': 'hybrid reconstruction + latent score',
        'decision': 'IMPLEMENTED',
        'implementation': 'MSE, top-k robusto, Mahalanobis Ledoit-Wolf, EPHAD-style',
    },
    {
        'recommendation': 'EVT and Recall@FPR',
        'decision': 'IMPLEMENTED',
        'implementation': 'POT-GPD e soglie percentile a FPR predefinito',
    },
    {
        'recommendation': 'BatchNorm recalibration',
        'decision': 'NOT_APPLICABLE',
        'implementation': 'configurazione congelata usa BatchNorm=False',
    },
    {
        'recommendation': 'DeepCORAL/DANN',
        'decision': 'REMOVED_AS_COMPLETED_ABLATION',
        'implementation': 'gia valutati e inferiori; non duplicati nella run v4',
    },
    {
        'recommendation': 'DAGMM/MemAE',
        'decision': 'DEFERRED',
        'implementation': 'cambio di modello fuori dal confronto AE one-class principale',
    },
])

v4_config_df = pd.DataFrame([{
    'frozen_best_config': str(FROZEN_BEST_CFG),
    'frozen_seed': FROZEN_RETRAIN_SEED,
    'risky_top_k': DRIFT_RISKY_TOP_K,
    'risky_mask_probability': ROBUST_RISKY_MASK_PROBABILITY,
    'stable_mask_probability': ROBUST_STABLE_MASK_PROBABILITY,
    'compactness_weight': ROBUST_COMPACTNESS_WEIGHT,
    'score_top_k': SCORE_TOP_K,
    'source_guardrail_max_f1_drop': SOURCE_GUARDRAIL_MAX_F1_DROP,
    'operational_fpr_levels': str(OPERATIONAL_FPR_LEVELS),
    'evt_tail_quantile': EVT_TAIL_QUANTILE,
    'selected_candidate': selected_v4_name,
}])

v4_exports = {
    'research_report_assessment': research_report_assessment_df,
    'protocol_config': v4_config_df,
    'protocol_splits': cross_dataset_v4_split_df,
    'baseline_retraining': final_retraining_df,
    'baseline_2018': results_2018_baseline_df,
    'pseudobenign_reference_diagnostics': pseudobenign_reference_diagnostics_df,
    'feature_drift': cross_dataset_v4_feature_drift_df,
    'robust_training': robust_training_df,
    'candidate_ranking': cross_dataset_v4_ranking_df,
    'operational_thresholds': cross_dataset_v4_operational_thresholds_df,
    'final_results': cross_dataset_v4_final_df,
    'delta_historical_v32': cross_dataset_v4_delta_historical_df,
    'recall_at_fpr': cross_dataset_v4_recall_at_fpr_df,
    'per_attack_2017': cross_dataset_v4_per_attack_df,
    'rf_diagnostic_metrics': rf_diagnostic_metrics_df,
    'rf_feature_importance': rf_feature_importance_df,
    'ae_feature_attribution_2018': ae_feature_attribution_2018_df,
    'feature_contract': cross_dataset_feature_contract_df,
}
for export_name, dataframe in v4_exports.items():
    dataframe.to_csv(
        V4_EXPORT_DIR / f'{export_name}.csv',
        index=False,
    )

print('ARTEFATTI V4 SALVATI')
print(f'Cartella: {V4_EXPORT_DIR.resolve()}')
print(f'Tabelle: {len(v4_exports)}')


ARTEFATTI V4 SALVATI
Cartella: D:\Users\Anton\Desktop\Progetti VsCode\Codex\Rilevamento di Anomalie del Traffico di Rete con Autoencoder\Rilevamento-di-Anomalie-del-Traffico-di-Rete-con-Autoencoder\Artefatti\final_protocol_v4
Tabelle: 18


## Figure del protocollo v4 per la tesi

Genera figure dalla run corrente in `Tesi/figures_v4`. Il file LaTeX non viene
modificato automaticamente: sarà aggiornato soltanto dopo aver verificato i
risultati prodotti dalla nuova run.


In [29]:
V4_FIGURE_DIR = Path('Tesi') / 'figures_v4'
V4_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
figure_manifest_rows = []

def save_v4_figure(figure, filename, caption):
    path = V4_FIGURE_DIR / filename
    figure.savefig(
        path,
        dpi=300,
        bbox_inches='tight',
        facecolor='white',
    )
    plt.close(figure)
    figure_manifest_rows.append({
        'filename': filename,
        'caption': caption,
        'path': str(path.resolve()),
    })
    print(f'[OK] {filename}')

# Curve training.
figure, axis = plt.subplots(figsize=(9, 5))
axis.plot(best_history.history['loss'], label='baseline train')
axis.plot(best_history.history['val_loss'], label='baseline validation')
if stable_core_history is not None:
    axis.plot(
        stable_core_history.history['val_loss'],
        label='stable-core validation',
    )
if drift_robust_history is not None:
    axis.plot(
        drift_robust_history.history['val_loss'],
        label='drift-mask compact validation objective',
    )
axis.set(
    xlabel='Epoca',
    ylabel='Loss',
    title='Training baseline e varianti robuste',
)
axis.legend()
save_v4_figure(
    figure,
    '01_training_varianti_robuste.png',
    'Confronto delle curve di training e validation.',
)

# Ranking.
ranking_plot = cross_dataset_v4_ranking_df.head(15).sort_values(
    'target_2017_rank_f1'
)
figure, axis = plt.subplots(figsize=(11, 7))
axis.barh(
    ranking_plot['candidate'],
    ranking_plot['target_2017_rank_f1'],
    color='#2563eb',
)
axis.axvline(
    HISTORICAL_V32_TARGET_F1,
    color='#dc2626',
    linestyle='--',
    label='F1 final storico v3.2 (solo riferimento)',
)
axis.set(
    xlabel='F1 sul ranking CIC-IDS2017',
    ylabel='Candidata',
    title='Top candidate protocollo v4 prima del final',
)
axis.legend()
save_v4_figure(
    figure,
    '02_ranking_candidate_v4.png',
    'Classifica delle candidate v4 sul ranking separato.',
)

# Policy di soglia sul final target.
target_final_plot = cross_dataset_v4_final_df.loc[
    cross_dataset_v4_final_df['dataset'].eq('CIC-IDS2017')
].copy()
figure, axis = plt.subplots(figsize=(12, 6))
x_positions = np.arange(len(target_final_plot))
width = 0.25
axis.bar(
    x_positions - width,
    target_final_plot['f1'],
    width,
    label='F1',
)
axis.bar(
    x_positions,
    target_final_plot['precision'],
    width,
    label='Precision',
)
axis.bar(
    x_positions + width,
    target_final_plot['recall'],
    width,
    label='Recall',
)
axis.set_xticks(x_positions)
axis.set_xticklabels(
    target_final_plot['scenario'].str.split('__').str[-1],
    rotation=45,
    ha='right',
)
axis.set(
    ylabel='Valore',
    title='Soglie supervisionate e operative sul final CIC-IDS2017',
)
axis.legend()
save_v4_figure(
    figure,
    '03_policy_soglia_final_2017.png',
    'Confronto delle policy di soglia sul final target.',
)

# Detection rate per attacco.
attack_plot = cross_dataset_v4_per_attack_df.sort_values('detection_rate')
figure, axis = plt.subplots(
    figsize=(10, max(5, 0.38 * len(attack_plot) + 1))
)
axis.barh(
    attack_plot['attack_type'],
    attack_plot['detection_rate'],
    color='#7c3aed',
)
axis.set(
    xlabel='Detection rate',
    ylabel='Famiglia di attacco',
    title='Rilevamento per attacco CIC-IDS2017 - v4 selezionato',
    xlim=(0, 1),
)
save_v4_figure(
    figure,
    '04_detection_rate_attacchi_2017.png',
    'Detection rate per famiglia di attacco del metodo v4 selezionato.',
)

# Feature drift.
drift_plot = cross_dataset_v4_feature_drift_df.head(20).sort_values(
    'drift_score'
)
figure, axis = plt.subplots(figsize=(10, 8))
axis.barh(
    drift_plot['feature'],
    drift_plot['drift_score'],
    color='#d97706',
)
axis.set(
    xlabel='Drift score',
    ylabel='Feature',
    title='Feature con maggiore drift pseudo-benigno 2018→2017',
)
save_v4_figure(
    figure,
    '05_feature_drift_pseudobenigno.png',
    'Drift non supervisionato delle feature.',
)

v4_figure_manifest_df = pd.DataFrame(figure_manifest_rows)
v4_figure_manifest_df.to_csv(
    V4_FIGURE_DIR / 'manifest_figure_v4.csv',
    index=False,
)
print(
    f'Esportazione completata: {len(v4_figure_manifest_df)} figure in '
    f'{V4_FIGURE_DIR.resolve()}'
)


[OK] 01_training_varianti_robuste.png
[OK] 02_ranking_candidate_v4.png
[OK] 03_policy_soglia_final_2017.png


C:\Users\Anton\AppData\Local\Temp\ipykernel_17244\2910471937.py:7: UserWarning: Glyph 65533 (\N{REPLACEMENT CHARACTER}) missing from current font.
  figure.savefig(


[OK] 04_detection_rate_attacchi_2017.png
[OK] 05_feature_drift_pseudobenigno.png
Esportazione completata: 5 figure in D:\Users\Anton\Desktop\Progetti VsCode\Codex\Rilevamento di Anomalie del Traffico di Rete con Autoencoder\Rilevamento-di-Anomalie-del-Traffico-di-Rete-con-Autoencoder\Tesi\figures_v4


## Protocollo v5 retrospettivo congelato e protezione degli holdout

La v5 usa soltanto adaptation, calibration e ranking per costruzione e
selezione. Il final v4 già osservato non guida alcuna scelta: l'eventuale
confronto sullo stesso final è marcato come retrospettivo e richiede conferma
su un nuovo holdout o un terzo dataset.


In [30]:
# V5_RETRO_CELL_1_CONFIG
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import StratifiedKFold

V5_SEED = 9123
V5_RETAIN_FRACTION = 0.55
V5_REFERENCE_BUILD_FRACTION = 0.60
V5_TRANSFORM_BATCH_SIZE = 100_000
V5_INFERENCE_BATCH_SIZE = 8192
V5_REFERENCE_FIT_MAX_ROWS = 100_000
V5_GMM_COMPONENTS = (2, 4, 6)
V5_GMM_REG_COVAR = 1e-4
V5_GMM_N_INIT = 2
V5_GMM_MAX_ITER = 200
V5_GMM_TRIM_FRACTION = 0.05
V5_SOURCE_CAL_MAX_ROWS = 600_000
V5_SOURCE_RANK_MAX_ROWS = 800_000
V5_SOURCE_MAX_F1_DROP = 0.01
V5_TARGET_CANDIDATE_MAX_SOURCE_RECALL_DROP = 0.05
V5_INTRA_MAX_RECALL_DROP = 0.02
V5_SOURCE_MAX_FPR_INCREASE = 0.02
V5_TARGET_RECALL_FLOOR = 0.68
V5_TARGET_FPR_CEILING = 0.25
V5_NEAR_BEST_F1_MARGIN = 0.002
V5_F2_MAX_CALIBRATION_FPR = 0.30
V5_RUN_LEGACY_FINAL_REPORT = True
V5_LEGACY_FINAL_STATUS = (
    'LEGACY_FINAL_POST_HOC_PROTOCOL_REQUIRES_EXTERNAL_CONFIRMATION'
)
V5_HISTORICAL_TARGET_F1 = 0.5349876268724738
V5_HISTORICAL_TARGET_RECALL = 0.6772308190003774
V5_HISTORICAL_TARGET_FPR = 0.20934545368088192
V5_HISTORICAL_TARGET_FPR_CEILING = V5_HISTORICAL_TARGET_FPR + 0.02
V5_SOURCE_BASELINE_F1 = 0.9191859705377542
V5_SOURCE_BASELINE_RECALL = 0.9515759180216884
V5_TARGET_CANDIDATE_SOURCE_F1_FLOOR = (
    V5_SOURCE_BASELINE_F1 - V5_SOURCE_MAX_F1_DROP
)
V5_TARGET_CANDIDATE_SOURCE_RECALL_FLOOR = (
    V5_SOURCE_BASELINE_RECALL
    - V5_TARGET_CANDIDATE_MAX_SOURCE_RECALL_DROP
)

V5_CANDIDATE_RECIPES = [
    {
        'name': 'full_mse_control',
        'kind': 'single',
        'key': 'full_mse',
    },
    {'name': 'full_topk_control', 'kind': 'single', 'key': 'full'},
    {'name': 'stable_control', 'kind': 'single', 'key': 'stable'},
    {'name': 'soft_full', 'kind': 'single', 'key': 'soft'},
    {
        'name': 'ensemble_stable_w025',
        'kind': 'convex',
        'weights': {'stable': 0.25, 'full': 0.75},
    },
    {
        'name': 'ensemble_stable_w050',
        'kind': 'convex',
        'weights': {'stable': 0.50, 'full': 0.50},
    },
    {
        'name': 'ensemble_stable_w075',
        'kind': 'convex',
        'weights': {'stable': 0.75, 'full': 0.25},
    },
    {
        'name': 'ensemble_stable_soft_w050',
        'kind': 'convex',
        'weights': {'stable': 0.50, 'soft': 0.50},
    },
    {
        'name': 'ensemble_density',
        'kind': 'convex',
        'weights': {'stable': 0.55, 'full': 0.25, 'gmm': 0.20},
    },
    {
        'name': 'gated_full_rescue_w030',
        'kind': 'gated_rescue',
        'base': 'stable',
        'rescue': 'full',
        'weight': 0.30,
    },
]

required_v5_variables = [
    'np',
    'pd',
    'gc',
    'Path',
    'Model',
    'QuantileTransformer',
    'plt',
    'best_model',
    'stable_core_model',
    'X_select',
    'X_test_internal',
    'X_stress_2017',
    'y_test_internal_binary',
    'y_stress_2017_binary',
    'y_stress_2017',
    'idx_cal',
    'idx_rank',
    'cda_adapt_idx',
    'cda_cal_idx',
    'cda_rank_idx',
    'target_reference_idx',
    'cluster_fit_idx',
    'cluster_labels',
    'dominant_cluster',
    'source_reference_idx',
    'STABLE_FEATURE_MASK',
    'FEATURE_NAMES_FINAL',
    'INPUT_DIM',
    'n_cont',
    'clip_lo',
    'clip_hi',
    'SCORE_TOP_K',
    'cross_dataset_v4_feature_drift_df',
    'stress_file_ids_2017',
    'stress_file_names_2017',
    'predict_latent',
    'compute_score',
    'empirical_cdf',
    'score_metrics',
    'binary_metrics_row',
    'result_matrix',
]
missing_v5_variables = [
    name for name in required_v5_variables if name not in globals()
]
if missing_v5_variables:
    raise RuntimeError(
        'Eseguire prima tutte le celle v4. Variabili mancanti: '
        f'{missing_v5_variables}'
    )
if stable_core_model is None:
    raise RuntimeError('La v5 richiede il ramo stable-core della v4.')

v5_adapt_idx = np.asarray(cda_adapt_idx, dtype=np.int64).copy()
v5_target_cal_idx = np.asarray(cda_cal_idx, dtype=np.int64).copy()
v5_target_rank_idx = np.asarray(cda_rank_idx, dtype=np.int64).copy()
v5_source_cal_idx = np.asarray(idx_cal, dtype=np.int64).copy()
v5_source_rank_idx = np.asarray(idx_rank, dtype=np.int64).copy()

assert np.intersect1d(
    v5_adapt_idx,
    v5_target_cal_idx,
    assume_unique=True,
).size == 0
assert np.intersect1d(
    v5_adapt_idx,
    v5_target_rank_idx,
    assume_unique=True,
).size == 0
assert np.intersect1d(
    v5_target_cal_idx,
    v5_target_rank_idx,
    assume_unique=True,
).size == 0
assert len(V5_CANDIDATE_RECIPES) == 10

v5_protocol_config_df = pd.DataFrame([{
    'protocol': 'v5_retro_frozen',
    'seed': V5_SEED,
    'retain_fraction': V5_RETAIN_FRACTION,
    'reference_build_fraction': V5_REFERENCE_BUILD_FRACTION,
    'gmm_components': str(V5_GMM_COMPONENTS),
    'gmm_covariance': 'diag',
    'candidate_count': len(V5_CANDIDATE_RECIPES),
    'candidate_names': ' | '.join(
        recipe['name'] for recipe in V5_CANDIDATE_RECIPES
    ),
    'target_recall_floor': V5_TARGET_RECALL_FLOOR,
    'target_fpr_ceiling': V5_TARGET_FPR_CEILING,
    'target_candidate_max_source_recall_drop': (
        V5_TARGET_CANDIDATE_MAX_SOURCE_RECALL_DROP
    ),
    'intra_max_recall_drop': V5_INTRA_MAX_RECALL_DROP,
    'historical_target_f1': V5_HISTORICAL_TARGET_F1,
    'historical_target_recall': V5_HISTORICAL_TARGET_RECALL,
    'historical_target_fpr_ceiling': (
        V5_HISTORICAL_TARGET_FPR_CEILING
    ),
    'target_candidate_source_f1_floor': (
        V5_TARGET_CANDIDATE_SOURCE_F1_FLOOR
    ),
    'target_candidate_source_recall_floor': (
        V5_TARGET_CANDIDATE_SOURCE_RECALL_FLOOR
    ),
    'legacy_final_status': V5_LEGACY_FINAL_STATUS,
}])

print('PROTOCOLLO V5 CONGELATO')
print(v5_protocol_config_df.to_string(index=False))
print('Nessun indice final è usato nelle celle di build/calibration/ranking.')


PROTOCOLLO V5 CONGELATO
       protocol  seed  retain_fraction  reference_build_fraction gmm_components gmm_covariance  candidate_count                                                                                                                                                                                                candidate_names  target_recall_floor  target_fpr_ceiling  target_candidate_max_source_recall_drop  intra_max_recall_drop  historical_target_f1  historical_target_recall  historical_target_fpr_ceiling  target_candidate_source_f1_floor  target_candidate_source_recall_floor                                           legacy_final_status
v5_retro_frozen  9123             0.55                       0.6      (2, 4, 6)           diag               10 full_mse_control | full_topk_control | stable_control | soft_full | ensemble_stable_w025 | ensemble_stable_w050 | ensemble_stable_w075 | ensemble_stable_soft_w050 | ensemble_density | gated_full_rescue_w030                 0.68

## Purificazione consensuale del riferimento pseudo-benigno

Il cluster dominante viene rifiltrato senza label tramite consenso fra
distanza latente robusta, MSE full e MSE stable. Il cluster è separato fra
build ed ECDF prima del filtro; un secondo QuantileTransformer, applicato a
chunk su memmap dal target source-scaled pristine, riallinea il dominio
preservando i buffer v4.


In [31]:
# V5_RETRO_CELL_2_PURIFICATION
if globals().get('V5_TARGET_TRANSFORM_STATE') is not None:
    raise RuntimeError(
        'La trasformazione target v5 è già stata avviata in questo kernel. '
        'Per evitare una doppia trasformazione, riavviare il kernel ed '
        'eseguire il notebook in ordine.'
    )


def v5_rank01(values):
    values = np.asarray(values)
    order = np.argsort(values, kind='mergesort')
    sorted_values = values[order]
    group_starts = np.r_[
        0,
        np.flatnonzero(sorted_values[1:] != sorted_values[:-1]) + 1,
    ]
    group_ends = np.r_[group_starts[1:], len(values)]
    sorted_ranks = np.empty(len(values), dtype=np.float32)
    for start, end in zip(group_starts, group_ends):
        sorted_ranks[start:end] = 0.5 * (start + end - 1)
    ranks = np.empty(len(values), dtype=np.float32)
    ranks[order] = sorted_ranks
    return ranks / max(len(values) - 1, 1)


# Ricostruisce l'ordine originale: gli array diagnostici v4 non erano stati
# riordinati insieme a target_reference_idx.
v5_original_reference_idx = np.asarray(cluster_fit_idx)[
    np.asarray(cluster_labels) == int(dominant_cluster)
].astype(np.int64)
assert len(v5_original_reference_idx) == len(target_reference_idx)
assert np.array_equal(
    np.sort(v5_original_reference_idx),
    np.sort(np.asarray(target_reference_idx)),
)
assert np.setdiff1d(
    v5_original_reference_idx,
    v5_adapt_idx,
    assume_unique=False,
).size == 0

# Split prima della purificazione: la distribuzione ECDF non partecipa alla
# stima del centro, delle scale, dei rank o del cutoff di burden.
v5_reference_rng = np.random.default_rng(V5_SEED + 101)
v5_original_permutation = v5_reference_rng.permutation(
    v5_original_reference_idx
)
v5_original_build_rows = int(np.floor(
    V5_REFERENCE_BUILD_FRACTION * len(v5_original_permutation)
))
v5_original_build_idx = v5_original_permutation[
    :v5_original_build_rows
]
v5_original_ecdf_idx = v5_original_permutation[
    v5_original_build_rows:
]
assert np.intersect1d(
    v5_original_build_idx,
    v5_original_ecdf_idx,
    assume_unique=True,
).size == 0

v5_build_values = X_stress_2017[v5_original_build_idx]
v5_ecdf_values = X_stress_2017[v5_original_ecdf_idx]
v5_build_latent = predict_latent(
    best_model,
    v5_build_values,
    batch_size=V5_INFERENCE_BATCH_SIZE,
)
v5_ecdf_latent = predict_latent(
    best_model,
    v5_ecdf_values,
    batch_size=V5_INFERENCE_BATCH_SIZE,
)
v5_latent_median = np.median(v5_build_latent, axis=0)
v5_latent_mad = np.median(
    np.abs(v5_build_latent - v5_latent_median),
    axis=0,
)
v5_latent_scale = np.maximum(1.4826 * v5_latent_mad, 1e-6)
v5_build_latent_distance = np.sqrt(np.mean(
    np.square(
        (v5_build_latent - v5_latent_median) / v5_latent_scale
    ),
    axis=1,
))
v5_ecdf_latent_distance = np.sqrt(np.mean(
    np.square(
        (v5_ecdf_latent - v5_latent_median) / v5_latent_scale
    ),
    axis=1,
))
v5_build_full_mse = compute_score(
    best_model,
    v5_build_values,
)
v5_ecdf_full_mse = compute_score(
    best_model,
    v5_ecdf_values,
)
v5_build_stable_mse = compute_score(
    stable_core_model,
    v5_build_values[:, STABLE_FEATURE_MASK],
)
v5_ecdf_stable_mse = compute_score(
    stable_core_model,
    v5_ecdf_values[:, STABLE_FEATURE_MASK],
)

v5_build_burden = np.maximum.reduce([
    v5_rank01(v5_build_latent_distance),
    v5_rank01(v5_build_full_mse),
    v5_rank01(v5_build_stable_mse),
])
v5_ecdf_burden = np.maximum.reduce([
    empirical_cdf(
        v5_ecdf_latent_distance,
        np.sort(v5_build_latent_distance),
    ),
    empirical_cdf(
        v5_ecdf_full_mse,
        np.sort(v5_build_full_mse),
    ),
    empirical_cdf(
        v5_ecdf_stable_mse,
        np.sort(v5_build_stable_mse),
    ),
])

v5_build_keep_rows = max(
    2,
    int(np.floor(V5_RETAIN_FRACTION * len(v5_original_build_idx))),
)
v5_build_order = np.argsort(v5_build_burden, kind='mergesort')
v5_core_build_idx = v5_original_build_idx[
    v5_build_order[:v5_build_keep_rows]
]
v5_burden_cutoff = float(
    v5_build_burden[v5_build_order[v5_build_keep_rows - 1]]
)
v5_core_ecdf_idx = v5_original_ecdf_idx[
    v5_ecdf_burden <= v5_burden_cutoff
]
v5_pure_core_idx = np.concatenate([
    v5_core_build_idx,
    v5_core_ecdf_idx,
])

assert len(v5_core_build_idx) > 0
assert len(v5_core_ecdf_idx) > 0
assert np.intersect1d(
    v5_core_build_idx,
    v5_core_ecdf_idx,
    assume_unique=True,
).size == 0
assert np.setdiff1d(
    v5_core_build_idx,
    v5_adapt_idx,
    assume_unique=False,
).size == 0
assert np.setdiff1d(
    v5_core_ecdf_idx,
    v5_adapt_idx,
    assume_unique=False,
).size == 0

v5_second_stage_quantile = QuantileTransformer(
    n_quantiles=min(1000, len(v5_core_build_idx)),
    output_distribution='normal',
    subsample=min(200_000, len(v5_core_build_idx)),
    random_state=V5_SEED + 102,
)
v5_second_stage_quantile.fit(
    X_stress_2017[v5_core_build_idx, :n_cont]
)

# Memmap separata: preserva il buffer v4 e limita la RAM aggiuntiva.
V5_TARGET_CACHE_DIR = (
    Path('Artefatti') / 'protocol_v5_retro' / 'cache'
)
V5_TARGET_CACHE_DIR.mkdir(parents=True, exist_ok=True)
V5_TARGET_MEMMAP_PATH = V5_TARGET_CACHE_DIR / 'X_target_v5_float32.npy'
X_target_v5 = np.lib.format.open_memmap(
    V5_TARGET_MEMMAP_PATH,
    mode='w+',
    dtype=np.float32,
    shape=X_stress_2017.shape,
)
V5_TARGET_TRANSFORM_STATE = 'IN_PROGRESS'
try:
    for v5_start in range(0, len(X_target_v5), V5_TRANSFORM_BATCH_SIZE):
        v5_end = min(v5_start + V5_TRANSFORM_BATCH_SIZE, len(X_target_v5))
        v5_continuous = v5_second_stage_quantile.transform(
            X_stress_2017[v5_start:v5_end, :n_cont]
        ).astype(np.float32, copy=False)
        X_target_v5[v5_start:v5_end, :n_cont] = np.clip(
            v5_continuous,
            clip_lo,
            clip_hi,
        )
        X_target_v5[v5_start:v5_end, n_cont:] = (
            X_stress_2017[v5_start:v5_end, n_cont:]
        )
    X_target_v5.flush()
except Exception:
    V5_TARGET_TRANSFORM_STATE = 'FAILED_PARTIAL_RESTART_REQUIRED'
    raise
V5_TARGET_TRANSFORM_STATE = 'COMPLETE'
# Il buffer v4 resta disponibile per il controllo esatto della v6.
# Non viene modificato dalla memmap v5 e consente di ricalcolare
# stable_core__mse_topk_w025 con il preprocessing originale.
if 'X_target_pseudobenign' not in globals():
    raise RuntimeError('Buffer target v4 non disponibile per il controllo v6.')

v5_purification_structure_df = pd.DataFrame([{
    'initial_cluster_rows': len(v5_original_reference_idx),
    'retained_rows': len(v5_pure_core_idx),
    'retain_fraction': len(v5_pure_core_idx) / len(v5_original_reference_idx),
    'initial_build_rows': len(v5_original_build_idx),
    'initial_ecdf_rows': len(v5_original_ecdf_idx),
    'build_rows': len(v5_core_build_idx),
    'ecdf_rows': len(v5_core_ecdf_idx),
    'build_retain_fraction': (
        len(v5_core_build_idx) / len(v5_original_build_idx)
    ),
    'ecdf_retain_fraction': (
        len(v5_core_ecdf_idx) / len(v5_original_ecdf_idx)
    ),
    'burden_cutoff_fit_on_build': v5_burden_cutoff,
    'selection_uses_target_labels': False,
    'consensus_rule': 'max(rank_latent_distance, rank_full_mse, rank_stable_mse)',
    'target_transform': (
        'source_scaled_to_purified_second_stage_quantile_disk_memmap'
    ),
}])

del (
    v5_build_values,
    v5_ecdf_values,
    v5_build_latent,
    v5_ecdf_latent,
    v5_build_latent_distance,
    v5_ecdf_latent_distance,
    v5_build_full_mse,
    v5_ecdf_full_mse,
    v5_build_stable_mse,
    v5_ecdf_stable_mse,
    v5_build_burden,
    v5_ecdf_burden,
    v5_continuous,
)
gc.collect()

print('PURIFICAZIONE V5 COMPLETATA SENZA LABEL')
print(v5_purification_structure_df.to_string(index=False))
print('La purezza diagnostica sarà calcolata solo dopo la selezione.')


PURIFICAZIONE V5 COMPLETATA SENZA LABEL
 initial_cluster_rows  retained_rows  retain_fraction  initial_build_rows  initial_ecdf_rows  build_rows  ecdf_rows  build_retain_fraction  ecdf_retain_fraction  burden_cutoff_fit_on_build  selection_uses_target_labels                                            consensus_rule                                            target_transform
               177956          98170         0.551653              106773              71183       58725      39445               0.549999              0.554135                    0.720245                         False max(rank_latent_distance, rank_full_mse, rank_stable_mse) source_scaled_to_purified_second_stage_quantile_disk_memmap
La purezza diagnostica sarà calcolata solo dopo la selezione.


## Riferimenti robusti, score soft-drift e GMM latente

Build ed ECDF sono disgiunti. Le feature ad alto drift vengono attenuate, non
eliminate; `min_seg_size_forward` conserva peso non nullo. Le GMM full e
stable sono scelte tramite BIC e rifittate dopo trimming non supervisionato.


In [32]:
# V5_RETRO_CELL_3_REFERENCES
if globals().get('V5_TARGET_TRANSFORM_STATE') != 'COMPLETE':
    raise RuntimeError(
        'La trasformazione target v5 non è completa. '
        'Riavviare il kernel e rieseguire il notebook in ordine.'
    )


def v5_joint_arrays(model, values, batch_size=V5_INFERENCE_BATCH_SIZE):
    joint = Model(
        model.input,
        [model.output, model.get_layer('latent').output],
    )
    residual_parts = []
    latent_parts = []
    for start in range(0, len(values), batch_size):
        batch = values[start:start + batch_size]
        reconstruction, latent = joint.predict_on_batch(batch)
        residual_parts.append(
            np.square(batch - reconstruction).astype(np.float32)
        )
        latent_parts.append(np.asarray(latent, dtype=np.float32))
    return (
        np.concatenate(residual_parts, axis=0),
        np.concatenate(latent_parts, axis=0),
    )


def v5_fit_gmm(latent_values, domain, branch, seed):
    def valid_mixture(mixture):
        return bool(
            mixture.converged_
            and np.isfinite(mixture.weights_).all()
            and np.isfinite(mixture.means_).all()
            and np.isfinite(mixture.covariances_).all()
            and (mixture.covariances_ > 0).all()
            and np.isclose(mixture.weights_.sum(), 1.0)
        )

    if len(latent_values) > V5_REFERENCE_FIT_MAX_ROWS:
        rng = np.random.default_rng(seed)
        fit_values = latent_values[rng.choice(
            len(latent_values),
            size=V5_REFERENCE_FIT_MAX_ROWS,
            replace=False,
        )]
    else:
        fit_values = latent_values

    fitted = []
    bic_rows = []
    for component_count in V5_GMM_COMPONENTS:
        mixture = GaussianMixture(
            n_components=int(component_count),
            covariance_type='diag',
            reg_covar=V5_GMM_REG_COVAR,
            n_init=V5_GMM_N_INIT,
            max_iter=V5_GMM_MAX_ITER,
            init_params='kmeans',
            random_state=seed + int(component_count),
        )
        mixture.fit(fit_values)
        bic_value = float(mixture.bic(fit_values))
        mixture_valid = valid_mixture(mixture) and np.isfinite(bic_value)
        if mixture_valid:
            fitted.append((bic_value, mixture))
        bic_rows.append({
            'domain': domain,
            'branch': branch,
            'components': int(component_count),
            'bic': bic_value,
            'initial_converged': bool(mixture.converged_),
            'initial_valid': bool(mixture_valid),
        })

    if not fitted:
        raise RuntimeError(
            f'Nessuna GMM convergente per {domain}/{branch}.'
        )
    _, initial_selected = min(fitted, key=lambda item: item[0])
    selected = initial_selected
    selected_components = int(initial_selected.n_components)
    initial_nll = -initial_selected.score_samples(fit_values)
    trim_limit = np.quantile(
        initial_nll,
        1.0 - V5_GMM_TRIM_FRACTION,
    )
    trim_keep = initial_nll <= trim_limit
    refit_used = False
    if trim_keep.sum() >= max(1000, selected_components * 20):
        refit_candidate = GaussianMixture(
            n_components=selected_components,
            covariance_type='diag',
            reg_covar=V5_GMM_REG_COVAR,
            n_init=V5_GMM_N_INIT,
            max_iter=V5_GMM_MAX_ITER,
            init_params='kmeans',
            random_state=seed + 100 + selected_components,
        )
        refit_candidate.fit(fit_values[trim_keep])
        if valid_mixture(refit_candidate):
            selected = refit_candidate
            refit_used = True

    for row in bic_rows:
        row['selected_components'] = selected_components
        row['refit_converged'] = bool(selected.converged_)
        row['refit_used'] = bool(refit_used)
        row['fallback_to_initial'] = bool(not refit_used)
        row['trimmed_fraction'] = float(1.0 - trim_keep.mean())
    return selected, bic_rows


def v5_raw_components(
    model,
    values,
    reference,
    top_k=SCORE_TOP_K,
    batch_size=V5_INFERENCE_BATCH_SIZE,
    feature_mask=None,
):
    joint = Model(
        model.input,
        [model.output, model.get_layer('latent').output],
    )
    output = {
        'mse': [],
        'topk': [],
        'soft_mse': [],
        'soft_topk': [],
        'gmm_nll': [],
    }
    weights = np.asarray(reference['feature_weights'], dtype=np.float32)
    weight_sum = max(float(weights.sum()), 1e-8)
    k = min(int(top_k), values.shape[1])
    if feature_mask is not None:
        k = min(int(top_k), int(np.asarray(feature_mask).sum()))

    for start in range(0, len(values), batch_size):
        full_batch = values[start:start + batch_size]
        batch = (
            full_batch
            if feature_mask is None
            else full_batch[:, feature_mask]
        )
        reconstruction, latent = joint.predict_on_batch(batch)
        residual = np.square(batch - reconstruction).astype(np.float32)
        robust_z = np.maximum(
            (residual - reference['residual_median'])
            / reference['residual_scale'],
            0.0,
        )
        top_values = np.partition(robust_z, -k, axis=1)[:, -k:]
        weighted_z = robust_z * weights[None, :]
        soft_top_values = np.partition(
            weighted_z,
            -k,
            axis=1,
        )[:, -k:]
        latent = np.asarray(latent, dtype=np.float32)

        output['mse'].append(np.mean(residual, axis=1))
        output['topk'].append(np.mean(top_values, axis=1))
        output['soft_mse'].append(
            np.sum(residual * weights[None, :], axis=1) / weight_sum
        )
        output['soft_topk'].append(np.mean(soft_top_values, axis=1))
        output['gmm_nll'].append(
            -reference['gmm'].score_samples(latent)
        )

    result = {
        key: np.concatenate(parts).astype(np.float32, copy=False)
        for key, parts in output.items()
    }
    for key, component in result.items():
        if not np.isfinite(component).all():
            raise RuntimeError(f'Componente score v5 non finita: {key}')
    return result


def v5_fit_reference(
    model,
    build_values,
    ecdf_values,
    feature_weights,
    domain,
    branch,
    seed,
):
    rng = np.random.default_rng(seed)
    if len(build_values) > V5_REFERENCE_FIT_MAX_ROWS:
        build_values = build_values[rng.choice(
            len(build_values),
            size=V5_REFERENCE_FIT_MAX_ROWS,
            replace=False,
        )]
    residual, latent = v5_joint_arrays(model, build_values)
    residual_median = np.median(residual, axis=0).astype(np.float32)
    residual_mad = np.median(
        np.abs(residual - residual_median),
        axis=0,
    ).astype(np.float32)
    gmm, gmm_rows = v5_fit_gmm(
        latent,
        domain=domain,
        branch=branch,
        seed=seed + 10,
    )
    reference = {
        'residual_median': residual_median,
        'residual_scale': np.maximum(
            1.4826 * residual_mad,
            1e-6,
        ),
        'feature_weights': np.asarray(
            feature_weights,
            dtype=np.float32,
        ),
        'gmm': gmm,
    }
    ecdf_raw = v5_raw_components(model, ecdf_values, reference)
    reference['sorted'] = {
        key: np.sort(values.astype(np.float32, copy=False))
        for key, values in ecdf_raw.items()
    }
    del residual, latent, ecdf_raw
    gc.collect()
    return reference, gmm_rows


def v5_percentile_components(raw_components, reference):
    return {
        key: empirical_cdf(values, reference['sorted'][key])
        for key, values in raw_components.items()
    }


# Drift rank alto -> peso minore, ma mai nullo.
v5_drift_order = (
    cross_dataset_v4_feature_drift_df['feature'].astype(str).tolist()
)
v5_drift_position = {
    name: position for position, name in enumerate(v5_drift_order)
}
v5_full_feature_weights = np.ones(INPUT_DIM, dtype=np.float32)
for feature_position, feature_name in enumerate(FEATURE_NAMES_FINAL):
    drift_position = v5_drift_position[feature_name]
    drift_severity = 1.0 - (
        drift_position / max(len(v5_drift_order) - 1, 1)
    )
    v5_full_feature_weights[feature_position] = (
        1.0 - 0.75 * drift_severity
    )
if 'min_seg_size_forward' in FEATURE_NAMES_FINAL:
    min_seg_position = FEATURE_NAMES_FINAL.index('min_seg_size_forward')
    v5_full_feature_weights[min_seg_position] = max(
        float(v5_full_feature_weights[min_seg_position]),
        0.65,
    )
assert np.isfinite(v5_full_feature_weights).all()
assert (v5_full_feature_weights >= 0.25).all()
assert (v5_full_feature_weights <= 1.0).all()

v5_feature_weights_df = pd.DataFrame({
    'feature': FEATURE_NAMES_FINAL,
    'soft_residual_weight': v5_full_feature_weights,
    'drift_rank_desc': [
        v5_drift_position[name] + 1 for name in FEATURE_NAMES_FINAL
    ],
})

v5_source_reference_rng = np.random.default_rng(V5_SEED + 201)
v5_source_reference_permutation = v5_source_reference_rng.permutation(
    np.asarray(source_reference_idx, dtype=np.int64)
)
v5_source_build_rows = int(np.floor(
    V5_REFERENCE_BUILD_FRACTION
    * len(v5_source_reference_permutation)
))
v5_source_build_idx = v5_source_reference_permutation[
    :v5_source_build_rows
]
v5_source_ecdf_idx = v5_source_reference_permutation[
    v5_source_build_rows:
]
assert np.intersect1d(
    v5_source_build_idx,
    v5_source_ecdf_idx,
    assume_unique=True,
).size == 0

v5_gmm_rows = []
v5_full_source_reference, local_gmm_rows = v5_fit_reference(
    best_model,
    X_select[v5_source_build_idx],
    X_select[v5_source_ecdf_idx],
    v5_full_feature_weights,
    domain='source',
    branch='full',
    seed=V5_SEED + 210,
)
v5_gmm_rows.extend(local_gmm_rows)
v5_stable_source_reference, local_gmm_rows = v5_fit_reference(
    stable_core_model,
    X_select[v5_source_build_idx][:, STABLE_FEATURE_MASK],
    X_select[v5_source_ecdf_idx][:, STABLE_FEATURE_MASK],
    np.ones(int(STABLE_FEATURE_MASK.sum()), dtype=np.float32),
    domain='source',
    branch='stable',
    seed=V5_SEED + 220,
)
v5_gmm_rows.extend(local_gmm_rows)
v5_full_target_reference, local_gmm_rows = v5_fit_reference(
    best_model,
    X_target_v5[v5_core_build_idx],
    X_target_v5[v5_core_ecdf_idx],
    v5_full_feature_weights,
    domain='target',
    branch='full',
    seed=V5_SEED + 230,
)
v5_gmm_rows.extend(local_gmm_rows)
v5_stable_target_reference, local_gmm_rows = v5_fit_reference(
    stable_core_model,
    X_target_v5[v5_core_build_idx][:, STABLE_FEATURE_MASK],
    X_target_v5[v5_core_ecdf_idx][:, STABLE_FEATURE_MASK],
    np.ones(int(STABLE_FEATURE_MASK.sum()), dtype=np.float32),
    domain='target',
    branch='stable',
    seed=V5_SEED + 240,
)
v5_gmm_rows.extend(local_gmm_rows)
v5_gmm_diagnostics_df = pd.DataFrame(v5_gmm_rows)
del local_gmm_rows
gc.collect()

print('RIFERIMENTI V5 COMPLETATI')
print(
    v5_gmm_diagnostics_df[
        [
            'domain',
            'branch',
            'components',
            'bic',
            'selected_components',
            'refit_converged',
        ]
    ].round(3).to_string(index=False)
)


RIFERIMENTI V5 COMPLETATI
domain branch  components         bic  selected_components  refit_converged
source   full           2 4225201.937                    6             True
source   full           4 3402567.872                    6             True
source   full           6 3033737.804                    6             True
source stable           2 2935991.250                    6             True
source stable           4 2099920.821                    6             True
source stable           6 1840093.047                    6             True
target   full           2 2037505.126                    6             True
target   full           4 1774011.614                    6             True
target   full           6 1671742.602                    6             True
target stable           2 1391683.367                    6             True
target stable           4 1263306.620                    6             True
target stable           6 1135731.861                    6    

## Score v5 e fusioni full + stable

Ogni modello viene inferito una sola volta per split. Le componenti sono
portate a percentile ECDF prima delle dieci fusioni congelate, includendo un
rescue moderato del ramo full per recuperare recall.


In [33]:
# V5_RETRO_CELL_4_SCORING
def v5_score_fundamentals(values, domain):
    if domain == 'source':
        full_reference = v5_full_source_reference
        stable_reference = v5_stable_source_reference
    elif domain == 'target':
        full_reference = v5_full_target_reference
        stable_reference = v5_stable_target_reference
    else:
        raise ValueError(f'Dominio v5 sconosciuto: {domain}')

    full_raw = v5_raw_components(
        best_model,
        values,
        full_reference,
    )
    stable_raw = v5_raw_components(
        stable_core_model,
        values,
        stable_reference,
        feature_mask=STABLE_FEATURE_MASK,
    )
    full_components = v5_percentile_components(
        full_raw,
        full_reference,
    )
    stable_components = v5_percentile_components(
        stable_raw,
        stable_reference,
    )
    fundamentals = {
        'full': (
            0.75 * full_components['mse']
            + 0.25 * full_components['topk']
        ).astype(np.float32),
        'stable': (
            0.75 * stable_components['mse']
            + 0.25 * stable_components['topk']
        ).astype(np.float32),
        'soft': (
            0.75 * full_components['soft_mse']
            + 0.25 * full_components['soft_topk']
        ).astype(np.float32),
        'gmm': (
            0.50 * full_components['gmm_nll']
            + 0.50 * stable_components['gmm_nll']
        ).astype(np.float32),
        'full_mse': full_components['mse'].astype(
            np.float32,
            copy=False,
        ),
    }
    del full_raw, stable_raw, full_components, stable_components
    gc.collect()
    return fundamentals


def v5_apply_candidate(fundamentals, recipe):
    if recipe['kind'] == 'single':
        return np.asarray(
            fundamentals[recipe['key']],
            dtype=np.float32,
        )
    if recipe['kind'] == 'convex':
        result = np.zeros(
            len(next(iter(fundamentals.values()))),
            dtype=np.float32,
        )
        weight_sum = 0.0
        for key, weight in recipe['weights'].items():
            if weight < 0:
                raise ValueError('I pesi ensemble devono essere non negativi.')
            result += float(weight) * fundamentals[key]
            weight_sum += float(weight)
        if not np.isclose(weight_sum, 1.0):
            raise ValueError(
                f'Pesi non normalizzati per {recipe["name"]}: {weight_sum}'
            )
        return result
    if recipe['kind'] == 'gated_rescue':
        base = fundamentals[recipe['base']]
        rescue = fundamentals[recipe['rescue']]
        return np.clip(
            base
            + float(recipe['weight']) * np.maximum(rescue - base, 0.0),
            0.0,
            1.0,
        ).astype(np.float32)
    raise ValueError(f'Ricetta v5 sconosciuta: {recipe}')


def v5_candidate_map(fundamentals):
    result = {
        recipe['name']: v5_apply_candidate(fundamentals, recipe)
        for recipe in V5_CANDIDATE_RECIPES
    }
    expected_length = len(next(iter(result.values())))
    for name, scores in result.items():
        assert len(scores) == expected_length
        assert np.isfinite(scores).all(), f'Score non finito: {name}'
    return result


def v5_stratified_sample(base_indices, labels, max_rows, seed):
    base_indices = np.asarray(base_indices, dtype=np.int64)
    if len(base_indices) <= max_rows:
        return base_indices.copy()
    selected, _ = train_test_split(
        base_indices,
        train_size=int(max_rows),
        random_state=seed,
        stratify=np.asarray(labels)[base_indices],
    )
    return np.asarray(selected, dtype=np.int64)


v5_source_cal_sample_idx = v5_stratified_sample(
    v5_source_cal_idx,
    y_test_internal_binary,
    V5_SOURCE_CAL_MAX_ROWS,
    V5_SEED + 301,
)
v5_source_rank_sample_idx = v5_stratified_sample(
    v5_source_rank_idx,
    y_test_internal_binary,
    V5_SOURCE_RANK_MAX_ROWS,
    V5_SEED + 302,
)

v5_target_cal_values = X_target_v5[v5_target_cal_idx]
v5_target_cal_fundamentals = v5_score_fundamentals(
    v5_target_cal_values,
    'target',
)
v5_target_cal_scores = v5_candidate_map(
    v5_target_cal_fundamentals
)
del v5_target_cal_values
gc.collect()

v5_target_rank_values = X_target_v5[v5_target_rank_idx]
v5_target_rank_fundamentals = v5_score_fundamentals(
    v5_target_rank_values,
    'target',
)
v5_target_rank_scores = v5_candidate_map(
    v5_target_rank_fundamentals
)
del v5_target_rank_values
gc.collect()

v5_source_cal_values = X_test_internal[v5_source_cal_sample_idx]
v5_source_cal_fundamentals = v5_score_fundamentals(
    v5_source_cal_values,
    'source',
)
v5_source_cal_scores = v5_candidate_map(
    v5_source_cal_fundamentals
)
del v5_source_cal_values
gc.collect()

v5_source_rank_values = X_test_internal[v5_source_rank_sample_idx]
v5_source_rank_fundamentals = v5_score_fundamentals(
    v5_source_rank_values,
    'source',
)
v5_source_rank_scores = v5_candidate_map(
    v5_source_rank_fundamentals
)
del v5_source_rank_values
gc.collect()

print('SCORE V5 CALIBRATION/RANKING COMPLETATI')
print(
    f'Target: calibration={len(v5_target_cal_idx):,}, '
    f'ranking={len(v5_target_rank_idx):,}'
)
print(
    f'Source campionato: calibration={len(v5_source_cal_sample_idx):,}, '
    f'ranking={len(v5_source_rank_sample_idx):,}'
)


SCORE V5 CALIBRATION/RANKING COMPLETATI
Target: calibration=471,272, ranking=471,271
Source campionato: calibration=600,000, ranking=800,000


## Calibrazione cross-fitted, ranking e guardrail intra-dataset

Le soglie target sono cross-fitted per capture-file sul solo calibration; le
soglie source usano cinque fold stratificati. La selezione avviene sul ranking
separato, con F1 primaria e recall come discriminante fra candidate vicine.


In [34]:
# V5_RETRO_CELL_5_SELECTION
def v5_find_capture_id(fragment):
    matches = [
        index
        for index, name in enumerate(stress_file_names_2017)
        if fragment.lower() in name.lower()
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f'Capture non univoca per {fragment}: {matches}'
        )
    return matches[0]


v5_capture_fold_rows = []


def v5_target_capture_folds(global_indices):
    global_indices = np.asarray(global_indices, dtype=np.int64)
    local_groups = np.asarray(stress_file_ids_2017)[global_indices]
    capture_pairs = [
        ('Monday-WorkingHours', 'Afternoon-DDos'),
        ('Tuesday-WorkingHours', 'Afternoon-PortScan'),
        ('Wednesday-workingHours', 'Morning-Bot'),
        ('Afternoon-Infilteration', 'Morning-WebAttacks'),
    ]
    folds = []
    coverage = np.zeros(len(global_indices), dtype=np.int8)
    for fold_number, (first_fragment, second_fragment) in enumerate(
        capture_pairs,
        start=1,
    ):
        holdout_ids = [
            v5_find_capture_id(first_fragment),
            v5_find_capture_id(second_fragment),
        ]
        validation_positions = np.flatnonzero(
            np.isin(local_groups, holdout_ids)
        )
        training_positions = np.flatnonzero(
            ~np.isin(local_groups, holdout_ids)
        )
        if not len(validation_positions) or not len(training_positions):
            raise RuntimeError(
                f'Fold capture vuoto: {first_fragment}, {second_fragment}'
            )
        training_y = y_stress_2017_binary[
            global_indices[training_positions]
        ]
        validation_y = y_stress_2017_binary[
            global_indices[validation_positions]
        ]
        if len(np.unique(training_y)) != 2 or len(np.unique(validation_y)) != 2:
            raise RuntimeError(
                f'Fold capture senza entrambe le classi: {fold_number}'
            )
        coverage[validation_positions] += 1
        folds.append((training_positions, validation_positions))
        v5_capture_fold_rows.append({
            'fold': fold_number,
            'holdout_capture_1': stress_file_names_2017[holdout_ids[0]],
            'holdout_capture_2': stress_file_names_2017[holdout_ids[1]],
            'train_rows': len(training_positions),
            'train_benign': int((training_y == 0).sum()),
            'train_attacks': int((training_y == 1).sum()),
            'validation_rows': len(validation_positions),
            'validation_benign': int((validation_y == 0).sum()),
            'validation_attacks': int((validation_y == 1).sum()),
        })
    assert np.all(coverage == 1)
    return folds


def v5_select_threshold(y_true, scores, beta=1.0, max_fpr=1.0):
    y_true = np.asarray(y_true, dtype=np.int8)
    scores = np.asarray(scores, dtype=np.float64)
    if len(y_true) != len(scores):
        raise ValueError('Label e score hanno lunghezze diverse.')
    if not np.isfinite(scores).all():
        raise ValueError('Score non finiti nella calibrazione.')

    order = np.argsort(scores, kind='mergesort')[::-1]
    sorted_scores = scores[order]
    sorted_y = y_true[order]
    true_positive = np.cumsum(sorted_y == 1)
    false_positive = np.cumsum(sorted_y == 0)
    distinct_last = np.r_[
        np.flatnonzero(sorted_scores[:-1] != sorted_scores[1:]),
        len(sorted_scores) - 1,
    ]
    tp = true_positive[distinct_last].astype(np.float64)
    fp = false_positive[distinct_last].astype(np.float64)
    positives = max(float((y_true == 1).sum()), 1.0)
    negatives = max(float((y_true == 0).sum()), 1.0)
    precision = tp / np.maximum(tp + fp, 1.0)
    recall = tp / positives
    fpr = fp / negatives
    beta_sq = float(beta) ** 2
    fbeta = (
        (1.0 + beta_sq) * precision * recall
        / np.maximum(beta_sq * precision + recall, 1e-12)
    )
    valid = fpr <= float(max_fpr)
    if not valid.any():
        return float(np.nextafter(sorted_scores.max(), np.inf))
    best_fbeta = float(np.max(fbeta[valid]))
    tied = np.flatnonzero(
        valid
        & np.isclose(
            fbeta,
            best_fbeta,
            rtol=0.0,
            atol=1e-12,
        )
    )
    best_recall = float(np.max(recall[tied]))
    recall_tied = tied[np.isclose(
        recall[tied],
        best_recall,
        rtol=0.0,
        atol=1e-12,
    )]
    best_local = recall_tied[np.argmin(fpr[recall_tied])]
    return float(sorted_scores[distinct_last[best_local]])


def v5_metrics_from_predictions(y_true, predictions, scores):
    y_true = np.asarray(y_true, dtype=np.int8)
    predictions = np.asarray(predictions, dtype=np.int8)
    return {
        'f1': f1_score(y_true, predictions, zero_division=0),
        'precision': precision_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        'recall': recall_score(y_true, predictions, zero_division=0),
        'fpr_benign': float(np.mean(predictions[y_true == 0])),
        'roc_auc': roc_auc_score(y_true, scores),
        'pr_auc': average_precision_score(y_true, scores),
    }


v5_threshold_rows = []
v5_cv_cache = {}


def v5_crossfit_thresholds(
    y_true,
    scores,
    folds,
    domain,
    candidate_name,
):
    y_true = np.asarray(y_true, dtype=np.int8)
    scores = np.asarray(scores, dtype=np.float32)
    oof_f1 = np.zeros(len(y_true), dtype=np.int8)
    oof_f2 = np.zeros(len(y_true), dtype=np.int8)
    coverage = np.zeros(len(y_true), dtype=np.int8)
    f1_thresholds = []
    f2_thresholds = []

    for fold_number, (training_positions, validation_positions) in enumerate(
        folds,
        start=1,
    ):
        fold_y = y_true[training_positions]
        fold_scores = scores[training_positions]
        f1_threshold = v5_select_threshold(
            fold_y,
            fold_scores,
            beta=1.0,
            max_fpr=1.0,
        )
        f2_threshold = v5_select_threshold(
            fold_y,
            fold_scores,
            beta=2.0,
            max_fpr=V5_F2_MAX_CALIBRATION_FPR,
        )
        f1_thresholds.append(f1_threshold)
        f2_thresholds.append(f2_threshold)
        oof_f1[validation_positions] = (
            scores[validation_positions] >= f1_threshold
        ).astype(np.int8)
        oof_f2[validation_positions] = (
            scores[validation_positions] >= f2_threshold
        ).astype(np.int8)
        coverage[validation_positions] += 1
        v5_threshold_rows.extend([
            {
                'domain': domain,
                'candidate': candidate_name,
                'policy': 'f1',
                'fold': fold_number,
                'threshold': f1_threshold,
            },
            {
                'domain': domain,
                'candidate': candidate_name,
                'policy': 'f2_foldtrain_fpr_le_030',
                'fold': fold_number,
                'threshold': f2_threshold,
            },
        ])

    assert np.all(coverage == 1)
    f1_median = float(np.median(f1_thresholds))
    f2_fold_median = float(np.median(f2_thresholds))
    f2_pooled_threshold = v5_select_threshold(
        y_true,
        scores,
        beta=2.0,
        max_fpr=V5_F2_MAX_CALIBRATION_FPR,
    )
    # Una soglia più alta non può aumentare il FPR: proietta la mediana dei
    # fold sul vincolo misurato nell'intera calibration.
    f2_median = max(f2_fold_median, f2_pooled_threshold)
    f2_pooled_predictions = (scores >= f2_median).astype(np.int8)
    f2_pooled_metrics = v5_metrics_from_predictions(
        y_true,
        f2_pooled_predictions,
        scores,
    )
    if (
        f2_pooled_metrics['fpr_benign']
        > V5_F2_MAX_CALIBRATION_FPR + 1e-12
    ):
        raise RuntimeError(
            f'Vincolo FPR F2 non rispettato per {domain}/{candidate_name}.'
        )
    v5_threshold_rows.extend([
        {
            'domain': domain,
            'candidate': candidate_name,
            'policy': 'f1_median',
            'fold': 'aggregate',
            'threshold': f1_median,
        },
        {
            'domain': domain,
            'candidate': candidate_name,
            'policy': 'f2_calibration_fpr_le_030_aggregate',
            'fold': 'aggregate',
            'threshold': f2_median,
        },
    ])
    return {
        'f1_threshold': f1_median,
        'f2_threshold': f2_median,
        'f1_oof': v5_metrics_from_predictions(
            y_true,
            oof_f1,
            scores,
        ),
        'f2_oof': v5_metrics_from_predictions(
            y_true,
            oof_f2,
            scores,
        ),
        'f2_pooled': f2_pooled_metrics,
        'f1_threshold_iqr': float(
            np.percentile(f1_thresholds, 75)
            - np.percentile(f1_thresholds, 25)
        ),
        'f2_threshold_iqr': float(
            np.percentile(f2_thresholds, 75)
            - np.percentile(f2_thresholds, 25)
        ),
    }


v5_target_cal_y = y_stress_2017_binary[v5_target_cal_idx]
v5_target_rank_y = y_stress_2017_binary[v5_target_rank_idx]
v5_source_cal_y = y_test_internal_binary[v5_source_cal_sample_idx]
v5_source_rank_y = y_test_internal_binary[v5_source_rank_sample_idx]
v5_target_folds = v5_target_capture_folds(v5_target_cal_idx)
v5_capture_fold_audit_df = pd.DataFrame(v5_capture_fold_rows)
v5_source_splitter = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=V5_SEED + 401,
)
v5_source_folds = list(v5_source_splitter.split(
    np.zeros(len(v5_source_cal_y), dtype=np.int8),
    v5_source_cal_y,
))

v5_ranking_rows = []
for recipe in V5_CANDIDATE_RECIPES:
    candidate_name = recipe['name']
    target_cv = v5_crossfit_thresholds(
        v5_target_cal_y,
        v5_target_cal_scores[candidate_name],
        v5_target_folds,
        domain='target',
        candidate_name=candidate_name,
    )
    source_cv = v5_crossfit_thresholds(
        v5_source_cal_y,
        v5_source_cal_scores[candidate_name],
        v5_source_folds,
        domain='source',
        candidate_name=candidate_name,
    )
    target_rank_f1 = score_metrics(
        v5_target_rank_y,
        v5_target_rank_scores[candidate_name],
        target_cv['f1_threshold'],
    )
    target_rank_f2 = score_metrics(
        v5_target_rank_y,
        v5_target_rank_scores[candidate_name],
        target_cv['f2_threshold'],
    )
    source_rank_f1 = score_metrics(
        v5_source_rank_y,
        v5_source_rank_scores[candidate_name],
        source_cv['f1_threshold'],
    )
    source_rank_f2 = score_metrics(
        v5_source_rank_y,
        v5_source_rank_scores[candidate_name],
        source_cv['f2_threshold'],
    )
    v5_cv_cache[candidate_name] = {
        'target': target_cv,
        'source': source_cv,
    }
    v5_ranking_rows.append({
        'candidate': candidate_name,
        'target_rank_f1': target_rank_f1['f1'],
        'target_rank_precision': target_rank_f1['precision'],
        'target_rank_recall': target_rank_f1['recall'],
        'target_rank_fpr_benign': target_rank_f1['fpr_benign'],
        'target_rank_roc_auc': target_rank_f1['roc_auc'],
        'target_rank_pr_auc': target_rank_f1['pr_auc'],
        'target_rank_f2_policy_f1': target_rank_f2['f1'],
        'target_rank_f2_policy_recall': target_rank_f2['recall'],
        'target_rank_f2_policy_fpr_benign': (
            target_rank_f2['fpr_benign']
        ),
        'target_cal_oof_f1': target_cv['f1_oof']['f1'],
        'target_cal_oof_recall': target_cv['f1_oof']['recall'],
        'target_cal_f2_pooled_fpr_benign': (
            target_cv['f2_pooled']['fpr_benign']
        ),
        'target_threshold': target_cv['f1_threshold'],
        'target_threshold_iqr': target_cv['f1_threshold_iqr'],
        'source_rank_f1': source_rank_f1['f1'],
        'source_rank_precision': source_rank_f1['precision'],
        'source_rank_recall': source_rank_f1['recall'],
        'source_rank_fpr_benign': source_rank_f1['fpr_benign'],
        'source_rank_pr_auc': source_rank_f1['pr_auc'],
        'source_rank_f2_policy_f1': source_rank_f2['f1'],
        'source_rank_f2_policy_recall': source_rank_f2['recall'],
        'source_rank_f2_policy_fpr_benign': (
            source_rank_f2['fpr_benign']
        ),
        'source_cal_oof_f1': source_cv['f1_oof']['f1'],
        'source_cal_f2_pooled_fpr_benign': (
            source_cv['f2_pooled']['fpr_benign']
        ),
        'source_threshold': source_cv['f1_threshold'],
        'source_threshold_iqr': source_cv['f1_threshold_iqr'],
    })

v5_ranking_df = pd.DataFrame(v5_ranking_rows)
v5_source_baseline = v5_ranking_df.loc[
    v5_ranking_df['candidate'].eq('full_mse_control')
].iloc[0]
v5_ranking_df['eligible_source_guardrail'] = (
    (
        v5_ranking_df['source_rank_f1']
        >= v5_source_baseline['source_rank_f1'] - V5_SOURCE_MAX_F1_DROP
    )
    & (
        v5_ranking_df['source_rank_recall']
        >= v5_source_baseline['source_rank_recall']
        - V5_TARGET_CANDIDATE_MAX_SOURCE_RECALL_DROP
    )
    & (
        v5_ranking_df['source_rank_fpr_benign']
        <= v5_source_baseline['source_rank_fpr_benign']
        + V5_SOURCE_MAX_FPR_INCREASE
    )
)
v5_ranking_df['eligible_target_joint'] = (
    v5_ranking_df['target_rank_recall'].ge(V5_TARGET_RECALL_FLOOR)
    & v5_ranking_df['target_rank_fpr_benign'].le(
        V5_TARGET_FPR_CEILING
    )
)
v5_ranking_df['eligible_joint'] = (
    v5_ranking_df['eligible_source_guardrail']
    & v5_ranking_df['eligible_target_joint']
)

v5_joint_pool = v5_ranking_df.loc[v5_ranking_df['eligible_joint']].copy()
if v5_joint_pool.empty:
    v5_target_selection_status = (
        'FALLBACK_SOURCE_GUARDRAIL_JOINT_TARGET_OBJECTIVE_NOT_MET'
    )
    v5_joint_pool = v5_ranking_df.loc[
        v5_ranking_df['eligible_source_guardrail']
    ].copy()
else:
    v5_target_selection_status = 'JOINT_TARGET_AND_SOURCE_GUARDRAILS_MET'
if v5_joint_pool.empty:
    raise RuntimeError('Nessuna candidata v5 supera il guardrail source.')

v5_best_target_f1 = float(v5_joint_pool['target_rank_f1'].max())
v5_near_target = v5_joint_pool.loc[
    v5_joint_pool['target_rank_f1']
    >= v5_best_target_f1 - V5_NEAR_BEST_F1_MARGIN
].sort_values(
    [
        'target_rank_recall',
        'target_rank_pr_auc',
        'target_rank_fpr_benign',
        'source_rank_f1',
    ],
    ascending=[False, False, True, False],
)
v5_selected_target_name = str(v5_near_target.iloc[0]['candidate'])

v5_source_pool = v5_ranking_df.loc[
    (
        v5_ranking_df['source_rank_recall']
        >= v5_source_baseline['source_rank_recall']
        - V5_INTRA_MAX_RECALL_DROP
    )
    & (
        v5_ranking_df['source_rank_fpr_benign']
        <= v5_source_baseline['source_rank_fpr_benign']
        + V5_SOURCE_MAX_FPR_INCREASE
    )
].copy()
v5_best_source_f1 = float(v5_source_pool['source_rank_f1'].max())
v5_near_source = v5_source_pool.loc[
    v5_source_pool['source_rank_f1']
    >= v5_best_source_f1 - V5_NEAR_BEST_F1_MARGIN
].sort_values(
    [
        'source_rank_recall',
        'source_rank_pr_auc',
        'source_rank_fpr_benign',
    ],
    ascending=[False, False, True],
)
v5_selected_source_name = str(v5_near_source.iloc[0]['candidate'])
v5_thresholds_df = pd.DataFrame(v5_threshold_rows)

v5_ranking_df = v5_ranking_df.sort_values(
    [
        'eligible_joint',
        'eligible_source_guardrail',
        'target_rank_f1',
        'target_rank_recall',
    ],
    ascending=[False, False, False, False],
).reset_index(drop=True)

print('RANKING V5 - FINAL NON USATO')
print(
    v5_ranking_df[
        [
            'candidate',
            'target_rank_f1',
            'target_rank_precision',
            'target_rank_recall',
            'target_rank_fpr_benign',
            'source_rank_f1',
            'source_rank_recall',
            'eligible_joint',
        ]
    ].round(4).to_string(index=False)
)
print(f'\nCandidata target congelata: {v5_selected_target_name}')
print(f'Stato: {v5_target_selection_status}')
print(f'Candidata source congelata: {v5_selected_source_name}')


RANKING V5 - FINAL NON USATO
                candidate  target_rank_f1  target_rank_precision  target_rank_recall  target_rank_fpr_benign  source_rank_f1  source_rank_recall  eligible_joint
                soft_full          0.4350                 0.2945              0.8318                  0.4881          0.9136              0.9228           False
ensemble_stable_soft_w050          0.4214                 0.2854              0.8054                  0.4941          0.9171              0.9108           False
     ensemble_stable_w025          0.4187                 0.2801              0.8288                  0.5218          0.9173              0.9359           False
     ensemble_stable_w050          0.4113                 0.2712              0.8511                  0.5603          0.9170              0.9123           False
         full_mse_control          0.4038                 0.2556              0.9612                  0.6858          0.9188              0.9510           False
     

## Report retrospettivo sul legacy final

Solo dopo il congelamento delle candidate vengono calcolate le metriche sul
final già usato dalla v4. I risultati servono per il confronto storico, non
costituiscono una nuova conferma esterna.


In [35]:
# V5_RETRO_CELL_6_LEGACY_FINAL
if V5_RUN_LEGACY_FINAL_REPORT:
    required_legacy_final = ['idx_final', 'cda_final_idx']
    missing_legacy_final = [
        name for name in required_legacy_final if name not in globals()
    ]
    if missing_legacy_final:
        raise RuntimeError(
            f'Indici legacy final mancanti: {missing_legacy_final}'
        )

    v5_target_final_idx = np.asarray(cda_final_idx, dtype=np.int64)
    v5_source_final_idx = np.asarray(idx_final, dtype=np.int64)

    v5_target_final_values = X_target_v5[v5_target_final_idx]
    v5_target_final_fundamentals = v5_score_fundamentals(
        v5_target_final_values,
        'target',
    )
    v5_target_final_scores = v5_candidate_map(
        v5_target_final_fundamentals
    )
    del v5_target_final_values
    gc.collect()

    v5_source_final_values = X_test_internal[v5_source_final_idx]
    v5_source_final_fundamentals = v5_score_fundamentals(
        v5_source_final_values,
        'source',
    )
    v5_source_final_scores = v5_candidate_map(
        v5_source_final_fundamentals
    )
    del v5_source_final_values
    gc.collect()

    v5_target_final_y = y_stress_2017_binary[v5_target_final_idx]
    v5_source_final_y = y_test_internal_binary[v5_source_final_idx]
    target_cv = v5_cv_cache[v5_selected_target_name]['target']
    target_source_cv = v5_cv_cache[v5_selected_target_name]['source']
    source_cv = v5_cv_cache[v5_selected_source_name]['source']

    v5_target_f1_predictions = (
        v5_target_final_scores[v5_selected_target_name]
        >= target_cv['f1_threshold']
    ).astype(np.int8)
    v5_target_f2_predictions = (
        v5_target_final_scores[v5_selected_target_name]
        >= target_cv['f2_threshold']
    ).astype(np.int8)
    v5_target_candidate_source_predictions = (
        v5_source_final_scores[v5_selected_target_name]
        >= target_source_cv['f1_threshold']
    ).astype(np.int8)
    v5_source_f1_predictions = (
        v5_source_final_scores[v5_selected_source_name]
        >= source_cv['f1_threshold']
    ).astype(np.int8)
    v5_source_f2_predictions = (
        v5_source_final_scores[v5_selected_source_name]
        >= source_cv['f2_threshold']
    ).astype(np.int8)

    legacy_note = (
        'Candidata congelata su calibration/ranking; il final è lo stesso '
        'già osservato nella v4 e richiede conferma esterna.'
    )
    v5_legacy_final_rows = [
        binary_metrics_row(
            'CIC-IDS2017',
            f'v5_target_{v5_selected_target_name}__f1',
            v5_target_final_y,
            v5_target_f1_predictions,
            v5_target_final_scores[v5_selected_target_name],
            V5_LEGACY_FINAL_STATUS,
            legacy_note,
            target_cv['f1_threshold'],
        ),
        binary_metrics_row(
            'CIC-IDS2017',
            f'v5_target_{v5_selected_target_name}__f2_cal_fpr_le_030',
            v5_target_final_y,
            v5_target_f2_predictions,
            v5_target_final_scores[v5_selected_target_name],
            V5_LEGACY_FINAL_STATUS,
            legacy_note,
            target_cv['f2_threshold'],
        ),
        binary_metrics_row(
            'CSE-CIC-IDS2018',
            f'v5_target_candidate_{v5_selected_target_name}__source_guardrail',
            v5_source_final_y,
            v5_target_candidate_source_predictions,
            v5_source_final_scores[v5_selected_target_name],
            V5_LEGACY_FINAL_STATUS,
            legacy_note,
            target_source_cv['f1_threshold'],
        ),
        binary_metrics_row(
            'CSE-CIC-IDS2018',
            f'v5_source_{v5_selected_source_name}__f1',
            v5_source_final_y,
            v5_source_f1_predictions,
            v5_source_final_scores[v5_selected_source_name],
            V5_LEGACY_FINAL_STATUS,
            legacy_note,
            source_cv['f1_threshold'],
        ),
        binary_metrics_row(
            'CSE-CIC-IDS2018',
            f'v5_source_{v5_selected_source_name}__f2_cal_fpr_le_030',
            v5_source_final_y,
            v5_source_f2_predictions,
            v5_source_final_scores[v5_selected_source_name],
            V5_LEGACY_FINAL_STATUS,
            legacy_note,
            source_cv['f2_threshold'],
        ),
    ]
    v5_legacy_final_df = pd.DataFrame(v5_legacy_final_rows)

    selected_target_final_row = v5_legacy_final_df.loc[
        v5_legacy_final_df['scenario'].eq(
            f'v5_target_{v5_selected_target_name}__f1'
        )
    ].iloc[0]
    selected_target_source_row = v5_legacy_final_df.loc[
        v5_legacy_final_df['scenario'].str.contains(
            '__source_guardrail',
            regex=False,
        )
    ].iloc[0]
    selected_source_final_row = v5_legacy_final_df.loc[
        v5_legacy_final_df['scenario'].eq(
            f'v5_source_{v5_selected_source_name}__f1'
        )
    ].iloc[0]

    v5_historical_comparison_df = pd.DataFrame([
        {
            'comparison': 'target_vs_v32',
            'reference_f1': V5_HISTORICAL_TARGET_F1,
            'current_f1': selected_target_final_row['f1'],
            'delta_f1': (
                selected_target_final_row['f1']
                - V5_HISTORICAL_TARGET_F1
            ),
            'reference_recall': V5_HISTORICAL_TARGET_RECALL,
            'current_recall': selected_target_final_row['recall'],
            'delta_recall': (
                selected_target_final_row['recall']
                - V5_HISTORICAL_TARGET_RECALL
            ),
            'reference_fpr_ceiling': V5_HISTORICAL_TARGET_FPR_CEILING,
            'current_fpr': selected_target_final_row['fpr_benign'],
            'success_f1_and_recall': bool(
                selected_target_final_row['f1'] > V5_HISTORICAL_TARGET_F1
                and selected_target_final_row['recall']
                > V5_HISTORICAL_TARGET_RECALL
            ),
            'target_metrics_success': bool(
                selected_target_final_row['f1'] > V5_HISTORICAL_TARGET_F1
                and selected_target_final_row['recall']
                > V5_HISTORICAL_TARGET_RECALL
                and selected_target_final_row['fpr_benign']
                <= V5_HISTORICAL_TARGET_FPR_CEILING
            ),
            'source_guardrail_success': bool(
                selected_target_source_row['f1']
                >= V5_TARGET_CANDIDATE_SOURCE_F1_FLOOR
                and selected_target_source_row['recall']
                >= V5_TARGET_CANDIDATE_SOURCE_RECALL_FLOOR
            ),
            'cross_dataset_success_all': bool(
                selected_target_final_row['f1'] > V5_HISTORICAL_TARGET_F1
                and selected_target_final_row['recall']
                > V5_HISTORICAL_TARGET_RECALL
                and selected_target_final_row['fpr_benign']
                <= V5_HISTORICAL_TARGET_FPR_CEILING
                and selected_target_source_row['f1']
                >= V5_TARGET_CANDIDATE_SOURCE_F1_FLOOR
                and selected_target_source_row['recall']
                >= V5_TARGET_CANDIDATE_SOURCE_RECALL_FLOOR
            ),
        },
        {
            'comparison': 'source_vs_baseline_2018',
            'reference_f1': V5_SOURCE_BASELINE_F1,
            'current_f1': selected_source_final_row['f1'],
            'delta_f1': (
                selected_source_final_row['f1']
                - V5_SOURCE_BASELINE_F1
            ),
            'reference_recall': V5_SOURCE_BASELINE_RECALL,
            'current_recall': selected_source_final_row['recall'],
            'delta_recall': (
                selected_source_final_row['recall']
                - V5_SOURCE_BASELINE_RECALL
            ),
            'success_f1_and_recall': bool(
                selected_source_final_row['f1'] > V5_SOURCE_BASELINE_F1
                and selected_source_final_row['recall']
                >= V5_SOURCE_BASELINE_RECALL
            ),
        },
        {
            'comparison': 'target_candidate_source_guardrail',
            'reference_f1': V5_SOURCE_BASELINE_F1,
            'current_f1': selected_target_source_row['f1'],
            'delta_f1': (
                selected_target_source_row['f1']
                - V5_SOURCE_BASELINE_F1
            ),
            'reference_recall': V5_SOURCE_BASELINE_RECALL,
            'current_recall': selected_target_source_row['recall'],
            'delta_recall': (
                selected_target_source_row['recall']
                - V5_SOURCE_BASELINE_RECALL
            ),
            'success_f1_and_recall': bool(
                selected_target_source_row['f1']
                >= V5_TARGET_CANDIDATE_SOURCE_F1_FLOOR
                and selected_target_source_row['recall']
                >= V5_TARGET_CANDIDATE_SOURCE_RECALL_FLOOR
            ),
        },
    ])

    v5_purity_diagnostics_df = pd.DataFrame([
        {
            'reference_stage': 'initial_dominant_cluster',
            'rows': len(v5_original_reference_idx),
            'diagnostic_benign_fraction_not_used': float(np.mean(
                y_stress_2017_binary[v5_original_reference_idx] == 0
            )),
        },
        {
            'reference_stage': 'purified_core',
            'rows': len(v5_pure_core_idx),
            'diagnostic_benign_fraction_not_used': float(np.mean(
                y_stress_2017_binary[v5_pure_core_idx] == 0
            )),
        },
        {
            'reference_stage': 'build',
            'rows': len(v5_core_build_idx),
            'diagnostic_benign_fraction_not_used': float(np.mean(
                y_stress_2017_binary[v5_core_build_idx] == 0
            )),
        },
        {
            'reference_stage': 'ecdf',
            'rows': len(v5_core_ecdf_idx),
            'diagnostic_benign_fraction_not_used': float(np.mean(
                y_stress_2017_binary[v5_core_ecdf_idx] == 0
            )),
        },
    ])

    v5_attack_labels = np.asarray(y_stress_2017)[
        v5_target_final_idx
    ].astype(str)
    v5_attack_frame = pd.DataFrame({
        'attack_type': v5_attack_labels,
        'detected': v5_target_f1_predictions,
    })
    v5_attack_frame = v5_attack_frame.loc[
        v5_attack_frame['attack_type'].str.strip().str.upper().ne(
            'BENIGN'
        )
    ]
    v5_per_attack_df = (
        v5_attack_frame.groupby('attack_type', as_index=False)
        .agg(
            total_samples=('detected', 'size'),
            detected_samples=('detected', 'sum'),
        )
    )
    v5_per_attack_df['detection_rate'] = (
        v5_per_attack_df['detected_samples']
        / v5_per_attack_df['total_samples']
    )
    v5_per_attack_df['included_in_macro_recall_ge_100'] = (
        v5_per_attack_df['total_samples'] >= 100
    )
    v5_per_attack_df = v5_per_attack_df.sort_values(
        ['detection_rate', 'total_samples'],
        ascending=[True, False],
    ).reset_index(drop=True)
else:
    v5_legacy_final_df = pd.DataFrame()
    v5_historical_comparison_df = pd.DataFrame()
    v5_purity_diagnostics_df = pd.DataFrame()
    v5_per_attack_df = pd.DataFrame()

print('REPORT LEGACY FINAL V5')
if len(v5_legacy_final_df):
    print(result_matrix(v5_legacy_final_df).to_string(index=False))
    print('\nCONFRONTO STORICO')
    print(v5_historical_comparison_df.round(6).to_string(index=False))
    print('\nPUREZZA DIAGNOSTICA - NON USATA PER LA SELEZIONE')
    print(v5_purity_diagnostics_df.round(5).to_string(index=False))
else:
    print('Disattivato da V5_RUN_LEGACY_FINAL_REPORT=False.')


REPORT LEGACY FINAL V5
        dataset                                        scenario  threshold     f1  precision  recall  roc_auc  pr_auc  fpr_benign  accuracy     tn     fp    fn     tp
    CIC-IDS2017                         v5_target_soft_full__f1     0.7824 0.4344     0.2939  0.8322   0.6597  0.2543      0.4898    0.5735 193117 185420 15557  77178
    CIC-IDS2017          v5_target_soft_full__f2_cal_fpr_le_030     0.9357 0.3072     0.2476  0.4045   0.6597  0.2543      0.3011    0.6410 264573 113964 55228  37507
CSE-CIC-IDS2018 v5_target_candidate_soft_full__source_guardrail     0.8524 0.9140     0.9048  0.9235   0.9269  0.9087      0.1335    0.8995 519922  80078 63079 761139
CSE-CIC-IDS2018                  v5_source_full_mse_control__f1     0.8367 0.9192     0.8890  0.9515   0.9196  0.8978      0.1633    0.9032 502036  97964 39942 784276
CSE-CIC-IDS2018   v5_source_full_mse_control__f2_cal_fpr_le_030     0.8201 0.9156     0.8794  0.9549   0.9196  0.8978      0.1800    0.8981 49

## Esportazione degli artefatti e delle figure v5

La cella esporta configurazione, riferimenti, soglie per fold, ranking,
diagnostiche e risultati retrospettivi. Non modifica il LaTeX della tesi.


In [36]:
# V5_RETRO_CELL_7_EXPORT
V5_EXPORT_DIR = Path('Artefatti') / 'protocol_v5_retro'
V5_FIGURE_DIR = Path('Tesi') / 'figures_v5_retro'
V5_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
V5_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

v5_protocol_config_df['selected_target_candidate'] = (
    v5_selected_target_name
)
v5_protocol_config_df['selected_source_candidate'] = (
    v5_selected_source_name
)
v5_protocol_config_df['target_selection_status'] = (
    v5_target_selection_status
)
v5_protocol_config_df['source_calibration_rows'] = len(
    v5_source_cal_sample_idx
)
v5_protocol_config_df['source_ranking_rows'] = len(
    v5_source_rank_sample_idx
)
v5_protocol_config_df['target_memmap_path'] = str(
    V5_TARGET_MEMMAP_PATH.resolve()
)

v5_exports = {
    'protocol_config': v5_protocol_config_df,
    'purification_structure': v5_purification_structure_df,
    'purity_diagnostics': v5_purity_diagnostics_df,
    'feature_weights': v5_feature_weights_df,
    'gmm_diagnostics': v5_gmm_diagnostics_df,
    'capture_fold_audit': v5_capture_fold_audit_df,
    'crossfit_thresholds': v5_thresholds_df,
    'candidate_ranking': v5_ranking_df,
    'legacy_final_results': v5_legacy_final_df,
    'historical_comparison': v5_historical_comparison_df,
    'per_attack_2017': v5_per_attack_df,
}
for export_name, dataframe in v5_exports.items():
    dataframe.to_csv(
        V5_EXPORT_DIR / f'{export_name}.csv',
        index=False,
    )

v5_figure_manifest = []
ranking_plot = v5_ranking_df.sort_values('target_rank_f1').copy()
figure, axis = plt.subplots(
    figsize=(10, max(5, 0.42 * len(ranking_plot) + 1.5))
)
y_positions = np.arange(len(ranking_plot))
axis.barh(
    y_positions - 0.16,
    ranking_plot['target_rank_f1'],
    height=0.30,
    label='F1 target ranking',
    color='#2563eb',
)
axis.barh(
    y_positions + 0.16,
    ranking_plot['target_rank_recall'],
    height=0.30,
    label='Recall target ranking',
    color='#16a34a',
)
axis.set(
    yticks=y_positions,
    yticklabels=ranking_plot['candidate'],
    xlim=(0, 1),
    xlabel='Valore',
    title='Protocollo v5: F1 e recall sul ranking target',
)
axis.legend()
ranking_figure_path = V5_FIGURE_DIR / '01_ranking_f1_recall_target.png'
figure.savefig(
    ranking_figure_path,
    dpi=300,
    bbox_inches='tight',
    facecolor='white',
)
plt.close(figure)
v5_figure_manifest.append({
    'filename': ranking_figure_path.name,
    'description': 'F1 e recall delle candidate v5 sul ranking target.',
})

if len(v5_historical_comparison_df):
    comparison_plot = v5_historical_comparison_df.copy()
    figure, axis = plt.subplots(figsize=(10, 5))
    x_positions = np.arange(len(comparison_plot))
    width = 0.35
    axis.bar(
        x_positions - width / 2,
        comparison_plot['current_f1'],
        width,
        label='F1 v5',
        color='#7c3aed',
    )
    axis.bar(
        x_positions + width / 2,
        comparison_plot['current_recall'],
        width,
        label='Recall v5',
        color='#059669',
    )
    axis.set(
        xticks=x_positions,
        xticklabels=comparison_plot['comparison'],
        ylim=(0, 1),
        ylabel='Valore',
        title='Protocollo v5: confronto retrospettivo',
    )
    axis.tick_params(axis='x', labelrotation=15)
    axis.legend()
    comparison_figure_path = (
        V5_FIGURE_DIR / '02_confronto_retrospettivo.png'
    )
    figure.savefig(
        comparison_figure_path,
        dpi=300,
        bbox_inches='tight',
        facecolor='white',
    )
    plt.close(figure)
    v5_figure_manifest.append({
        'filename': comparison_figure_path.name,
        'description': (
            'Confronto retrospettivo v5; richiede conferma esterna.'
        ),
    })

v5_figure_manifest_df = pd.DataFrame(v5_figure_manifest)
v5_figure_manifest_df.to_csv(
    V5_FIGURE_DIR / 'manifest_figure_v5.csv',
    index=False,
)

print('ARTEFATTI V5 ESPORTATI')
print(f'Cartella tabelle: {V5_EXPORT_DIR.resolve()}')
print(f'Cartella figure: {V5_FIGURE_DIR.resolve()}')
print('Il LaTeX non è stato modificato.')


ARTEFATTI V5 ESPORTATI
Cartella tabelle: D:\Users\Anton\Desktop\Progetti VsCode\Codex\Rilevamento di Anomalie del Traffico di Rete con Autoencoder\Rilevamento-di-Anomalie-del-Traffico-di-Rete-con-Autoencoder\Artefatti\protocol_v5_retro
Cartella figure: D:\Users\Anton\Desktop\Progetti VsCode\Codex\Rilevamento di Anomalie del Traffico di Rete con Autoencoder\Rilevamento-di-Anomalie-del-Traffico-di-Rete-con-Autoencoder\Tesi\figures_v5_retro
Il LaTeX non è stato modificato.


## Protocollo v6 congelato e fedeltà alla letteratura

La v6 confronta un controllo v4 esatto con due sole candidate nuove:
calibrazione CADET e fine-tuning IAD. L'ablation 2×2 sul preprocessing è
diagnostica; non genera ulteriori candidate. Il legacy final resta escluso
da costruzione, calibrazione e selezione.


In [37]:
# V6_RETRO_CELL_1_CONFIG
from sklearn.linear_model import QuantileRegressor
from sklearn.preprocessing import SplineTransformer
from tensorflow.keras.models import clone_model

V6_SEED = 10123
V6_TRANSFORM_BATCH_SIZE = 100_000
V6_INFERENCE_BATCH_SIZE = 8192
V6_REFERENCE_FIT_MAX_ROWS = 100_000
V6_QT_FIT_MAX_ROWS = 100_000
V6_CADET_FIT_MAX_ROWS = 20_000
V6_CADET_EPSILON = 0.20
V6_CADET_BASELINE_QUANTILE = 1.0 - V6_CADET_EPSILON
V6_CADET_N_KNOTS = 7
V6_CADET_SPLINE_DEGREE = 3
V6_CADET_REGULARIZATION = 1e-4
V6_CADET_PRIOR_LAMBDA = 0.0
V6_IAD_TAU = 0.25
V6_IAD_MAX_ROUNDS = 3
V6_IAD_EPOCHS_PER_ROUND = 2
V6_IAD_LEARNING_RATE = 1e-4
V6_IAD_BATCH_SIZE = 2048
V6_SOURCE_MAX_F1_DROP = 0.01
V6_SOURCE_MAX_RECALL_DROP = 0.05
V6_SOURCE_MAX_FPR_INCREASE = 0.02
V6_INTRA_MAX_RECALL_DROP = 0.02
V6_RECALL_AT_FPR_LEVELS = (0.20, 0.25)
V6_RUN_LEGACY_FINAL_REPORT = True
V6_LEGACY_FINAL_STATUS = (
    'LEGACY_FINAL_POST_HOC_PROTOCOL_REQUIRES_EXTERNAL_CONFIRMATION'
)
V6_CANDIDATE_NAMES = (
    'v4_control',
    'v6_cadet_full_soft',
    'v6_iad_full_soft',
)

required_v6_variables = [
    'np',
    'pd',
    'tf',
    'gc',
    'Path',
    'QuantileTransformer',
    'plt',
    'best_model',
    'stable_core_model',
    'X_select',
    'X_test_internal',
    'X_stress_2017',
    'X_target_pseudobenign',
    'X_target_v5',
    'y_test_internal_binary',
    'y_stress_2017_binary',
    'y_stress_2017',
    'v5_original_reference_idx',
    'v5_original_build_idx',
    'v5_original_ecdf_idx',
    'v5_core_build_idx',
    'v5_core_ecdf_idx',
    'v5_source_build_idx',
    'v5_source_ecdf_idx',
    'v5_target_cal_idx',
    'v5_target_rank_idx',
    'v5_source_cal_sample_idx',
    'v5_source_rank_sample_idx',
    'v5_target_folds',
    'v5_source_folds',
    'v5_latent_median',
    'v5_latent_scale',
    'v5_full_feature_weights',
    'v5_rank01',
    'v5_select_threshold',
    'v5_metrics_from_predictions',
    'predict_latent',
    'compute_score',
    'score_metrics',
    'binary_metrics_row',
    'model_variants',
    'selected_v4_name',
    'selected_v4_state',
    'score_variant_values',
    'STABLE_FEATURE_MASK',
    'FEATURE_NAMES_FINAL',
    'INPUT_DIM',
    'n_cont',
    'clip_lo',
    'clip_hi',
    'SCORE_TOP_K',
]
missing_v6_variables = [
    name for name in required_v6_variables if name not in globals()
]
if missing_v6_variables:
    raise RuntimeError(
        'Eseguire prima tutte le celle v4 e v5 in ordine. Variabili mancanti: '
        f'{missing_v6_variables}'
    )
if selected_v4_name != 'stable_core__mse_topk_w025':
    raise RuntimeError(
        'Il controllo v4 congelato non coincide con quello atteso: '
        f'{selected_v4_name}'
    )
if V6_CADET_PRIOR_LAMBDA != 0.0:
    raise RuntimeError('La v6 congela CADET senza prior di hardness.')
if len(V6_CANDIDATE_NAMES) != 3:
    raise RuntimeError('La v6 deve contenere esattamente tre candidate.')

v6_protocol_config_df = pd.DataFrame([{
    'protocol': 'v6_retro_frozen',
    'seed': V6_SEED,
    'candidates': ' | '.join(V6_CANDIDATE_NAMES),
    'candidate_count': len(V6_CANDIDATE_NAMES),
    'cadet_epsilon': V6_CADET_EPSILON,
    'cadet_conditional_quantiles': '0.25 | 0.75 | 0.80',
    'cadet_spline_knots': V6_CADET_N_KNOTS,
    'cadet_prior_lambda': V6_CADET_PRIOR_LAMBDA,
    'iad_tau': V6_IAD_TAU,
    'iad_max_rounds': V6_IAD_MAX_ROUNDS,
    'iad_epochs_per_round': V6_IAD_EPOCHS_PER_ROUND,
    'iad_learning_rate': V6_IAD_LEARNING_RATE,
    'preprocessing_for_new_candidates': (
        'continuous_normality_weights_plus_source_scaling'
    ),
    'selection_order': (
        'source_guardrail > target_pr_auc > recall_at_fpr_020 > '
        'recall_at_fpr_025 > crossfitted_f1'
    ),
    'legacy_final_status': V6_LEGACY_FINAL_STATUS,
}])

v6_literature_fidelity_df = pd.DataFrame([
    {
        'method': 'CADET',
        'faithful_component': (
            'conditional quantiles with cubic B-spline basis; '
            'IQR conditional spread'
        ),
        'local_adaptation': (
            'soft drift-weighted reconstruction MSE; lambda prior fixed to 0'
        ),
        'labels_used_for_fit': False,
    },
    {
        'method': 'IAD',
        'faithful_component': (
            'score-statistic sigmoid weights and median-crossing termination'
        ),
        'local_adaptation': (
            'three light fine-tuning rounds on full AE with soft feature loss'
        ),
        'labels_used_for_fit': False,
    },
])

print('PROTOCOLLO V6 CONGELATO')
print(v6_protocol_config_df.to_string(index=False))
print('\nFEDELTÀ E ADATTAMENTI')
print(v6_literature_fidelity_df.to_string(index=False))
print('Il legacy final non viene usato prima della cella post-hoc dedicata.')


PROTOCOLLO V6 CONGELATO
       protocol  seed                                         candidates  candidate_count  cadet_epsilon cadet_conditional_quantiles  cadet_spline_knots  cadet_prior_lambda  iad_tau  iad_max_rounds  iad_epochs_per_round  iad_learning_rate                 preprocessing_for_new_candidates                                                                           selection_order                                           legacy_final_status
v6_retro_frozen 10123 v4_control | v6_cadet_full_soft | v6_iad_full_soft                3            0.2          0.25 | 0.75 | 0.80                   7                 0.0     0.25               3                     2             0.0001 continuous_normality_weights_plus_source_scaling source_guardrail > target_pr_auc > recall_at_fpr_020 > recall_at_fpr_025 > crossfitted_f1 LEGACY_FINAL_POST_HOC_PROTOCOL_REQUIRES_EXTERNAL_CONFIRMATION

FEDELTÀ E ADATTAMENTI
method                                                      faithful_comp

## Pesi continui di normalità e trasformazione target alternativa

Il cluster dominante non viene tagliato. Distanza latente, MSE full e MSE
stable producono un burden consensuale; la sigmoide IAD lo converte in pesi
continui. Viene inoltre costruita, su memmap, la sola variante target-QT
necessaria all'ablation 2×2.


In [38]:
# V6_RETRO_CELL_2_CONTINUOUS_WEIGHTS
def v6_iad_weight_update(scores, statistics=None):
    scores = np.asarray(scores, dtype=np.float64)
    if not np.isfinite(scores).all():
        raise ValueError('Score non finiti nell aggiornamento IAD.')
    if statistics is None:
        minimum = float(np.min(scores))
        median = float(np.median(scores))
        maximum = float(np.max(scores))
        side = min(median - minimum, maximum - median)
        alpha = 1.0 / max(side * V6_IAD_TAU, 1e-8)
        statistics = {
            'minimum': minimum,
            'median': median,
            'maximum': maximum,
            'alpha': alpha,
        }
    logits = np.clip(
        float(statistics['alpha'])
        * (scores - float(statistics['median'])),
        -60.0,
        60.0,
    )
    weights = 1.0 / (1.0 + np.exp(logits))
    return weights.astype(np.float32), dict(statistics)


def v6_consensus_components(indices):
    indices = np.asarray(indices, dtype=np.int64)
    values = X_stress_2017[indices]
    latent = predict_latent(
        best_model,
        values,
        batch_size=V6_INFERENCE_BATCH_SIZE,
    )
    latent_distance = np.sqrt(np.mean(
        np.square(
            (latent - v5_latent_median) / v5_latent_scale
        ),
        axis=1,
    ))
    full_mse = compute_score(best_model, values)
    stable_mse = compute_score(
        stable_core_model,
        values[:, STABLE_FEATURE_MASK],
    )
    del values, latent
    gc.collect()
    return {
        'latent_distance': latent_distance.astype(np.float32),
        'full_mse': full_mse.astype(np.float32),
        'stable_mse': stable_mse.astype(np.float32),
    }


v6_build_components = v6_consensus_components(v5_original_build_idx)
v6_ecdf_components = v6_consensus_components(v5_original_ecdf_idx)
v6_build_burden = np.maximum.reduce([
    v5_rank01(v6_build_components['latent_distance']),
    v5_rank01(v6_build_components['full_mse']),
    v5_rank01(v6_build_components['stable_mse']),
]).astype(np.float32)
v6_ecdf_burden = np.maximum.reduce([
    empirical_cdf(
        v6_ecdf_components['latent_distance'],
        np.sort(v6_build_components['latent_distance']),
    ),
    empirical_cdf(
        v6_ecdf_components['full_mse'],
        np.sort(v6_build_components['full_mse']),
    ),
    empirical_cdf(
        v6_ecdf_components['stable_mse'],
        np.sort(v6_build_components['stable_mse']),
    ),
]).astype(np.float32)
v6_build_normality_weights, v6_burden_weight_statistics = (
    v6_iad_weight_update(v6_build_burden)
)
v6_ecdf_normality_weights, _ = v6_iad_weight_update(
    v6_ecdf_burden,
    statistics=v6_burden_weight_statistics,
)

assert len(v6_build_normality_weights) == len(v5_original_build_idx)
assert len(v6_ecdf_normality_weights) == len(v5_original_ecdf_idx)
assert np.all((v6_build_normality_weights > 0.0))
assert np.all((v6_build_normality_weights < 1.0))
assert np.all((v6_ecdf_normality_weights > 0.0))
assert np.all((v6_ecdf_normality_weights < 1.0))

v6_weight_rng = np.random.default_rng(V6_SEED + 101)
v6_qt_fit_size = min(
    V6_QT_FIT_MAX_ROWS,
    len(v5_original_build_idx),
)
v6_qt_probabilities = (
    v6_build_normality_weights
    / np.sum(v6_build_normality_weights)
)
v6_qt_fit_positions = v6_weight_rng.choice(
    len(v5_original_build_idx),
    size=v6_qt_fit_size,
    replace=True,
    p=v6_qt_probabilities,
)
v6_continuous_quantile = QuantileTransformer(
    n_quantiles=min(1000, v6_qt_fit_size),
    output_distribution='normal',
    subsample=v6_qt_fit_size,
    random_state=V6_SEED + 102,
)
v6_continuous_quantile.fit(
    X_stress_2017[
        np.asarray(v5_original_build_idx)[v6_qt_fit_positions],
        :n_cont,
    ]
)

V6_TARGET_CACHE_DIR = (
    Path('Artefatti') / 'protocol_v6_retro' / 'cache'
)
V6_TARGET_CACHE_DIR.mkdir(parents=True, exist_ok=True)
V6_CONTINUOUS_QT_MEMMAP_PATH = (
    V6_TARGET_CACHE_DIR / 'X_target_v6_continuous_qt_float32.npy'
)
X_target_v6_continuous_qt = np.lib.format.open_memmap(
    V6_CONTINUOUS_QT_MEMMAP_PATH,
    mode='w+',
    dtype=np.float32,
    shape=X_stress_2017.shape,
)
V6_CONTINUOUS_QT_STATE = 'IN_PROGRESS'
try:
    for v6_start in range(
        0,
        len(X_target_v6_continuous_qt),
        V6_TRANSFORM_BATCH_SIZE,
    ):
        v6_end = min(
            v6_start + V6_TRANSFORM_BATCH_SIZE,
            len(X_target_v6_continuous_qt),
        )
        v6_transformed = v6_continuous_quantile.transform(
            X_stress_2017[v6_start:v6_end, :n_cont]
        ).astype(np.float32, copy=False)
        X_target_v6_continuous_qt[v6_start:v6_end, :n_cont] = np.clip(
            v6_transformed,
            clip_lo,
            clip_hi,
        )
        X_target_v6_continuous_qt[v6_start:v6_end, n_cont:] = (
            X_stress_2017[v6_start:v6_end, n_cont:]
        )
    X_target_v6_continuous_qt.flush()
except Exception:
    V6_CONTINUOUS_QT_STATE = 'FAILED_PARTIAL_RESTART_REQUIRED'
    raise
V6_CONTINUOUS_QT_STATE = 'COMPLETE'

v6_normality_weight_summary_df = pd.DataFrame([
    {
        'partition': 'build',
        'rows': len(v6_build_normality_weights),
        'weight_mean': float(np.mean(v6_build_normality_weights)),
        'weight_median': float(np.median(v6_build_normality_weights)),
        'weight_p05': float(np.quantile(
            v6_build_normality_weights,
            0.05,
        )),
        'weight_p95': float(np.quantile(
            v6_build_normality_weights,
            0.95,
        )),
        'labels_used': False,
    },
    {
        'partition': 'ecdf',
        'rows': len(v6_ecdf_normality_weights),
        'weight_mean': float(np.mean(v6_ecdf_normality_weights)),
        'weight_median': float(np.median(v6_ecdf_normality_weights)),
        'weight_p05': float(np.quantile(
            v6_ecdf_normality_weights,
            0.05,
        )),
        'weight_p95': float(np.quantile(
            v6_ecdf_normality_weights,
            0.95,
        )),
        'labels_used': False,
    },
])

del (
    v6_build_components,
    v6_ecdf_components,
    v6_qt_probabilities,
    v6_qt_fit_positions,
    v6_transformed,
)
gc.collect()

print('PESI CONTINUI V6 E TARGET-QT COMPLETATI SENZA LABEL')
print(v6_normality_weight_summary_df.round(5).to_string(index=False))


PESI CONTINUI V6 E TARGET-QT COMPLETATI SENZA LABEL
partition   rows  weight_mean  weight_median  weight_p05  weight_p95  labels_used
    build 106773      0.47849        0.50000     0.02286     0.99078        False
     ecdf  71183      0.48143        0.50635     0.02296     0.99161        False


## Ablation 2×2: purificazione hard/continua e scaling target/source

Le quattro combinazioni condividono modello e split. Cambiano soltanto
riferimento robusto e trasformazione. L'esito è riportato come diagnostica:
CADET e IAD restano congelati su pesi continui più solo scaling source,
indipendentemente dal vincitore dell'ablation.


In [39]:
# V6_RETRO_CELL_3_PREPROCESSING_ABLATION
def v6_weighted_quantile_1d(values, weights, quantile):
    values = np.asarray(values, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    order = np.argsort(values, kind='mergesort')
    sorted_values = values[order]
    sorted_weights = np.maximum(weights[order], 0.0)
    total = float(np.sum(sorted_weights))
    if total <= 0.0:
        raise ValueError('Somma dei pesi nulla nel quantile pesato.')
    centers = (
        np.cumsum(sorted_weights) - 0.5 * sorted_weights
    ) / total
    return float(np.interp(
        float(quantile),
        centers,
        sorted_values,
        left=sorted_values[0],
        right=sorted_values[-1],
    ))


def v6_weighted_column_quantile(values, weights, quantile):
    return np.asarray([
        v6_weighted_quantile_1d(values[:, column], weights, quantile)
        for column in range(values.shape[1])
    ], dtype=np.float32)


def v6_soft_raw_components(model, values, reference):
    output = {'soft_mse': [], 'soft_topk': []}
    feature_weights = np.asarray(
        reference['feature_weights'],
        dtype=np.float32,
    )
    feature_weight_sum = max(float(feature_weights.sum()), 1e-8)
    k = min(int(SCORE_TOP_K), values.shape[1])
    for start in range(0, len(values), V6_INFERENCE_BATCH_SIZE):
        batch = values[start:start + V6_INFERENCE_BATCH_SIZE]
        reconstruction = model.predict_on_batch(batch)
        residual = np.square(batch - reconstruction).astype(np.float32)
        robust_z = np.maximum(
            (residual - reference['residual_median'])
            / reference['residual_scale'],
            0.0,
        )
        weighted_z = robust_z * feature_weights[None, :]
        top_values = np.partition(
            weighted_z,
            -k,
            axis=1,
        )[:, -k:]
        output['soft_mse'].append(
            np.sum(
                residual * feature_weights[None, :],
                axis=1,
            ) / feature_weight_sum
        )
        output['soft_topk'].append(np.mean(top_values, axis=1))
    return {
        key: np.concatenate(parts).astype(np.float32, copy=False)
        for key, parts in output.items()
    }


def v6_make_weighted_ecdf(values, weights):
    order = np.argsort(values, kind='mergesort')
    sorted_values = np.asarray(values, dtype=np.float32)[order]
    sorted_weights = np.asarray(weights, dtype=np.float64)[order]
    cumulative = np.cumsum(np.maximum(sorted_weights, 0.0))
    if cumulative[-1] <= 0.0:
        raise ValueError('ECDF pesata con somma dei pesi nulla.')
    cumulative /= cumulative[-1]
    return {
        'values': sorted_values,
        'cumulative': cumulative.astype(np.float32),
    }


def v6_apply_weighted_ecdf(values, ecdf):
    positions = np.searchsorted(
        ecdf['values'],
        np.asarray(values),
        side='right',
    ) - 1
    result = np.zeros(len(values), dtype=np.float32)
    valid = positions >= 0
    result[valid] = ecdf['cumulative'][positions[valid]]
    return result


def v6_fit_soft_reference(
    model,
    build_values,
    build_weights,
    ecdf_values,
    ecdf_weights,
    seed,
):
    build_weights = np.asarray(build_weights, dtype=np.float32)
    ecdf_weights = np.asarray(ecdf_weights, dtype=np.float32)
    if len(build_values) != len(build_weights):
        raise ValueError('Build e pesi hanno lunghezze diverse.')
    if len(ecdf_values) != len(ecdf_weights):
        raise ValueError('ECDF e pesi hanno lunghezze diverse.')

    if len(build_values) > V6_REFERENCE_FIT_MAX_ROWS:
        rng = np.random.default_rng(seed)
        positions = rng.choice(
            len(build_values),
            size=V6_REFERENCE_FIT_MAX_ROWS,
            replace=False,
        )
        fit_values = build_values[positions]
        fit_weights = build_weights[positions]
    else:
        fit_values = build_values
        fit_weights = build_weights

    residual_parts = []
    for start in range(0, len(fit_values), V6_INFERENCE_BATCH_SIZE):
        batch = fit_values[start:start + V6_INFERENCE_BATCH_SIZE]
        reconstruction = model.predict_on_batch(batch)
        residual_parts.append(
            np.square(batch - reconstruction).astype(np.float32)
        )
    residual = np.concatenate(residual_parts, axis=0)
    residual_median = v6_weighted_column_quantile(
        residual,
        fit_weights,
        0.50,
    )
    residual_mad = v6_weighted_column_quantile(
        np.abs(residual - residual_median[None, :]),
        fit_weights,
        0.50,
    )
    reference = {
        'residual_median': residual_median,
        'residual_scale': np.maximum(1.4826 * residual_mad, 1e-6),
        'feature_weights': np.asarray(
            v5_full_feature_weights,
            dtype=np.float32,
        ),
    }
    ecdf_raw = v6_soft_raw_components(model, ecdf_values, reference)
    reference['ecdf'] = {
        key: v6_make_weighted_ecdf(component, ecdf_weights)
        for key, component in ecdf_raw.items()
    }
    del residual_parts, residual, ecdf_raw
    gc.collect()
    return reference


def v6_score_soft(model, values, reference):
    raw = v6_soft_raw_components(model, values, reference)
    soft_mse_percentile = v6_apply_weighted_ecdf(
        raw['soft_mse'],
        reference['ecdf']['soft_mse'],
    )
    soft_topk_percentile = v6_apply_weighted_ecdf(
        raw['soft_topk'],
        reference['ecdf']['soft_topk'],
    )
    result = (
        0.75 * soft_mse_percentile
        + 0.25 * soft_topk_percentile
    ).astype(np.float32)
    del raw, soft_mse_percentile, soft_topk_percentile
    return result


def v6_crossfit_f1_thresholds(
    y_true,
    scores,
    folds,
    domain,
    candidate,
):
    y_true = np.asarray(y_true, dtype=np.int8)
    scores = np.asarray(scores, dtype=np.float32)
    coverage = np.zeros(len(y_true), dtype=np.int8)
    oof_predictions = np.zeros(len(y_true), dtype=np.int8)
    thresholds = []
    for fold_number, (train_positions, valid_positions) in enumerate(
        folds,
        start=1,
    ):
        threshold = v5_select_threshold(
            y_true[train_positions],
            scores[train_positions],
            beta=1.0,
            max_fpr=1.0,
        )
        thresholds.append(threshold)
        oof_predictions[valid_positions] = (
            scores[valid_positions] >= threshold
        ).astype(np.int8)
        coverage[valid_positions] += 1
        v6_threshold_rows.append({
            'domain': domain,
            'candidate': candidate,
            'fold': fold_number,
            'policy': 'f1',
            'threshold': threshold,
        })
    assert np.all(coverage == 1)
    median_threshold = float(np.median(thresholds))
    v6_threshold_rows.append({
        'domain': domain,
        'candidate': candidate,
        'fold': 'aggregate',
        'policy': 'f1_median',
        'threshold': median_threshold,
    })
    return {
        'threshold': median_threshold,
        'threshold_iqr': float(
            np.percentile(thresholds, 75)
            - np.percentile(thresholds, 25)
        ),
        'oof_metrics': v5_metrics_from_predictions(
            y_true,
            oof_predictions,
            scores,
        ),
    }


def v6_recall_at_fpr(y_true, scores, target_fpr):
    y_true = np.asarray(y_true, dtype=np.int8)
    scores = np.asarray(scores, dtype=np.float64)
    benign_scores = scores[y_true == 0]
    attack_scores = scores[y_true == 1]
    if not len(benign_scores) or not len(attack_scores):
        raise ValueError('Entrambe le classi sono richieste per Recall@FPR.')
    rank = int(np.ceil((1.0 - float(target_fpr)) * len(benign_scores))) - 1
    rank = int(np.clip(rank, 0, len(benign_scores) - 1))
    threshold = float(np.sort(benign_scores)[rank])
    return {
        'recall': float(np.mean(attack_scores >= threshold)),
        'threshold': threshold,
        'actual_fpr': float(np.mean(benign_scores >= threshold)),
    }


v6_threshold_rows = []
v6_preprocessing_specs = {
    'hard_target_qt': {
        'values': X_target_v5,
        'build_idx': np.asarray(v5_core_build_idx, dtype=np.int64),
        'ecdf_idx': np.asarray(v5_core_ecdf_idx, dtype=np.int64),
        'build_weights': np.ones(len(v5_core_build_idx), dtype=np.float32),
        'ecdf_weights': np.ones(len(v5_core_ecdf_idx), dtype=np.float32),
        'purification': 'hard_55_percent',
        'transformation': 'target_quantile',
    },
    'hard_source_scaling': {
        'values': X_stress_2017,
        'build_idx': np.asarray(v5_core_build_idx, dtype=np.int64),
        'ecdf_idx': np.asarray(v5_core_ecdf_idx, dtype=np.int64),
        'build_weights': np.ones(len(v5_core_build_idx), dtype=np.float32),
        'ecdf_weights': np.ones(len(v5_core_ecdf_idx), dtype=np.float32),
        'purification': 'hard_55_percent',
        'transformation': 'source_scaling_only',
    },
    'continuous_target_qt': {
        'values': X_target_v6_continuous_qt,
        'build_idx': np.asarray(v5_original_build_idx, dtype=np.int64),
        'ecdf_idx': np.asarray(v5_original_ecdf_idx, dtype=np.int64),
        'build_weights': v6_build_normality_weights,
        'ecdf_weights': v6_ecdf_normality_weights,
        'purification': 'continuous_weights',
        'transformation': 'target_quantile',
    },
    'continuous_source_scaling': {
        'values': X_stress_2017,
        'build_idx': np.asarray(v5_original_build_idx, dtype=np.int64),
        'ecdf_idx': np.asarray(v5_original_ecdf_idx, dtype=np.int64),
        'build_weights': v6_build_normality_weights,
        'ecdf_weights': v6_ecdf_normality_weights,
        'purification': 'continuous_weights',
        'transformation': 'source_scaling_only',
    },
}

v6_preprocessing_references = {}
v6_preprocessing_scores = {}
v6_preprocessing_rows = []
v6_target_cal_y = y_stress_2017_binary[v5_target_cal_idx]
v6_target_rank_y = y_stress_2017_binary[v5_target_rank_idx]
for position, (name, spec) in enumerate(
    v6_preprocessing_specs.items(),
    start=1,
):
    print(f'ABLATION PREPROCESSING {position}/4: {name}')
    reference = v6_fit_soft_reference(
        best_model,
        spec['values'][spec['build_idx']],
        spec['build_weights'],
        spec['values'][spec['ecdf_idx']],
        spec['ecdf_weights'],
        seed=V6_SEED + 200 + position,
    )
    calibration_scores = v6_score_soft(
        best_model,
        spec['values'][v5_target_cal_idx],
        reference,
    )
    ranking_scores = v6_score_soft(
        best_model,
        spec['values'][v5_target_rank_idx],
        reference,
    )
    cv = v6_crossfit_f1_thresholds(
        v6_target_cal_y,
        calibration_scores,
        v5_target_folds,
        domain='target_preprocessing_ablation',
        candidate=name,
    )
    rank_metrics = score_metrics(
        v6_target_rank_y,
        ranking_scores,
        cv['threshold'],
    )
    recall_020 = v6_recall_at_fpr(
        v6_target_rank_y,
        ranking_scores,
        0.20,
    )
    recall_025 = v6_recall_at_fpr(
        v6_target_rank_y,
        ranking_scores,
        0.25,
    )
    v6_preprocessing_references[name] = reference
    v6_preprocessing_scores[name] = {
        'calibration': calibration_scores,
        'ranking': ranking_scores,
        'cv': cv,
    }
    v6_preprocessing_rows.append({
        'variant': name,
        'purification': spec['purification'],
        'transformation': spec['transformation'],
        'target_rank_f1': rank_metrics['f1'],
        'target_rank_precision': rank_metrics['precision'],
        'target_rank_recall': rank_metrics['recall'],
        'target_rank_fpr_benign': rank_metrics['fpr_benign'],
        'target_rank_roc_auc': rank_metrics['roc_auc'],
        'target_rank_pr_auc': rank_metrics['pr_auc'],
        'target_rank_recall_at_fpr_020': recall_020['recall'],
        'target_rank_recall_at_fpr_025': recall_025['recall'],
        'target_threshold': cv['threshold'],
        'eligible_for_candidate_selection': False,
    })

v6_preprocessing_ablation_df = pd.DataFrame(
    v6_preprocessing_rows
).sort_values(
    [
        'target_rank_pr_auc',
        'target_rank_recall_at_fpr_020',
        'target_rank_recall_at_fpr_025',
        'target_rank_f1',
    ],
    ascending=False,
).reset_index(drop=True)
v6_continuous_source_reference = v6_preprocessing_references[
    'continuous_source_scaling'
]

print('\nABLATION 2×2 V6 - NON CREA CANDIDATE')
print(v6_preprocessing_ablation_df.round(5).to_string(index=False))
print(
    'Le candidate CADET/IAD restano fissate su continuous_source_scaling.'
)


ABLATION PREPROCESSING 1/4: hard_target_qt
ABLATION PREPROCESSING 2/4: hard_source_scaling
ABLATION PREPROCESSING 3/4: continuous_target_qt
ABLATION PREPROCESSING 4/4: continuous_source_scaling

ABLATION 2×2 V6 - NON CREA CANDIDATE
                  variant       purification      transformation  target_rank_f1  target_rank_precision  target_rank_recall  target_rank_fpr_benign  target_rank_roc_auc  target_rank_pr_auc  target_rank_recall_at_fpr_020  target_rank_recall_at_fpr_025  target_threshold  eligible_for_candidate_selection
      hard_source_scaling    hard_55_percent source_scaling_only         0.46058                0.35339             0.66109                 0.29634              0.68493             0.30319                        0.50675                        0.58827           0.81937                             False
continuous_source_scaling continuous_weights source_scaling_only         0.46822                0.36595             0.64982                 0.27583              0

## CADET: calibrazione condizionale della hardness

La hardness tabulare è la distanza dal vettore medio del riferimento. Tre
regressioni quantili su base B-spline stimano quantile 0,80 e IQR
condizionali. Il prior di hardness è congelato a zero per isolare la
correzione dei falsi positivi dovuti alla difficoltà di ricostruzione.


In [40]:
# V6_RETRO_CELL_4_CADET
def v6_soft_reconstruction_mse(model, values):
    parts = []
    feature_weights = np.asarray(
        v5_full_feature_weights,
        dtype=np.float32,
    )
    denominator = max(float(feature_weights.sum()), 1e-8)
    for start in range(0, len(values), V6_INFERENCE_BATCH_SIZE):
        batch = values[start:start + V6_INFERENCE_BATCH_SIZE]
        reconstruction = model.predict_on_batch(batch)
        residual = np.square(batch - reconstruction).astype(np.float32)
        parts.append(
            np.sum(
                residual * feature_weights[None, :],
                axis=1,
            ) / denominator
        )
    return np.concatenate(parts).astype(np.float32, copy=False)


def v6_weighted_mean(values, weights):
    weights = np.asarray(weights, dtype=np.float64)
    denominator = max(float(np.sum(weights)), 1e-12)
    return np.sum(
        np.asarray(values, dtype=np.float64) * weights[:, None],
        axis=0,
    ) / denominator


def v6_cadet_hardness(values, null_mean):
    return np.linalg.norm(
        np.asarray(values, dtype=np.float32)
        - np.asarray(null_mean, dtype=np.float32)[None, :],
        axis=1,
    ).astype(np.float32)


def v6_fit_cadet(model, values, weights, seed, domain):
    weights = np.asarray(weights, dtype=np.float32)
    if len(values) != len(weights):
        raise ValueError('CADET: valori e pesi hanno lunghezze diverse.')
    null_mean = v6_weighted_mean(values, weights).astype(np.float32)
    hardness = v6_cadet_hardness(values, null_mean)
    errors = v6_soft_reconstruction_mse(model, values)

    rng = np.random.default_rng(seed)
    fit_size = min(V6_CADET_FIT_MAX_ROWS, len(values))
    if fit_size < len(values):
        positions = rng.choice(
            len(values),
            size=fit_size,
            replace=False,
        )
    else:
        positions = np.arange(len(values), dtype=np.int64)
    fit_hardness = hardness[positions, None]
    fit_errors = errors[positions]
    fit_weights = weights[positions]
    spline = SplineTransformer(
        n_knots=V6_CADET_N_KNOTS,
        degree=V6_CADET_SPLINE_DEGREE,
        knots='quantile',
        include_bias=True,
        extrapolation='linear',
    )
    basis = spline.fit_transform(fit_hardness)
    quantile_models = {}
    for quantile in (0.25, 0.75, V6_CADET_BASELINE_QUANTILE):
        estimator = QuantileRegressor(
            quantile=float(quantile),
            alpha=V6_CADET_REGULARIZATION,
            fit_intercept=False,
            solver='highs',
        )
        estimator.fit(
            basis,
            fit_errors,
            sample_weight=fit_weights,
        )
        quantile_models[float(quantile)] = estimator

    q25 = quantile_models[0.25].predict(basis)
    q75 = quantile_models[0.75].predict(basis)
    positive_spread = np.maximum(q75 - q25, 0.0)
    spread_floor = max(
        float(np.quantile(positive_spread, 0.10)),
        float(np.quantile(fit_errors, 0.50)) * 1e-3,
        1e-8,
    )
    state = {
        'model': model,
        'null_mean': null_mean,
        'spline': spline,
        'quantile_models': quantile_models,
        'spread_floor': spread_floor,
        'hardness_mean': float(np.mean(hardness)),
        'hardness_std': max(float(np.std(hardness)), 1e-8),
        'domain': domain,
    }
    diagnostics = {
        'domain': domain,
        'fit_rows': fit_size,
        'reference_rows': len(values),
        'baseline_quantile': V6_CADET_BASELINE_QUANTILE,
        'spread_floor': spread_floor,
        'hardness_mean': state['hardness_mean'],
        'hardness_std': state['hardness_std'],
        'prior_lambda': V6_CADET_PRIOR_LAMBDA,
        'labels_used': False,
    }
    del hardness, errors, positions, fit_weights, basis
    gc.collect()
    return state, diagnostics


def v6_score_cadet(state, values):
    hardness = v6_cadet_hardness(values, state['null_mean'])
    errors = v6_soft_reconstruction_mse(state['model'], values)
    basis = state['spline'].transform(hardness[:, None])
    q25 = state['quantile_models'][0.25].predict(basis)
    q75 = state['quantile_models'][0.75].predict(basis)
    baseline = state['quantile_models'][
        V6_CADET_BASELINE_QUANTILE
    ].predict(basis)
    spread = np.maximum(q75 - q25, state['spread_floor'])
    conditional_score = (errors - baseline) / spread
    hardness_prior = np.abs(
        (hardness - state['hardness_mean'])
        / state['hardness_std']
    )
    result = (
        conditional_score
        + V6_CADET_PRIOR_LAMBDA * hardness_prior
    ).astype(np.float32)
    if not np.isfinite(result).all():
        raise RuntimeError('Score CADET non finiti.')
    return result


v6_cadet_target_state, v6_cadet_target_diagnostics = v6_fit_cadet(
    best_model,
    X_stress_2017[v5_original_build_idx],
    v6_build_normality_weights,
    seed=V6_SEED + 401,
    domain='target',
)
v6_cadet_source_state, v6_cadet_source_diagnostics = v6_fit_cadet(
    best_model,
    X_select[v5_source_build_idx],
    np.ones(len(v5_source_build_idx), dtype=np.float32),
    seed=V6_SEED + 402,
    domain='source',
)
v6_cadet_diagnostics_df = pd.DataFrame([
    v6_cadet_target_diagnostics,
    v6_cadet_source_diagnostics,
])

v6_cadet_target_cal_scores = v6_score_cadet(
    v6_cadet_target_state,
    X_stress_2017[v5_target_cal_idx],
)
v6_cadet_target_rank_scores = v6_score_cadet(
    v6_cadet_target_state,
    X_stress_2017[v5_target_rank_idx],
)
v6_cadet_source_cal_scores = v6_score_cadet(
    v6_cadet_source_state,
    X_test_internal[v5_source_cal_sample_idx],
)
v6_cadet_source_rank_scores = v6_score_cadet(
    v6_cadet_source_state,
    X_test_internal[v5_source_rank_sample_idx],
)

print('CADET V6 COMPLETATO SENZA NUOVO TRAINING AE')
print(v6_cadet_diagnostics_df.round(6).to_string(index=False))


CADET V6 COMPLETATO SENZA NUOVO TRAINING AE
domain  fit_rows  reference_rows  baseline_quantile  spread_floor  hardness_mean  hardness_std  prior_lambda  labels_used
target     20000          106773                0.8      0.011330      14.481923      5.407555           0.0        False
source     20000           90000                0.8      0.000974      17.643372      3.534058           0.0        False


## IAD: fine-tuning iterativo con pesi continui

Il full AE viene clonato e aggiornato con loss combined per campione,
ponderata sia per drift di feature sia per normalità del campione. Ogni round
ricalcola i pesi con la sigmoide IAD; il modello scelto è quello con il minor
numero di attraversamenti della partizione mediana, senza label.


In [41]:
# V6_RETRO_CELL_5_IAD
v6_tf_feature_weights = tf.constant(
    np.asarray(v5_full_feature_weights, dtype=np.float32),
    dtype=tf.float32,
)
v6_tf_feature_weight_sum = tf.reduce_sum(v6_tf_feature_weights)


def v6_soft_combined_per_sample(y_true, y_pred):
    absolute = tf.abs(y_true - y_pred)
    squared = tf.square(y_true - y_pred)
    mae = tf.reduce_sum(
        absolute * v6_tf_feature_weights,
        axis=-1,
    ) / v6_tf_feature_weight_sum
    mse = tf.reduce_sum(
        squared * v6_tf_feature_weights,
        axis=-1,
    ) / v6_tf_feature_weight_sum
    return 0.5 * mae + 0.5 * mse


def v6_median_partition_crossings(previous_scores, current_scores):
    previous_scores = np.asarray(previous_scores)
    current_scores = np.asarray(current_scores)
    if len(previous_scores) != len(current_scores):
        raise ValueError('IAD: ranking con lunghezze diverse.')
    halfway = len(previous_scores) // 2
    previous_order = np.argsort(previous_scores, kind='mergesort')
    current_order = np.argsort(current_scores, kind='mergesort')
    previous_low = np.zeros(len(previous_scores), dtype=bool)
    current_low = np.zeros(len(current_scores), dtype=bool)
    previous_low[previous_order[:halfway]] = True
    current_low[current_order[:halfway]] = True
    return int(np.count_nonzero(previous_low != current_low))


v6_iad_model = clone_model(best_model)
v6_iad_model.set_weights(best_model.get_weights())
v6_iad_model.compile(
    optimizer=Adam(learning_rate=V6_IAD_LEARNING_RATE),
    loss=v6_soft_combined_per_sample,
)
v6_iad_training_values = X_stress_2017[v5_original_reference_idx]
v6_iad_previous_scores = v6_soft_reconstruction_mse(
    v6_iad_model,
    v6_iad_training_values,
)
v6_iad_round_states = []
v6_iad_history_rows = []

for v6_round in range(1, V6_IAD_MAX_ROUNDS + 1):
    v6_round_weights, v6_round_weight_statistics = (
        v6_iad_weight_update(v6_iad_previous_scores)
    )
    tf.random.set_seed(V6_SEED + 500 + v6_round)
    v6_round_history = v6_iad_model.fit(
        v6_iad_training_values,
        v6_iad_training_values,
        sample_weight=v6_round_weights,
        epochs=V6_IAD_EPOCHS_PER_ROUND,
        batch_size=V6_IAD_BATCH_SIZE,
        shuffle=True,
        verbose=2,
        callbacks=[tf.keras.callbacks.TerminateOnNaN()],
    )
    v6_iad_current_scores = v6_soft_reconstruction_mse(
        v6_iad_model,
        v6_iad_training_values,
    )
    v6_crossings = v6_median_partition_crossings(
        v6_iad_previous_scores,
        v6_iad_current_scores,
    )
    v6_iad_round_states.append({
        'round': v6_round,
        'crossings': v6_crossings,
        'weights': [
            value.copy() for value in v6_iad_model.get_weights()
        ],
    })
    v6_iad_history_rows.append({
        'round': v6_round,
        'epochs': V6_IAD_EPOCHS_PER_ROUND,
        'final_loss': float(v6_round_history.history['loss'][-1]),
        'median_partition_crossings': v6_crossings,
        'crossing_fraction': v6_crossings / len(v6_iad_current_scores),
        'sample_weight_mean': float(np.mean(v6_round_weights)),
        'sample_weight_median': float(np.median(v6_round_weights)),
        'score_median': float(np.median(v6_iad_current_scores)),
        'labels_used_for_training_or_termination': False,
    })
    print(
        f'IAD round {v6_round}/{V6_IAD_MAX_ROUNDS}: '
        f'crossings={v6_crossings:,}, '
        f'loss={v6_round_history.history["loss"][-1]:.6f}'
    )
    v6_iad_previous_scores = v6_iad_current_scores

v6_iad_selected_state = min(
    v6_iad_round_states,
    key=lambda item: (item['crossings'], item['round']),
)
v6_iad_selected_round = int(v6_iad_selected_state['round'])
v6_iad_model.set_weights(v6_iad_selected_state['weights'])
v6_iad_history_df = pd.DataFrame(v6_iad_history_rows)
v6_iad_history_df['selected_round'] = (
    v6_iad_history_df['round'] == v6_iad_selected_round
)

v6_iad_build_scores = v6_soft_reconstruction_mse(
    v6_iad_model,
    X_stress_2017[v5_original_build_idx],
)
v6_iad_ecdf_scores = v6_soft_reconstruction_mse(
    v6_iad_model,
    X_stress_2017[v5_original_ecdf_idx],
)
v6_iad_build_weights, v6_iad_reference_weight_statistics = (
    v6_iad_weight_update(v6_iad_build_scores)
)
v6_iad_ecdf_weights, _ = v6_iad_weight_update(
    v6_iad_ecdf_scores,
    statistics=v6_iad_reference_weight_statistics,
)
v6_iad_target_reference = v6_fit_soft_reference(
    v6_iad_model,
    X_stress_2017[v5_original_build_idx],
    v6_iad_build_weights,
    X_stress_2017[v5_original_ecdf_idx],
    v6_iad_ecdf_weights,
    seed=V6_SEED + 551,
)
v6_iad_source_reference = v6_fit_soft_reference(
    v6_iad_model,
    X_select[v5_source_build_idx],
    np.ones(len(v5_source_build_idx), dtype=np.float32),
    X_select[v5_source_ecdf_idx],
    np.ones(len(v5_source_ecdf_idx), dtype=np.float32),
    seed=V6_SEED + 552,
)

v6_iad_target_cal_scores = v6_score_soft(
    v6_iad_model,
    X_stress_2017[v5_target_cal_idx],
    v6_iad_target_reference,
)
v6_iad_target_rank_scores = v6_score_soft(
    v6_iad_model,
    X_stress_2017[v5_target_rank_idx],
    v6_iad_target_reference,
)
v6_iad_source_cal_scores = v6_score_soft(
    v6_iad_model,
    X_test_internal[v5_source_cal_sample_idx],
    v6_iad_source_reference,
)
v6_iad_source_rank_scores = v6_score_soft(
    v6_iad_model,
    X_test_internal[v5_source_rank_sample_idx],
    v6_iad_source_reference,
)

del (
    v6_iad_training_values,
    v6_iad_previous_scores,
    v6_iad_current_scores,
    v6_round_weights,
    v6_round_history,
    v6_iad_round_states,
    v6_iad_selected_state,
)
gc.collect()

print('\nIAD V6 COMPLETATO')
print(v6_iad_history_df.round(6).to_string(index=False))
print(f'Round selezionato senza label: {v6_iad_selected_round}')


Epoch 1/2
87/87 - 1s - loss: 0.0306 - 1s/epoch - 12ms/step
Epoch 2/2
87/87 - 0s - loss: 0.0210 - 458ms/epoch - 5ms/step
IAD round 1/3: crossings=31,904, loss=0.020974
Epoch 1/2
87/87 - 1s - loss: 0.0130 - 635ms/epoch - 7ms/step
Epoch 2/2
87/87 - 1s - loss: 0.0123 - 517ms/epoch - 6ms/step
IAD round 2/3: crossings=12,338, loss=0.012309
Epoch 1/2
87/87 - 1s - loss: 0.0122 - 502ms/epoch - 6ms/step
Epoch 2/2
87/87 - 0s - loss: 0.0118 - 404ms/epoch - 5ms/step
IAD round 3/3: crossings=8,124, loss=0.011810

IAD V6 COMPLETATO
 round  epochs  final_loss  median_partition_crossings  crossing_fraction  sample_weight_mean  sample_weight_median  score_median  labels_used_for_training_or_termination  selected_round
     1       2    0.020974                       31904           0.179280            0.479643                   0.5      0.038190                                    False           False
     2       2    0.012309                       12338           0.069332            0.434295          

## Selezione v6 cross-dataset e ramo intra-dataset

Il controllo v4 viene ricalcolato con preprocessing e score originali, ma con
la stessa calibrazione cross-fitted delle nuove candidate. Prima si applica
il guardrail source; poi si ordinano PR-AUC, Recall@FPR 0,20/0,25 e infine
F1. La scelta intra-dataset è separata.


In [42]:
# V6_RETRO_CELL_6_SELECTION
v6_control_variant = model_variants[
    selected_v4_state['variant_name']
]
v6_control_recipe = selected_v4_state['recipe']
v6_control_target_cal_scores = score_variant_values(
    v6_control_variant,
    X_target_pseudobenign[v5_target_cal_idx],
    'target',
    v6_control_recipe,
)
v6_control_target_rank_scores = score_variant_values(
    v6_control_variant,
    X_target_pseudobenign[v5_target_rank_idx],
    'target',
    v6_control_recipe,
)
v6_control_source_cal_scores = score_variant_values(
    v6_control_variant,
    X_test_internal[v5_source_cal_sample_idx],
    'source',
    v6_control_recipe,
)
v6_control_source_rank_scores = score_variant_values(
    v6_control_variant,
    X_test_internal[v5_source_rank_sample_idx],
    'source',
    v6_control_recipe,
)

v6_candidate_scores = {
    'v4_control': {
        'target_cal': v6_control_target_cal_scores,
        'target_rank': v6_control_target_rank_scores,
        'source_cal': v6_control_source_cal_scores,
        'source_rank': v6_control_source_rank_scores,
    },
    'v6_cadet_full_soft': {
        'target_cal': v6_cadet_target_cal_scores,
        'target_rank': v6_cadet_target_rank_scores,
        'source_cal': v6_cadet_source_cal_scores,
        'source_rank': v6_cadet_source_rank_scores,
    },
    'v6_iad_full_soft': {
        'target_cal': v6_iad_target_cal_scores,
        'target_rank': v6_iad_target_rank_scores,
        'source_cal': v6_iad_source_cal_scores,
        'source_rank': v6_iad_source_rank_scores,
    },
}
assert tuple(v6_candidate_scores) == V6_CANDIDATE_NAMES

v6_source_cal_y = y_test_internal_binary[v5_source_cal_sample_idx]
v6_source_rank_y = y_test_internal_binary[v5_source_rank_sample_idx]
v6_cv_cache = {}
v6_ranking_rows = []
for candidate_name in V6_CANDIDATE_NAMES:
    scores = v6_candidate_scores[candidate_name]
    target_cv = v6_crossfit_f1_thresholds(
        v6_target_cal_y,
        scores['target_cal'],
        v5_target_folds,
        domain='target_candidate',
        candidate=candidate_name,
    )
    source_cv = v6_crossfit_f1_thresholds(
        v6_source_cal_y,
        scores['source_cal'],
        v5_source_folds,
        domain='source_candidate',
        candidate=candidate_name,
    )
    target_rank_metrics = score_metrics(
        v6_target_rank_y,
        scores['target_rank'],
        target_cv['threshold'],
    )
    source_rank_metrics = score_metrics(
        v6_source_rank_y,
        scores['source_rank'],
        source_cv['threshold'],
    )
    recall_020 = v6_recall_at_fpr(
        v6_target_rank_y,
        scores['target_rank'],
        0.20,
    )
    recall_025 = v6_recall_at_fpr(
        v6_target_rank_y,
        scores['target_rank'],
        0.25,
    )
    v6_cv_cache[candidate_name] = {
        'target': target_cv,
        'source': source_cv,
    }
    v6_ranking_rows.append({
        'candidate': candidate_name,
        'target_rank_f1': target_rank_metrics['f1'],
        'target_rank_precision': target_rank_metrics['precision'],
        'target_rank_recall': target_rank_metrics['recall'],
        'target_rank_fpr_benign': target_rank_metrics['fpr_benign'],
        'target_rank_roc_auc': target_rank_metrics['roc_auc'],
        'target_rank_pr_auc': target_rank_metrics['pr_auc'],
        'target_rank_recall_at_fpr_020': recall_020['recall'],
        'target_rank_recall_at_fpr_025': recall_025['recall'],
        'target_cal_oof_f1': target_cv['oof_metrics']['f1'],
        'target_threshold': target_cv['threshold'],
        'target_threshold_iqr': target_cv['threshold_iqr'],
        'source_rank_f1': source_rank_metrics['f1'],
        'source_rank_precision': source_rank_metrics['precision'],
        'source_rank_recall': source_rank_metrics['recall'],
        'source_rank_fpr_benign': source_rank_metrics['fpr_benign'],
        'source_rank_roc_auc': source_rank_metrics['roc_auc'],
        'source_rank_pr_auc': source_rank_metrics['pr_auc'],
        'source_cal_oof_f1': source_cv['oof_metrics']['f1'],
        'source_threshold': source_cv['threshold'],
        'source_threshold_iqr': source_cv['threshold_iqr'],
    })

v6_ranking_df = pd.DataFrame(v6_ranking_rows)
v6_control_row = v6_ranking_df.loc[
    v6_ranking_df['candidate'].eq('v4_control')
].iloc[0]
v6_ranking_df['eligible_source_guardrail'] = (
    (
        v6_ranking_df['source_rank_f1']
        >= v6_control_row['source_rank_f1'] - V6_SOURCE_MAX_F1_DROP
    )
    & (
        v6_ranking_df['source_rank_recall']
        >= v6_control_row['source_rank_recall'] - V6_SOURCE_MAX_RECALL_DROP
    )
    & (
        v6_ranking_df['source_rank_fpr_benign']
        <= (
            v6_control_row['source_rank_fpr_benign']
            + V6_SOURCE_MAX_FPR_INCREASE
        )
    )
)
v6_target_pool = v6_ranking_df.loc[
    v6_ranking_df['eligible_source_guardrail']
].sort_values(
    [
        'target_rank_pr_auc',
        'target_rank_recall_at_fpr_020',
        'target_rank_recall_at_fpr_025',
        'target_rank_f1',
        'target_rank_recall',
    ],
    ascending=False,
)
if v6_target_pool.empty:
    raise RuntimeError('Nessuna candidata v6 supera il guardrail source.')
v6_selected_target_name = str(v6_target_pool.iloc[0]['candidate'])

v6_source_pool = v6_ranking_df.loc[
    (
        v6_ranking_df['source_rank_recall']
        >= (
            v6_control_row['source_rank_recall']
            - V6_INTRA_MAX_RECALL_DROP
        )
    )
    & (
        v6_ranking_df['source_rank_fpr_benign']
        <= (
            v6_control_row['source_rank_fpr_benign']
            + V6_SOURCE_MAX_FPR_INCREASE
        )
    )
].sort_values(
    [
        'source_rank_f1',
        'source_rank_recall',
        'source_rank_pr_auc',
        'source_rank_fpr_benign',
    ],
    ascending=[False, False, False, True],
)
if v6_source_pool.empty:
    raise RuntimeError('Nessuna candidata v6 eleggibile sul source.')
v6_selected_source_name = str(v6_source_pool.iloc[0]['candidate'])

v6_ranking_df['selected_target'] = (
    v6_ranking_df['candidate'] == v6_selected_target_name
)
v6_ranking_df['selected_source'] = (
    v6_ranking_df['candidate'] == v6_selected_source_name
)
v6_ranking_df = v6_ranking_df.sort_values(
    [
        'eligible_source_guardrail',
        'target_rank_pr_auc',
        'target_rank_recall_at_fpr_020',
        'target_rank_recall_at_fpr_025',
        'target_rank_f1',
    ],
    ascending=False,
).reset_index(drop=True)
v6_thresholds_df = pd.DataFrame(v6_threshold_rows)

print('RANKING V6 - LEGACY FINAL NON USATO')
print(v6_ranking_df.round(5).to_string(index=False))
print(f'\nCandidata cross-dataset: {v6_selected_target_name}')
print(f'Candidata intra-dataset: {v6_selected_source_name}')


RANKING V6 - LEGACY FINAL NON USATO
         candidate  target_rank_f1  target_rank_precision  target_rank_recall  target_rank_fpr_benign  target_rank_roc_auc  target_rank_pr_auc  target_rank_recall_at_fpr_020  target_rank_recall_at_fpr_025  target_cal_oof_f1  target_threshold  target_threshold_iqr  source_rank_f1  source_rank_precision  source_rank_recall  source_rank_fpr_benign  source_rank_roc_auc  source_rank_pr_auc  source_cal_oof_f1  source_threshold  source_threshold_iqr  eligible_source_guardrail  selected_target  selected_source
        v4_control         0.52709                0.43323             0.67285                 0.21565              0.68804             0.31492                        0.64814                        0.69299            0.50346           0.80985               0.00259         0.91654                0.92771             0.90563                 0.09694              0.92376             0.91609            0.91650           0.88432               0.00006          

## Protocollo v7 congelato: adapter low-rank anti-forgetting

La v7 mantiene congelato l'autoencoder source e addestra senza label target
soltanto un adapter residuo low-rank. Il replay di benigni 2018 usa come
target la ricostruzione del modello source, preservandone la funzione. La
selezione avviene prima di qualsiasi lettura del legacy final.


In [43]:
# V7_RETRO_CELL_1_CONFIG
from tensorflow.keras.layers import Add
from tensorflow.keras.regularizers import l2

V7_SEED = 11123
V7_INFERENCE_BATCH_SIZE = 8192
V7_ADAPTER_RANK = 4
V7_ADAPTER_L2 = 1e-5
V7_LEARNING_RATE = 5e-4
V7_MAX_ROUNDS = 2
V7_EPOCHS_PER_ROUND = 2
V7_BATCH_SIZE = 2048
V7_SOURCE_REPLAY_FRACTION = 0.25
V7_SOURCE_REPLAY_WEIGHT = 1.0
V7_ROUND_MAX_F1_DROP = 0.01
V7_ROUND_MAX_RECALL_DROP = 0.03
V7_ROUND_MAX_FPR_INCREASE = 0.02
V7_SOURCE_MAX_F1_DROP = 0.01
V7_SOURCE_MAX_RECALL_DROP = 0.05
V7_SOURCE_MAX_FPR_INCREASE = 0.02
V7_INTRA_MAX_RECALL_DROP = 0.02
V7_RUN_LEGACY_FINAL_REPORT = True
V7_LEGACY_FINAL_STATUS = (
    'LEGACY_FINAL_POST_HOC_PROTOCOL_REQUIRES_EXTERNAL_CONFIRMATION'
)
V7_CANDIDATE_NAMES = (
    'v4_control',
    'v7_adapter_ecdf',
    'v7_adapter_robust_log',
)

required_v7_variables = [
    'np',
    'pd',
    'tf',
    'gc',
    'Path',
    'plt',
    'Input',
    'Dense',
    'Model',
    'Adam',
    'clone_model',
    'best_model',
    'X_train',
    'X_select',
    'X_test_internal',
    'X_stress_2017',
    'X_target_pseudobenign',
    'y_test_internal_binary',
    'y_stress_2017_binary',
    'y_stress_2017',
    'v5_original_reference_idx',
    'v5_original_build_idx',
    'v5_original_ecdf_idx',
    'v5_source_build_idx',
    'v5_source_ecdf_idx',
    'v5_target_cal_idx',
    'v5_target_rank_idx',
    'v5_source_cal_sample_idx',
    'v5_source_rank_sample_idx',
    'v5_target_folds',
    'v5_source_folds',
    'v5_full_feature_weights',
    'v5_select_threshold',
    'v5_metrics_from_predictions',
    'score_metrics',
    'binary_metrics_row',
    'result_matrix',
    'selected_v4_name',
    'v6_build_normality_weights',
    'v6_ecdf_normality_weights',
    'v6_iad_weight_update',
    'v6_soft_reconstruction_mse',
    'v6_soft_combined_per_sample',
    'v6_median_partition_crossings',
    'v6_fit_soft_reference',
    'v6_soft_raw_components',
    'v6_apply_weighted_ecdf',
    'v6_recall_at_fpr',
    'v6_control_variant',
    'v6_control_recipe',
    'v6_control_target_cal_scores',
    'v6_control_target_rank_scores',
    'v6_control_source_cal_scores',
    'v6_control_source_rank_scores',
    'v6_target_cal_y',
    'v6_target_rank_y',
    'v6_source_cal_y',
    'v6_source_rank_y',
    'INPUT_DIM',
]
missing_v7_variables = [
    name for name in required_v7_variables if name not in globals()
]
if missing_v7_variables:
    raise RuntimeError(
        'Eseguire prima le celle v4-v6 fino alla selezione v6. '
        f'Variabili mancanti: {missing_v7_variables}'
    )
if selected_v4_name != 'stable_core__mse_topk_w025':
    raise RuntimeError(
        'Il controllo v4 congelato non coincide con quello atteso: '
        f'{selected_v4_name}'
    )
if len(V7_CANDIDATE_NAMES) != 3:
    raise RuntimeError('La v7 deve contenere esattamente tre candidate.')
if V7_MAX_ROUNDS != 2:
    raise RuntimeError('La v7 congela esattamente due round di adapter.')

v7_threshold_rows = []


def v7_crossfit_f1_thresholds(
    y_true,
    scores,
    folds,
    domain,
    candidate,
):
    y_true = np.asarray(y_true, dtype=np.int8)
    scores = np.asarray(scores, dtype=np.float32)
    coverage = np.zeros(len(y_true), dtype=np.int8)
    oof_predictions = np.zeros(len(y_true), dtype=np.int8)
    thresholds = []
    for fold_number, (train_positions, valid_positions) in enumerate(
        folds,
        start=1,
    ):
        threshold = v5_select_threshold(
            y_true[train_positions],
            scores[train_positions],
            beta=1.0,
            max_fpr=1.0,
        )
        thresholds.append(threshold)
        oof_predictions[valid_positions] = (
            scores[valid_positions] >= threshold
        ).astype(np.int8)
        coverage[valid_positions] += 1
        v7_threshold_rows.append({
            'domain': domain,
            'candidate': candidate,
            'fold': fold_number,
            'policy': 'f1',
            'threshold': threshold,
        })
    assert np.all(coverage == 1)
    median_threshold = float(np.median(thresholds))
    v7_threshold_rows.append({
        'domain': domain,
        'candidate': candidate,
        'fold': 'aggregate',
        'policy': 'f1_median',
        'threshold': median_threshold,
    })
    return {
        'threshold': median_threshold,
        'threshold_iqr': float(
            np.percentile(thresholds, 75)
            - np.percentile(thresholds, 25)
        ),
        'oof_metrics': v5_metrics_from_predictions(
            y_true,
            oof_predictions,
            scores,
        ),
    }


v7_protocol_config_df = pd.DataFrame([{
    'protocol': 'v7_retro_frozen',
    'seed': V7_SEED,
    'candidates': ' | '.join(V7_CANDIDATE_NAMES),
    'candidate_count': len(V7_CANDIDATE_NAMES),
    'base_model_updates': 0,
    'adapter_rank': V7_ADAPTER_RANK,
    'adapter_l2': V7_ADAPTER_L2,
    'learning_rate': V7_LEARNING_RATE,
    'max_rounds': V7_MAX_ROUNDS,
    'epochs_per_round': V7_EPOCHS_PER_ROUND,
    'source_replay_fraction': V7_SOURCE_REPLAY_FRACTION,
    'source_replay_target': 'frozen_source_reconstruction',
    'target_labels_used_for_training': False,
    'preprocessing': 'source_scaling_only',
    'selection_order': (
        'source_guardrail > target_pr_auc > recall_at_fpr_020 > '
        'recall_at_fpr_025 > crossfitted_f1'
    ),
    'legacy_final_status': V7_LEGACY_FINAL_STATUS,
}])

print('PROTOCOLLO V7 CONGELATO')
print(v7_protocol_config_df.to_string(index=False))
print(
    'Il modello source resta congelato; vengono aggiornati soltanto '
    'i due layer dell adapter.'
)
print('Il legacy final non viene letto prima della cella post-hoc dedicata.')


PROTOCOLLO V7 CONGELATO
       protocol  seed                                           candidates  candidate_count  base_model_updates  adapter_rank  adapter_l2  learning_rate  max_rounds  epochs_per_round  source_replay_fraction         source_replay_target  target_labels_used_for_training       preprocessing                                                                           selection_order                                           legacy_final_status
v7_retro_frozen 11123 v4_control | v7_adapter_ecdf | v7_adapter_robust_log                3                   0             4     0.00001         0.0005           2                 2                    0.25 frozen_source_reconstruction                            False source_scaling_only source_guardrail > target_pr_auc > recall_at_fpr_020 > recall_at_fpr_025 > crossfitted_f1 LEGACY_FINAL_POST_HOC_PROTOCOL_REQUIRES_EXTERNAL_CONFIRMATION
Il modello source resta congelato; vengono aggiornati soltanto i due layer dell adapter.
Il le

## Addestramento non supervisionato dell'adapter con replay source

Il target di adaptation usa ricostruzione pesata e continua. Un campione
benigno source ogni quattro campioni target viene distillato dalla copia
congelata dell'AE: l'adapter è quindi penalizzato se altera la funzione
source. Il round viene scelto esclusivamente tramite guardrail source.


In [44]:
# V7_RETRO_CELL_2_ADAPTER_TRAINING
v7_frozen_base = clone_model(best_model)
v7_frozen_base.set_weights(best_model.get_weights())
v7_frozen_base.trainable = False
v7_frozen_features = Model(
    v7_frozen_base.input,
    [
        v7_frozen_base.get_layer('latent').output,
        v7_frozen_base.output,
    ],
    name='v7_frozen_source_features',
)
v7_frozen_features.trainable = False

v7_input = Input(shape=(INPUT_DIM,), name='v7_adapter_input')
v7_latent, v7_base_reconstruction = v7_frozen_features(
    v7_input,
    training=False,
)
v7_adapter_hidden = Dense(
    V7_ADAPTER_RANK,
    activation='tanh',
    kernel_regularizer=l2(V7_ADAPTER_L2),
    name='v7_adapter_down',
)(v7_latent)
v7_adapter_delta = Dense(
    INPUT_DIM,
    kernel_initializer='zeros',
    bias_initializer='zeros',
    kernel_regularizer=l2(V7_ADAPTER_L2),
    name='v7_adapter_up',
)(v7_adapter_hidden)
v7_reconstruction = Add(name='v7_adapter_reconstruction')([
    v7_base_reconstruction,
    v7_adapter_delta,
])
v7_adapter_model = Model(
    v7_input,
    v7_reconstruction,
    name='v7_low_rank_residual_adapter',
)
v7_adapter_model.compile(
    optimizer=Adam(learning_rate=V7_LEARNING_RATE),
    loss=v6_soft_combined_per_sample,
)
v7_trainable_names = [
    variable.name for variable in v7_adapter_model.trainable_weights
]
if not v7_trainable_names or not all(
    'v7_adapter_' in name for name in v7_trainable_names
):
    raise RuntimeError(
        'Sono diventati trainabili pesi diversi dall adapter: '
        f'{v7_trainable_names}'
    )

v7_weight_lookup = np.zeros(len(X_stress_2017), dtype=np.float32)
v7_weight_coverage = np.zeros(len(X_stress_2017), dtype=bool)
v7_weight_lookup[v5_original_build_idx] = v6_build_normality_weights
v7_weight_lookup[v5_original_ecdf_idx] = v6_ecdf_normality_weights
v7_weight_coverage[v5_original_build_idx] = True
v7_weight_coverage[v5_original_ecdf_idx] = True
v7_target_indices = np.asarray(
    v5_original_reference_idx,
    dtype=np.int64,
)
if not np.all(v7_weight_coverage[v7_target_indices]):
    raise RuntimeError(
        'I pesi continui non coprono tutto il riferimento target v7.'
    )
v7_target_consensus_weights = v7_weight_lookup[
    v7_target_indices
].copy()
del v7_weight_lookup, v7_weight_coverage

v7_target_training_values = X_stress_2017[v7_target_indices]
v7_source_rng = np.random.default_rng(V7_SEED + 10)
v7_source_replay_rows = min(
    len(X_train),
    max(
        1,
        int(round(
            V7_SOURCE_REPLAY_FRACTION
            * len(v7_target_training_values)
        )),
    ),
)
v7_source_replay_indices = v7_source_rng.choice(
    len(X_train),
    size=v7_source_replay_rows,
    replace=False,
)
v7_source_replay_values = X_train[v7_source_replay_indices]
v7_source_distillation_targets = v7_frozen_base.predict(
    v7_source_replay_values,
    batch_size=V7_INFERENCE_BATCH_SIZE,
    verbose=0,
).astype(np.float32, copy=False)


def v7_audit_source_round(model, round_number):
    reference = v6_fit_soft_reference(
        model,
        X_select[v5_source_build_idx],
        np.ones(len(v5_source_build_idx), dtype=np.float32),
        X_select[v5_source_ecdf_idx],
        np.ones(len(v5_source_ecdf_idx), dtype=np.float32),
        seed=V7_SEED + 100 + int(round_number),
    )
    calibration_scores = v6_score_soft(
        model,
        X_test_internal[v5_source_cal_sample_idx],
        reference,
    )
    ranking_scores = v6_score_soft(
        model,
        X_test_internal[v5_source_rank_sample_idx],
        reference,
    )
    cv = v7_crossfit_f1_thresholds(
        v6_source_cal_y,
        calibration_scores,
        v5_source_folds,
        domain='source_adapter_round_audit',
        candidate=f'v7_adapter_round_{round_number}',
    )
    metrics = score_metrics(
        v6_source_rank_y,
        ranking_scores,
        cv['threshold'],
    )
    return {
        'source_rank_f1': metrics['f1'],
        'source_rank_precision': metrics['precision'],
        'source_rank_recall': metrics['recall'],
        'source_rank_fpr_benign': metrics['fpr_benign'],
        'source_rank_roc_auc': metrics['roc_auc'],
        'source_rank_pr_auc': metrics['pr_auc'],
        'source_threshold': cv['threshold'],
        'source_threshold_iqr': cv['threshold_iqr'],
    }


v7_round_states = [{
    'round': 0,
    'weights': [
        value.copy() for value in v7_adapter_model.get_weights()
    ],
}]
v7_round_rows = [{
    'round': 0,
    'epochs': 0,
    'training_loss': np.nan,
    'median_partition_crossings': 0,
    'crossing_fraction': 0.0,
    'target_weight_mean': float(
        np.mean(v7_target_consensus_weights)
    ),
    'target_weight_median': float(
        np.median(v7_target_consensus_weights)
    ),
    'target_labels_used_for_training_or_stopping': False,
    **v7_audit_source_round(v7_adapter_model, 0),
}]
v7_previous_target_scores = v6_soft_reconstruction_mse(
    v7_adapter_model,
    v7_target_training_values,
)

for v7_round in range(1, V7_MAX_ROUNDS + 1):
    v7_dynamic_weights, _ = v6_iad_weight_update(
        v7_previous_target_scores
    )
    v7_target_round_weights = np.sqrt(
        np.maximum(v7_target_consensus_weights, 1e-8)
        * np.maximum(v7_dynamic_weights, 1e-8)
    ).astype(np.float32)
    v7_mixed_inputs = np.concatenate([
        v7_target_training_values,
        v7_source_replay_values,
    ], axis=0)
    v7_mixed_targets = np.concatenate([
        v7_target_training_values,
        v7_source_distillation_targets,
    ], axis=0)
    v7_mixed_weights = np.concatenate([
        v7_target_round_weights,
        np.full(
            len(v7_source_replay_values),
            V7_SOURCE_REPLAY_WEIGHT,
            dtype=np.float32,
        ),
    ])

    tf.random.set_seed(V7_SEED + 200 + v7_round)
    v7_history = v7_adapter_model.fit(
        v7_mixed_inputs,
        v7_mixed_targets,
        sample_weight=v7_mixed_weights,
        epochs=V7_EPOCHS_PER_ROUND,
        batch_size=V7_BATCH_SIZE,
        shuffle=True,
        verbose=2,
        callbacks=[tf.keras.callbacks.TerminateOnNaN()],
    )
    v7_current_target_scores = v6_soft_reconstruction_mse(
        v7_adapter_model,
        v7_target_training_values,
    )
    v7_crossings = v6_median_partition_crossings(
        v7_previous_target_scores,
        v7_current_target_scores,
    )
    v7_round_states.append({
        'round': v7_round,
        'weights': [
            value.copy() for value in v7_adapter_model.get_weights()
        ],
    })
    v7_round_rows.append({
        'round': v7_round,
        'epochs': V7_EPOCHS_PER_ROUND,
        'training_loss': float(v7_history.history['loss'][-1]),
        'median_partition_crossings': v7_crossings,
        'crossing_fraction': (
            v7_crossings / len(v7_current_target_scores)
        ),
        'target_weight_mean': float(
            np.mean(v7_target_round_weights)
        ),
        'target_weight_median': float(
            np.median(v7_target_round_weights)
        ),
        'target_labels_used_for_training_or_stopping': False,
        **v7_audit_source_round(v7_adapter_model, v7_round),
    })
    print(
        f'Adapter v7 round {v7_round}/{V7_MAX_ROUNDS}: '
        f'crossings={v7_crossings:,}, '
        f'loss={v7_history.history["loss"][-1]:.6f}'
    )
    v7_previous_target_scores = v7_current_target_scores
    del (
        v7_mixed_inputs,
        v7_mixed_targets,
        v7_mixed_weights,
        v7_history,
    )
    gc.collect()

v7_adapter_round_audit_df = pd.DataFrame(v7_round_rows)
v7_round_zero = v7_adapter_round_audit_df.loc[
    v7_adapter_round_audit_df['round'].eq(0)
].iloc[0]
v7_adapter_round_audit_df['eligible_source_round_guardrail'] = (
    (
        v7_adapter_round_audit_df['source_rank_f1']
        >= v7_round_zero['source_rank_f1'] - V7_ROUND_MAX_F1_DROP
    )
    & (
        v7_adapter_round_audit_df['source_rank_recall']
        >= (
            v7_round_zero['source_rank_recall']
            - V7_ROUND_MAX_RECALL_DROP
        )
    )
    & (
        v7_adapter_round_audit_df['source_rank_fpr_benign']
        <= (
            v7_round_zero['source_rank_fpr_benign']
            + V7_ROUND_MAX_FPR_INCREASE
        )
    )
)
v7_eligible_rounds = v7_adapter_round_audit_df.loc[
    v7_adapter_round_audit_df['eligible_source_round_guardrail']
]
if v7_eligible_rounds.empty:
    raise RuntimeError(
        'Nemmeno il round zero supera il guardrail source interno.'
    )
v7_selected_round = int(v7_eligible_rounds['round'].max())
v7_selected_round_state = next(
    state
    for state in v7_round_states
    if state['round'] == v7_selected_round
)
v7_adapter_model.set_weights(v7_selected_round_state['weights'])
v7_adapter_round_audit_df['selected_round'] = (
    v7_adapter_round_audit_df['round'] == v7_selected_round
)

del (
    v7_round_states,
    v7_round_rows,
    v7_selected_round_state,
    v7_eligible_rounds,
    v7_previous_target_scores,
    v7_current_target_scores,
    v7_dynamic_weights,
    v7_target_round_weights,
    v7_source_distillation_targets,
    v7_target_training_values,
)
gc.collect()

print('\nAUDIT ROUND ADAPTER V7')
print(v7_adapter_round_audit_df.round(6).to_string(index=False))
print(f'Round selezionato tramite solo guardrail source: {v7_selected_round}')


Epoch 1/2
109/109 - 1s - loss: 0.0573 - 826ms/epoch - 8ms/step
Epoch 2/2
109/109 - 0s - loss: 0.0527 - 428ms/epoch - 4ms/step
Adapter v7 round 1/2: crossings=7,524, loss=0.052659
Epoch 1/2
109/109 - 1s - loss: 0.0474 - 540ms/epoch - 5ms/step
Epoch 2/2
109/109 - 0s - loss: 0.0432 - 453ms/epoch - 4ms/step
Adapter v7 round 2/2: crossings=11,960, loss=0.043247

AUDIT ROUND ADAPTER V7
 round  epochs  training_loss  median_partition_crossings  crossing_fraction  target_weight_mean  target_weight_median  target_labels_used_for_training_or_stopping  source_rank_f1  source_rank_precision  source_rank_recall  source_rank_fpr_benign  source_rank_roc_auc  source_rank_pr_auc  source_threshold  source_threshold_iqr  eligible_source_round_guardrail  selected_round
     0       0            NaN                           0           0.000000            0.479665              0.502545                                        False        0.913597               0.904612            0.922762                0.

## Score duale v7: ECDF e robust-log non saturo

La stessa ricostruzione dell'adapter produce due candidate di score. Il
controllo ECDF replica la normalizzazione v6; `robust_log` usa mediana e IQR
del log-score e conserva informazione oltre il massimo del riferimento.
Vengono esportati tie rate e saturazione per ogni split.


In [45]:
# V7_RETRO_CELL_3_DUAL_SCORING
def v7_reference_quantile(ecdf, quantile):
    position = int(np.searchsorted(
        ecdf['cumulative'],
        float(quantile),
        side='left',
    ))
    position = int(np.clip(position, 0, len(ecdf['values']) - 1))
    return float(ecdf['values'][position])


def v7_attach_robust_log_reference(reference):
    robust_log = {}
    for component, ecdf in reference['ecdf'].items():
        q25 = np.log1p(max(v7_reference_quantile(ecdf, 0.25), 0.0))
        q50 = np.log1p(max(v7_reference_quantile(ecdf, 0.50), 0.0))
        q75 = np.log1p(max(v7_reference_quantile(ecdf, 0.75), 0.0))
        robust_log[component] = {
            'center': float(q50),
            'scale': float(max(q75 - q25, 1e-6)),
        }
    reference['robust_log'] = robust_log
    return reference


def v7_score_bundle(model, values, reference, split_name, collector):
    raw = v6_soft_raw_components(model, values, reference)
    ecdf_components = {
        key: v6_apply_weighted_ecdf(
            component,
            reference['ecdf'][key],
        )
        for key, component in raw.items()
    }
    robust_components = {
        key: (
            np.log1p(np.maximum(component, 0.0))
            - reference['robust_log'][key]['center']
        ) / reference['robust_log'][key]['scale']
        for key, component in raw.items()
    }
    scores = {
        'ecdf': (
            0.75 * ecdf_components['soft_mse']
            + 0.25 * ecdf_components['soft_topk']
        ).astype(np.float32),
        'robust_log': (
            0.75 * robust_components['soft_mse']
            + 0.25 * robust_components['soft_topk']
        ).astype(np.float32),
    }
    for component, values_raw in raw.items():
        reference_max = float(reference['ecdf'][component]['values'][-1])
        collector.append({
            'split': split_name,
            'score_kind': f'raw_{component}',
            'rows': len(values_raw),
            'fraction_at_or_above_reference_max': float(np.mean(
                values_raw >= reference_max
            )),
            'fraction_ge_099': np.nan,
            'fraction_ge_0999': np.nan,
            'fraction_equal_max': np.nan,
            'unique_fraction_rounded_1e7': np.nan,
        })
    for score_kind, score_values in scores.items():
        maximum = float(np.max(score_values))
        collector.append({
            'split': split_name,
            'score_kind': score_kind,
            'rows': len(score_values),
            'fraction_at_or_above_reference_max': np.nan,
            'fraction_ge_099': float(np.mean(score_values >= 0.99))
            if score_kind == 'ecdf' else np.nan,
            'fraction_ge_0999': float(np.mean(score_values >= 0.999))
            if score_kind == 'ecdf' else np.nan,
            'fraction_equal_max': float(np.mean(np.isclose(
                score_values,
                maximum,
                rtol=0.0,
                atol=1e-7,
            ))),
            'unique_fraction_rounded_1e7': (
                len(np.unique(np.round(score_values, 7)))
                / max(len(score_values), 1)
            ),
        })
    del raw, ecdf_components, robust_components
    return scores


v7_target_build_current_scores = v6_soft_reconstruction_mse(
    v7_adapter_model,
    X_stress_2017[v5_original_build_idx],
)
v7_target_ecdf_current_scores = v6_soft_reconstruction_mse(
    v7_adapter_model,
    X_stress_2017[v5_original_ecdf_idx],
)
v7_target_build_dynamic_weights, v7_target_weight_statistics = (
    v6_iad_weight_update(v7_target_build_current_scores)
)
v7_target_ecdf_dynamic_weights, _ = v6_iad_weight_update(
    v7_target_ecdf_current_scores,
    statistics=v7_target_weight_statistics,
)
v7_target_build_weights = np.sqrt(
    np.maximum(v6_build_normality_weights, 1e-8)
    * np.maximum(v7_target_build_dynamic_weights, 1e-8)
).astype(np.float32)
v7_target_ecdf_weights = np.sqrt(
    np.maximum(v6_ecdf_normality_weights, 1e-8)
    * np.maximum(v7_target_ecdf_dynamic_weights, 1e-8)
).astype(np.float32)

v7_adapter_target_reference = v7_attach_robust_log_reference(
    v6_fit_soft_reference(
        v7_adapter_model,
        X_stress_2017[v5_original_build_idx],
        v7_target_build_weights,
        X_stress_2017[v5_original_ecdf_idx],
        v7_target_ecdf_weights,
        seed=V7_SEED + 301,
    )
)
v7_adapter_source_reference = v7_attach_robust_log_reference(
    v6_fit_soft_reference(
        v7_adapter_model,
        X_select[v5_source_build_idx],
        np.ones(len(v5_source_build_idx), dtype=np.float32),
        X_select[v5_source_ecdf_idx],
        np.ones(len(v5_source_ecdf_idx), dtype=np.float32),
        seed=V7_SEED + 302,
    )
)

v7_saturation_rows = []
v7_adapter_target_cal_bundle = v7_score_bundle(
    v7_adapter_model,
    X_stress_2017[v5_target_cal_idx],
    v7_adapter_target_reference,
    'target_cal',
    v7_saturation_rows,
)
v7_adapter_target_rank_bundle = v7_score_bundle(
    v7_adapter_model,
    X_stress_2017[v5_target_rank_idx],
    v7_adapter_target_reference,
    'target_rank',
    v7_saturation_rows,
)
v7_adapter_source_cal_bundle = v7_score_bundle(
    v7_adapter_model,
    X_test_internal[v5_source_cal_sample_idx],
    v7_adapter_source_reference,
    'source_cal',
    v7_saturation_rows,
)
v7_adapter_source_rank_bundle = v7_score_bundle(
    v7_adapter_model,
    X_test_internal[v5_source_rank_sample_idx],
    v7_adapter_source_reference,
    'source_rank',
    v7_saturation_rows,
)

for split_name, score_values in (
    ('target_cal', v6_control_target_cal_scores),
    ('target_rank', v6_control_target_rank_scores),
    ('source_cal', v6_control_source_cal_scores),
    ('source_rank', v6_control_source_rank_scores),
):
    maximum = float(np.max(score_values))
    v7_saturation_rows.append({
        'split': split_name,
        'score_kind': 'v4_control_ecdf',
        'rows': len(score_values),
        'fraction_at_or_above_reference_max': np.nan,
        'fraction_ge_099': float(np.mean(score_values >= 0.99)),
        'fraction_ge_0999': float(np.mean(score_values >= 0.999)),
        'fraction_equal_max': float(np.mean(np.isclose(
            score_values,
            maximum,
            rtol=0.0,
            atol=1e-7,
        ))),
        'unique_fraction_rounded_1e7': (
            len(np.unique(np.round(score_values, 7)))
            / max(len(score_values), 1)
        ),
    })

v7_score_saturation_df = pd.DataFrame(v7_saturation_rows)
v7_reference_weight_summary_df = pd.DataFrame([
    {
        'partition': 'target_build',
        'rows': len(v7_target_build_weights),
        'weight_mean': float(np.mean(v7_target_build_weights)),
        'weight_median': float(np.median(v7_target_build_weights)),
        'weight_p05': float(np.quantile(v7_target_build_weights, 0.05)),
        'weight_p95': float(np.quantile(v7_target_build_weights, 0.95)),
    },
    {
        'partition': 'target_ecdf',
        'rows': len(v7_target_ecdf_weights),
        'weight_mean': float(np.mean(v7_target_ecdf_weights)),
        'weight_median': float(np.median(v7_target_ecdf_weights)),
        'weight_p05': float(np.quantile(v7_target_ecdf_weights, 0.05)),
        'weight_p95': float(np.quantile(v7_target_ecdf_weights, 0.95)),
    },
])

del (
    v7_target_build_current_scores,
    v7_target_ecdf_current_scores,
    v7_target_build_dynamic_weights,
    v7_target_ecdf_dynamic_weights,
)
gc.collect()

print('DIAGNOSTICA SATURAZIONE SCORE V7')
print(v7_score_saturation_df.round(6).to_string(index=False))


DIAGNOSTICA SATURAZIONE SCORE V7
      split      score_kind   rows  fraction_at_or_above_reference_max  fraction_ge_099  fraction_ge_0999  fraction_equal_max  unique_fraction_rounded_1e7
 target_cal    raw_soft_mse 471272                            0.000028              NaN               NaN                 NaN                          NaN
 target_cal   raw_soft_topk 471272                            0.000013              NaN               NaN                 NaN                          NaN
 target_cal            ecdf 471272                                 NaN         0.172238          0.008946            0.000004                     0.725952
 target_cal      robust_log 471272                                 NaN              NaN               NaN            0.000002                     0.827138
target_rank    raw_soft_mse 471271                            0.000025              NaN               NaN                 NaN                          NaN
target_rank   raw_soft_topk 471271   

## Selezione v7 cross-dataset e guardrail intra-dataset

Le tre candidate sono valutate sugli stessi split cross-fitted. Il guardrail
source precede sempre PR-AUC e recall target. I round dell'adapter non sono
scelti con label target e il legacy final non è accessibile in questa cella.


In [46]:
# V7_RETRO_CELL_4_SELECTION
v7_candidate_scores = {
    'v4_control': {
        'target_cal': v6_control_target_cal_scores,
        'target_rank': v6_control_target_rank_scores,
        'source_cal': v6_control_source_cal_scores,
        'source_rank': v6_control_source_rank_scores,
    },
    'v7_adapter_ecdf': {
        'target_cal': v7_adapter_target_cal_bundle['ecdf'],
        'target_rank': v7_adapter_target_rank_bundle['ecdf'],
        'source_cal': v7_adapter_source_cal_bundle['ecdf'],
        'source_rank': v7_adapter_source_rank_bundle['ecdf'],
    },
    'v7_adapter_robust_log': {
        'target_cal': v7_adapter_target_cal_bundle['robust_log'],
        'target_rank': v7_adapter_target_rank_bundle['robust_log'],
        'source_cal': v7_adapter_source_cal_bundle['robust_log'],
        'source_rank': v7_adapter_source_rank_bundle['robust_log'],
    },
}
assert tuple(v7_candidate_scores) == V7_CANDIDATE_NAMES

v7_cv_cache = {}
v7_ranking_rows = []
for candidate_name in V7_CANDIDATE_NAMES:
    scores = v7_candidate_scores[candidate_name]
    target_cv = v7_crossfit_f1_thresholds(
        v6_target_cal_y,
        scores['target_cal'],
        v5_target_folds,
        domain='target_candidate',
        candidate=candidate_name,
    )
    source_cv = v7_crossfit_f1_thresholds(
        v6_source_cal_y,
        scores['source_cal'],
        v5_source_folds,
        domain='source_candidate',
        candidate=candidate_name,
    )
    target_rank_metrics = score_metrics(
        v6_target_rank_y,
        scores['target_rank'],
        target_cv['threshold'],
    )
    source_rank_metrics = score_metrics(
        v6_source_rank_y,
        scores['source_rank'],
        source_cv['threshold'],
    )
    recall_020 = v6_recall_at_fpr(
        v6_target_rank_y,
        scores['target_rank'],
        0.20,
    )
    recall_025 = v6_recall_at_fpr(
        v6_target_rank_y,
        scores['target_rank'],
        0.25,
    )
    v7_cv_cache[candidate_name] = {
        'target': target_cv,
        'source': source_cv,
    }
    v7_ranking_rows.append({
        'candidate': candidate_name,
        'adapter_round': (
            0 if candidate_name == 'v4_control' else v7_selected_round
        ),
        'target_rank_f1': target_rank_metrics['f1'],
        'target_rank_precision': target_rank_metrics['precision'],
        'target_rank_recall': target_rank_metrics['recall'],
        'target_rank_fpr_benign': target_rank_metrics['fpr_benign'],
        'target_rank_roc_auc': target_rank_metrics['roc_auc'],
        'target_rank_pr_auc': target_rank_metrics['pr_auc'],
        'target_rank_recall_at_fpr_020': recall_020['recall'],
        'target_rank_recall_at_fpr_025': recall_025['recall'],
        'target_cal_oof_f1': target_cv['oof_metrics']['f1'],
        'target_threshold': target_cv['threshold'],
        'target_threshold_iqr': target_cv['threshold_iqr'],
        'source_rank_f1': source_rank_metrics['f1'],
        'source_rank_precision': source_rank_metrics['precision'],
        'source_rank_recall': source_rank_metrics['recall'],
        'source_rank_fpr_benign': source_rank_metrics['fpr_benign'],
        'source_rank_roc_auc': source_rank_metrics['roc_auc'],
        'source_rank_pr_auc': source_rank_metrics['pr_auc'],
        'source_cal_oof_f1': source_cv['oof_metrics']['f1'],
        'source_threshold': source_cv['threshold'],
        'source_threshold_iqr': source_cv['threshold_iqr'],
    })

v7_ranking_df = pd.DataFrame(v7_ranking_rows)
v7_control_row = v7_ranking_df.loc[
    v7_ranking_df['candidate'].eq('v4_control')
].iloc[0]
v7_ranking_df['eligible_source_guardrail'] = (
    (
        v7_ranking_df['source_rank_f1']
        >= v7_control_row['source_rank_f1'] - V7_SOURCE_MAX_F1_DROP
    )
    & (
        v7_ranking_df['source_rank_recall']
        >= (
            v7_control_row['source_rank_recall']
            - V7_SOURCE_MAX_RECALL_DROP
        )
    )
    & (
        v7_ranking_df['source_rank_fpr_benign']
        <= (
            v7_control_row['source_rank_fpr_benign']
            + V7_SOURCE_MAX_FPR_INCREASE
        )
    )
)
v7_target_pool = v7_ranking_df.loc[
    v7_ranking_df['eligible_source_guardrail']
].sort_values(
    [
        'target_rank_pr_auc',
        'target_rank_recall_at_fpr_020',
        'target_rank_recall_at_fpr_025',
        'target_rank_f1',
        'target_rank_recall',
    ],
    ascending=False,
)
if v7_target_pool.empty:
    raise RuntimeError('Nessuna candidata v7 supera il guardrail source.')
v7_selected_target_name = str(v7_target_pool.iloc[0]['candidate'])

v7_source_pool = v7_ranking_df.loc[
    (
        v7_ranking_df['source_rank_recall']
        >= (
            v7_control_row['source_rank_recall']
            - V7_INTRA_MAX_RECALL_DROP
        )
    )
    & (
        v7_ranking_df['source_rank_fpr_benign']
        <= (
            v7_control_row['source_rank_fpr_benign']
            + V7_SOURCE_MAX_FPR_INCREASE
        )
    )
].sort_values(
    [
        'source_rank_f1',
        'source_rank_recall',
        'source_rank_pr_auc',
        'source_rank_fpr_benign',
    ],
    ascending=[False, False, False, True],
)
if v7_source_pool.empty:
    raise RuntimeError('Nessuna candidata v7 eleggibile sul source.')
v7_selected_source_name = str(v7_source_pool.iloc[0]['candidate'])

v7_ranking_df['selected_target'] = (
    v7_ranking_df['candidate'] == v7_selected_target_name
)
v7_ranking_df['selected_source'] = (
    v7_ranking_df['candidate'] == v7_selected_source_name
)
v7_ranking_df = v7_ranking_df.sort_values(
    [
        'eligible_source_guardrail',
        'target_rank_pr_auc',
        'target_rank_recall_at_fpr_020',
        'target_rank_recall_at_fpr_025',
        'target_rank_f1',
    ],
    ascending=False,
).reset_index(drop=True)
v7_thresholds_df = pd.DataFrame(v7_threshold_rows)

print('RANKING V7 - LEGACY FINAL NON USATO')
print(v7_ranking_df.round(5).to_string(index=False))
print(f'\nCandidata cross-dataset: {v7_selected_target_name}')
print(f'Candidata intra-dataset: {v7_selected_source_name}')


RANKING V7 - LEGACY FINAL NON USATO
            candidate  adapter_round  target_rank_f1  target_rank_precision  target_rank_recall  target_rank_fpr_benign  target_rank_roc_auc  target_rank_pr_auc  target_rank_recall_at_fpr_020  target_rank_recall_at_fpr_025  target_cal_oof_f1  target_threshold  target_threshold_iqr  source_rank_f1  source_rank_precision  source_rank_recall  source_rank_fpr_benign  source_rank_roc_auc  source_rank_pr_auc  source_cal_oof_f1  source_threshold  source_threshold_iqr  eligible_source_guardrail  selected_target  selected_source
           v4_control              0         0.52709                0.43323             0.67285                 0.21565              0.68804             0.31492                        0.64814                        0.69299            0.50346           0.80985               0.00259         0.91654                0.92771             0.90563                 0.09694              0.92376             0.91609            0.91650           0.8

## Report post-hoc sul legacy final ed esportazione v6

Solo dopo il congelamento delle scelte vengono valutate tutte le candidate
sul final già osservato. I risultati sono retrospettivi, non una nuova prova
esterna. Vengono esportati tabelle, diagnostiche, figure e i pesi del ramo
IAD selezionato dal criterio non supervisionato.


In [47]:
# V6_RETRO_CELL_7_LEGACY_FINAL_EXPORT
V6_EXPORT_DIR = Path('Artefatti') / 'protocol_v6_retro'
V6_FIGURE_DIR = V6_EXPORT_DIR / 'figures'
V6_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
V6_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if V6_RUN_LEGACY_FINAL_REPORT:
    required_v6_final = ['idx_final', 'cda_final_idx']
    missing_v6_final = [
        name for name in required_v6_final if name not in globals()
    ]
    if missing_v6_final:
        raise RuntimeError(
            f'Indici legacy final mancanti: {missing_v6_final}'
        )
    v6_target_final_idx = np.asarray(cda_final_idx, dtype=np.int64)
    v6_source_final_idx = np.asarray(idx_final, dtype=np.int64)
    v6_target_final_y = y_stress_2017_binary[v6_target_final_idx]
    v6_source_final_y = y_test_internal_binary[v6_source_final_idx]

    v6_final_score_map = {
        'v4_control': {
            'target': score_variant_values(
                v6_control_variant,
                X_target_pseudobenign[v6_target_final_idx],
                'target',
                v6_control_recipe,
            ),
            'source': score_variant_values(
                v6_control_variant,
                X_test_internal[v6_source_final_idx],
                'source',
                v6_control_recipe,
            ),
        },
        'v6_cadet_full_soft': {
            'target': v6_score_cadet(
                v6_cadet_target_state,
                X_stress_2017[v6_target_final_idx],
            ),
            'source': v6_score_cadet(
                v6_cadet_source_state,
                X_test_internal[v6_source_final_idx],
            ),
        },
        'v6_iad_full_soft': {
            'target': v6_score_soft(
                v6_iad_model,
                X_stress_2017[v6_target_final_idx],
                v6_iad_target_reference,
            ),
            'source': v6_score_soft(
                v6_iad_model,
                X_test_internal[v6_source_final_idx],
                v6_iad_source_reference,
            ),
        },
    }

    v6_legacy_final_rows = []
    for candidate_name in V6_CANDIDATE_NAMES:
        target_threshold = v6_cv_cache[candidate_name][
            'target'
        ]['threshold']
        source_threshold = v6_cv_cache[candidate_name][
            'source'
        ]['threshold']
        target_scores = v6_final_score_map[candidate_name]['target']
        source_scores = v6_final_score_map[candidate_name]['source']
        target_predictions = (
            target_scores >= target_threshold
        ).astype(np.int8)
        source_predictions = (
            source_scores >= source_threshold
        ).astype(np.int8)
        v6_legacy_final_rows.extend([
            binary_metrics_row(
                'CIC-IDS2017',
                f'{candidate_name}__crossfitted_f1',
                v6_target_final_y,
                target_predictions,
                target_scores,
                V6_LEGACY_FINAL_STATUS,
                (
                    'Candidata congelata senza legacy final; risultato '
                    'retrospettivo da confermare esternamente.'
                ),
                target_threshold,
            ),
            binary_metrics_row(
                'CSE-CIC-IDS2018',
                f'{candidate_name}__source_crossfitted_f1',
                v6_source_final_y,
                source_predictions,
                source_scores,
                V6_LEGACY_FINAL_STATUS,
                (
                    'Guardrail intra-dataset post-hoc con soglia '
                    'cross-fitted.'
                ),
                source_threshold,
            ),
        ])
    v6_legacy_final_df = pd.DataFrame(v6_legacy_final_rows)

    v6_selected_target_scores = v6_final_score_map[
        v6_selected_target_name
    ]['target']
    v6_selected_target_threshold = v6_cv_cache[
        v6_selected_target_name
    ]['target']['threshold']
    v6_selected_target_predictions = (
        v6_selected_target_scores >= v6_selected_target_threshold
    ).astype(np.int8)
    v6_attack_frame = pd.DataFrame({
        'attack_type': np.asarray(y_stress_2017)[
            v6_target_final_idx
        ].astype(str),
        'detected': v6_selected_target_predictions,
    })
    v6_attack_frame = v6_attack_frame.loc[
        v6_attack_frame['attack_type'].str.strip().str.upper().ne(
            'BENIGN'
        )
    ]
    v6_per_attack_df = (
        v6_attack_frame.groupby('attack_type', as_index=False)
        .agg(
            total_samples=('detected', 'size'),
            detected_samples=('detected', 'sum'),
        )
    )
    v6_per_attack_df['detection_rate'] = (
        v6_per_attack_df['detected_samples']
        / v6_per_attack_df['total_samples']
    )
    v6_per_attack_df = v6_per_attack_df.sort_values(
        'detection_rate'
    ).reset_index(drop=True)

    v6_purity_diagnostics_df = pd.DataFrame([
        {
            'reference_stage': 'continuous_build_all_rows',
            'rows': len(v5_original_build_idx),
            'diagnostic_benign_fraction_not_used': float(np.mean(
                y_stress_2017_binary[v5_original_build_idx] == 0
            )),
            'weighted_benign_fraction_not_used': float(np.average(
                y_stress_2017_binary[v5_original_build_idx] == 0,
                weights=v6_build_normality_weights,
            )),
        },
        {
            'reference_stage': 'continuous_ecdf_all_rows',
            'rows': len(v5_original_ecdf_idx),
            'diagnostic_benign_fraction_not_used': float(np.mean(
                y_stress_2017_binary[v5_original_ecdf_idx] == 0
            )),
            'weighted_benign_fraction_not_used': float(np.average(
                y_stress_2017_binary[v5_original_ecdf_idx] == 0,
                weights=v6_ecdf_normality_weights,
            )),
        },
    ])
else:
    v6_legacy_final_df = pd.DataFrame()
    v6_per_attack_df = pd.DataFrame()
    v6_purity_diagnostics_df = pd.DataFrame()

v6_iad_weights_path = (
    V6_EXPORT_DIR / 'v6_iad_selected_unsupervised_round.weights.h5'
)
v6_iad_model.save_weights(v6_iad_weights_path)

v6_exports = {
    'protocol_config.csv': v6_protocol_config_df,
    'literature_fidelity.csv': v6_literature_fidelity_df,
    'normality_weight_summary.csv': v6_normality_weight_summary_df,
    'preprocessing_ablation.csv': v6_preprocessing_ablation_df,
    'cadet_diagnostics.csv': v6_cadet_diagnostics_df,
    'iad_history.csv': v6_iad_history_df,
    'candidate_ranking.csv': v6_ranking_df,
    'crossfitted_thresholds.csv': v6_thresholds_df,
    'legacy_final_results.csv': v6_legacy_final_df,
    'purity_diagnostics_posthoc.csv': v6_purity_diagnostics_df,
    'per_attack_selected_target.csv': v6_per_attack_df,
}
for filename, frame in v6_exports.items():
    frame.to_csv(V6_EXPORT_DIR / filename, index=False)

v6_figure_manifest = []
ranking_plot = v6_ranking_df.sort_values('target_rank_pr_auc')
figure, axis = plt.subplots(figsize=(9, 4.8))
axis.barh(
    ranking_plot['candidate'],
    ranking_plot['target_rank_pr_auc'],
    color=['#64748b', '#2563eb', '#16a34a'],
)
axis.set(
    xlabel='PR-AUC target ranking',
    title='Protocollo v6: ranking cross-dataset prima del legacy final',
    xlim=(0, 1),
)
ranking_figure_path = V6_FIGURE_DIR / '01_target_ranking_pr_auc.png'
figure.savefig(
    ranking_figure_path,
    dpi=300,
    bbox_inches='tight',
    facecolor='white',
)
plt.close(figure)
v6_figure_manifest.append({
    'filename': ranking_figure_path.name,
    'description': 'PR-AUC delle tre candidate sul ranking target.',
})

if len(v6_per_attack_df):
    attack_plot = v6_per_attack_df.sort_values('detection_rate')
    figure, axis = plt.subplots(figsize=(10, 6))
    axis.barh(
        attack_plot['attack_type'],
        attack_plot['detection_rate'],
        color='#7c3aed',
    )
    axis.set(
        xlabel='Detection rate',
        title=(
            'Protocollo v6: detection rate per attacco '
            f'({v6_selected_target_name})'
        ),
        xlim=(0, 1),
    )
    attack_figure_path = (
        V6_FIGURE_DIR / '02_per_attack_selected_target.png'
    )
    figure.savefig(
        attack_figure_path,
        dpi=300,
        bbox_inches='tight',
        facecolor='white',
    )
    plt.close(figure)
    v6_figure_manifest.append({
        'filename': attack_figure_path.name,
        'description': (
            'Detection rate post-hoc per famiglia della candidata target.'
        ),
    })

v6_figure_manifest_df = pd.DataFrame(v6_figure_manifest)
v6_figure_manifest_df.to_csv(
    V6_FIGURE_DIR / 'manifest_figure_v6.csv',
    index=False,
)

print('ARTEFATTI V6 ESPORTATI')
print(f'Cartella: {V6_EXPORT_DIR.resolve()}')
print(f'Pesi IAD: {v6_iad_weights_path.resolve()}')
if len(v6_legacy_final_df):
    print('\nRISULTATI LEGACY FINAL - SOLO POST-HOC')
    print(result_matrix(v6_legacy_final_df).to_string(index=False))
print('Il LaTeX non è stato modificato.')


C:\Users\Anton\AppData\Local\Temp\ipykernel_17244\492000690.py:236: UserWarning: Glyph 65533 (\N{REPLACEMENT CHARACTER}) missing from current font.
  figure.savefig(


ARTEFATTI V6 ESPORTATI
Cartella: D:\Users\Anton\Desktop\Progetti VsCode\Codex\Rilevamento di Anomalie del Traffico di Rete con Autoencoder\Rilevamento-di-Anomalie-del-Traffico-di-Rete-con-Autoencoder\Artefatti\protocol_v6_retro
Pesi IAD: D:\Users\Anton\Desktop\Progetti VsCode\Codex\Rilevamento di Anomalie del Traffico di Rete con Autoencoder\Rilevamento-di-Anomalie-del-Traffico-di-Rete-con-Autoencoder\Artefatti\protocol_v6_retro\v6_iad_selected_unsupervised_round.weights.h5

RISULTATI LEGACY FINAL - SOLO POST-HOC
        dataset                                  scenario  threshold     f1  precision  recall  roc_auc  pr_auc  fpr_benign  accuracy     tn     fp    fn     tp
    CIC-IDS2017                v4_control__crossfitted_f1     0.8098 0.5255     0.4319  0.6708   0.6874  0.3142      0.2161    0.7616 296732  81805 30530  62205
CSE-CIC-IDS2018         v4_control__source_crossfitted_f1     0.8843 0.9173     0.9282  0.9067   0.9242  0.9161      0.0964    0.9054 542158  57842 76915 74730

## Report post-hoc ed esportazione completa degli artefatti v7

Solo dopo il congelamento della candidata vengono letti i legacy final 2017
e 2018. La cella esporta risultati, diagnostica di saturazione, audit dei
round e detection rate per attacco di tutte le candidate. Il risultato resta
retrospettivo finché non viene confermato su un nuovo dominio esterno.


In [48]:
# V7_RETRO_CELL_5_LEGACY_FINAL_EXPORT
V7_EXPORT_DIR = Path('Artefatti') / 'protocol_v7_retro'
V7_FIGURE_DIR = V7_EXPORT_DIR / 'figures'
V7_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
V7_FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def v7_clean_attack_label(value):
    return (
        str(value)
        .replace('ï¿½', '-')
        .replace('\ufffd', '-')
        .replace('  ', ' ')
        .strip()
    )


if V7_RUN_LEGACY_FINAL_REPORT:
    required_v7_final = ['idx_final', 'cda_final_idx']
    missing_v7_final = [
        name for name in required_v7_final if name not in globals()
    ]
    if missing_v7_final:
        raise RuntimeError(
            f'Indici legacy final mancanti: {missing_v7_final}'
        )
    v7_target_final_idx = np.asarray(cda_final_idx, dtype=np.int64)
    v7_source_final_idx = np.asarray(idx_final, dtype=np.int64)
    v7_target_final_y = y_stress_2017_binary[v7_target_final_idx]
    v7_source_final_y = y_test_internal_binary[v7_source_final_idx]

    v7_adapter_target_final_bundle = v7_score_bundle(
        v7_adapter_model,
        X_stress_2017[v7_target_final_idx],
        v7_adapter_target_reference,
        'target_legacy_final_posthoc',
        v7_saturation_rows,
    )
    v7_adapter_source_final_bundle = v7_score_bundle(
        v7_adapter_model,
        X_test_internal[v7_source_final_idx],
        v7_adapter_source_reference,
        'source_legacy_final_posthoc',
        v7_saturation_rows,
    )
    v7_final_score_map = {
        'v4_control': {
            'target': score_variant_values(
                v6_control_variant,
                X_target_pseudobenign[v7_target_final_idx],
                'target',
                v6_control_recipe,
            ),
            'source': score_variant_values(
                v6_control_variant,
                X_test_internal[v7_source_final_idx],
                'source',
                v6_control_recipe,
            ),
        },
        'v7_adapter_ecdf': {
            'target': v7_adapter_target_final_bundle['ecdf'],
            'source': v7_adapter_source_final_bundle['ecdf'],
        },
        'v7_adapter_robust_log': {
            'target': v7_adapter_target_final_bundle['robust_log'],
            'source': v7_adapter_source_final_bundle['robust_log'],
        },
    }

    v7_legacy_final_rows = []
    v7_per_attack_rows = []
    for candidate_name in V7_CANDIDATE_NAMES:
        target_threshold = v7_cv_cache[candidate_name][
            'target'
        ]['threshold']
        source_threshold = v7_cv_cache[candidate_name][
            'source'
        ]['threshold']
        target_scores = v7_final_score_map[candidate_name]['target']
        source_scores = v7_final_score_map[candidate_name]['source']
        target_predictions = (
            target_scores >= target_threshold
        ).astype(np.int8)
        source_predictions = (
            source_scores >= source_threshold
        ).astype(np.int8)
        v7_legacy_final_rows.extend([
            binary_metrics_row(
                'CIC-IDS2017',
                f'{candidate_name}__crossfitted_f1',
                v7_target_final_y,
                target_predictions,
                target_scores,
                V7_LEGACY_FINAL_STATUS,
                (
                    'Legacy final post-hoc; non usato per addestramento, '
                    'round, score o selezione.'
                ),
                target_threshold,
            ),
            binary_metrics_row(
                'CSE-CIC-IDS2018',
                f'{candidate_name}__source_crossfitted_f1',
                v7_source_final_y,
                source_predictions,
                source_scores,
                V7_LEGACY_FINAL_STATUS,
                'Guardrail intra-dataset post-hoc.',
                source_threshold,
            ),
        ])

        attack_frame = pd.DataFrame({
            'attack_type': [
                v7_clean_attack_label(value)
                for value in y_stress_2017[v7_target_final_idx]
            ],
            'detected': target_predictions,
        })
        attack_frame = attack_frame.loc[
            attack_frame['attack_type'].str.upper().ne('BENIGN')
        ]
        grouped = (
            attack_frame.groupby('attack_type', as_index=False)
            .agg(
                total_samples=('detected', 'size'),
                detected_samples=('detected', 'sum'),
            )
        )
        grouped['detection_rate'] = (
            grouped['detected_samples'] / grouped['total_samples']
        )
        grouped.insert(0, 'candidate', candidate_name)
        v7_per_attack_rows.extend(grouped.to_dict('records'))

    v7_legacy_final_df = pd.DataFrame(v7_legacy_final_rows)
    v7_per_attack_all_candidates_df = pd.DataFrame(v7_per_attack_rows)
else:
    v7_legacy_final_df = pd.DataFrame()
    v7_per_attack_all_candidates_df = pd.DataFrame()

v7_score_saturation_df = pd.DataFrame(v7_saturation_rows)
v7_adapter_weights_path = (
    V7_EXPORT_DIR / 'v7_adapter_selected_round.weights.h5'
)
v7_adapter_model.save_weights(v7_adapter_weights_path)

v7_exports = {
    'protocol_config.csv': v7_protocol_config_df,
    'adapter_round_audit.csv': v7_adapter_round_audit_df,
    'reference_weight_summary.csv': v7_reference_weight_summary_df,
    'score_saturation.csv': v7_score_saturation_df,
    'candidate_ranking.csv': v7_ranking_df,
    'crossfitted_thresholds.csv': v7_thresholds_df,
    'legacy_final_results.csv': v7_legacy_final_df,
    'per_attack_all_candidates.csv': v7_per_attack_all_candidates_df,
}
for filename, frame in v7_exports.items():
    frame.to_csv(V7_EXPORT_DIR / filename, index=False)

v7_figure_manifest = []
ranking_plot = v7_ranking_df.sort_values('target_rank_pr_auc')
figure, axis = plt.subplots(figsize=(9.5, 4.8))
axis.barh(
    ranking_plot['candidate'],
    ranking_plot['target_rank_pr_auc'],
    color='#35618f',
)
axis.set_xlabel('PR-AUC target ranking')
axis.set_title('Protocollo v7: ranking cross-dataset')
axis.grid(axis='x', alpha=0.25)
figure.tight_layout()
ranking_figure_path = V7_FIGURE_DIR / '01_target_ranking_pr_auc.png'
figure.savefig(
    ranking_figure_path,
    dpi=300,
    bbox_inches='tight',
    facecolor='white',
)
plt.close(figure)
v7_figure_manifest.append({
    'filename': ranking_figure_path.name,
    'description': 'PR-AUC delle tre candidate sul ranking target.',
})

if len(v7_per_attack_all_candidates_df):
    selected_attack_plot = v7_per_attack_all_candidates_df.loc[
        v7_per_attack_all_candidates_df['candidate'].eq(
            v7_selected_target_name
        )
    ].sort_values('detection_rate')
    figure_height = max(5.5, 0.38 * len(selected_attack_plot))
    figure, axis = plt.subplots(figsize=(10.5, figure_height))
    axis.barh(
        selected_attack_plot['attack_type'],
        selected_attack_plot['detection_rate'],
        color='#a64b42',
    )
    axis.set_xlim(0.0, 1.0)
    axis.set_xlabel('Detection rate')
    axis.set_title(
        'Detection rate per attacco - '
        f'{v7_selected_target_name}'
    )
    axis.grid(axis='x', alpha=0.25)
    figure.tight_layout()
    attack_figure_path = (
        V7_FIGURE_DIR / '02_per_attack_selected_target.png'
    )
    figure.savefig(
        attack_figure_path,
        dpi=300,
        bbox_inches='tight',
        facecolor='white',
    )
    plt.close(figure)
    v7_figure_manifest.append({
        'filename': attack_figure_path.name,
        'description': (
            'Detection rate post-hoc per famiglia della candidata target.'
        ),
    })

v7_figure_manifest_df = pd.DataFrame(v7_figure_manifest)
v7_figure_manifest_df.to_csv(
    V7_FIGURE_DIR / 'manifest_figure_v7.csv',
    index=False,
)

print('ARTEFATTI V7 ESPORTATI')
print(f'Cartella: {V7_EXPORT_DIR.resolve()}')
print(f'Pesi adapter: {v7_adapter_weights_path.resolve()}')
if len(v7_legacy_final_df):
    print('\nRISULTATI LEGACY FINAL - SOLO POST-HOC')
    print(result_matrix(v7_legacy_final_df).to_string(index=False))
print('Il LaTeX non è stato modificato.')


ARTEFATTI V7 ESPORTATI
Cartella: D:\Users\Anton\Desktop\Progetti VsCode\Codex\Rilevamento di Anomalie del Traffico di Rete con Autoencoder\Rilevamento-di-Anomalie-del-Traffico-di-Rete-con-Autoencoder\Artefatti\protocol_v7_retro
Pesi adapter: D:\Users\Anton\Desktop\Progetti VsCode\Codex\Rilevamento di Anomalie del Traffico di Rete con Autoencoder\Rilevamento-di-Anomalie-del-Traffico-di-Rete-con-Autoencoder\Artefatti\protocol_v7_retro\v7_adapter_selected_round.weights.h5

RISULTATI LEGACY FINAL - SOLO POST-HOC
        dataset                                     scenario  threshold     f1  precision  recall  roc_auc  pr_auc  fpr_benign  accuracy     tn     fp    fn     tp
    CIC-IDS2017                   v4_control__crossfitted_f1     0.8098 0.5255     0.4319  0.6708   0.6874  0.3142      0.2161    0.7616 296732  81805 30530  62205
CSE-CIC-IDS2018            v4_control__source_crossfitted_f1     0.8843 0.9173     0.9282  0.9067   0.9242  0.9161      0.0964    0.9054 542158  57842 76915 7